# Procesamiento Gold - Danper

## Objetivo
Construir una capa analitica a partir de las tablas Silver, generando indicadores agrocomerciales por cultivo, ubicacion y periodo.

La implementacion utiliza PySpark y Hadoop HDFS. Se conservaran las unidades y granularidades de cada fuente para evitar integraciones incorrectas.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('Danper_Procesamiento_Gold')
    .master('local[2]')
    .config('spark.hadoop.fs.defaultFS', 'hdfs://localhost:9000')
    .config('spark.sql.shuffle.partitions', '2')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')
BASE_SILVER = 'hdfs://localhost:9000/danper/silver'
BASE_GOLD = 'hdfs://localhost:9000/danper/gold'

print('Spark iniciado:', spark.version)
print('Origen Silver:', BASE_SILVER)


26/10/11 12:38:09 WARN Utils: Your hostname, Debian12 resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/11 12:38:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/11 12:38:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark iniciado: 3.5.1
Origen Silver: hdfs://localhost:9000/danper/silver


In [2]:
produccion = spark.read.parquet(f'{BASE_SILVER}/produccion')
print('Columnas:', len(produccion.columns))
print('Registros:', produccion.count())


Columnas: 13
Registros: 65922


In [3]:
produccion.select(
    'provincia', 'distrito', 'cultivo',
    'variable', 'valor', 'periodo'
).show(10, truncate=False)


+---------+--------+-------+---------------+-----+----------+
|provincia|distrito|cultivo|variable       |valor|periodo   |
+---------+--------+-------+---------------+-----+----------+
|JULCAN   |JULCAN  |ALFALFA|Sup.Verde (ha.)|12.0 |2025-10-01|
|JULCAN   |JULCAN  |ALFALFA|Sup.Verde (ha.)|12.0 |2025-11-01|
|JULCAN   |JULCAN  |ALFALFA|Sup.Verde (ha.)|12.0 |2025-12-01|
|JULCAN   |JULCAN  |ALFALFA|Cosechas (ha.) |12.0 |2025-12-01|
|JULCAN   |JULCAN  |ALFALFA|Produccion (t.)|26.8 |2025-01-01|
|JULCAN   |JULCAN  |ALFALFA|Produccion (t.)|29.9 |2025-02-01|
|JULCAN   |JULCAN  |ALFALFA|Produccion (t.)|31.5 |2025-03-01|
|JULCAN   |JULCAN  |ALFALFA|Produccion (t.)|30.5 |2025-04-01|
|JULCAN   |JULCAN  |ALFALFA|Produccion (t.)|27.6 |2025-05-01|
|JULCAN   |JULCAN  |ALFALFA|Produccion (t.)|27.9 |2025-06-01|
+---------+--------+-------+---------------+-----+----------+
only showing top 10 rows



## Transformación Gold: producción agrícola mensual

En esta etapa se transforma la información de producción agrícola
procedente de Silver en una tabla analítica organizada por provincia,
distrito, cultivo y periodo.

Se generan indicadores de producción en toneladas, precio en chacra
en S/kg, superficie cosechada en hectáreas y otros indicadores
disponibles.

Se conserva la granularidad mensual y no se reemplazan los
valores faltantes por cero.

In [4]:
from pyspark.sql import functions as F

print("===== PASO 24: TRANSFORMACION GOLD =====")
print("PROGRESO: 0% - Preparando datos")

# Seleccionar los cultivos priorizados en el proyecto
cultivos_prioritarios = [
    "ESPARRAGO",
    "ALCACHOFA",
    "ARANDANO",
    "PALTA",
    "UVA"
]

produccion_filtrada = (
    produccion
    .withColumn("cultivo", F.upper(F.trim(F.col("cultivo"))))
    .filter(F.col("cultivo").isin(cultivos_prioritarios))
    .filter(F.col("periodo").isNotNull())
)

print("PROGRESO: 25% - Cultivos filtrados")

# Mantener separadas las unidades de cada indicador
variables = {
    "Produccion (t.)": "produccion_t",
    "Precio Chacra (S/Kg.)": "precio_chacra_s_kg",
    "Cosechas (ha.)": "cosechas_ha",
    "Siembras (ha.)": "siembras_ha",
    "Sup.Verde (ha.)": "superficie_verde_ha",
    "Sup Perdida (ha.)": "superficie_perdida_ha",
    "Rendimiento (Kg./ha.)": "rendimiento_kg_ha"
}

# Comprobar que cada cultivo, ubicación, periodo y
# variable tenga una sola observación.
claves = [
    "provincia",
    "distrito",
    "cultivo",
    "periodo",
    "variable"
]

duplicados = (
    produccion_filtrada
    .groupBy(*claves)
    .count()
    .filter(F.col("count") > 1)
)

numero_duplicados = duplicados.count()

print("Grupos con más de un registro:", numero_duplicados)

if numero_duplicados > 0:
    print("ATENCION: Revisar duplicados antes de transformar.")
    duplicados.show(10, truncate=False)
else:
    print("PROGRESO: 50% - Claves verificadas")

    # Convertir los indicadores de filas a columnas.
    gold_produccion_mensual = (
        produccion_filtrada
        .groupBy(
            "provincia",
            "distrito",
            "cultivo",
            "periodo"
        )
        .pivot("variable", list(variables.keys()))
        .agg(F.first("valor", ignorenulls=True))
    )

    # Renombrar cada columna según su unidad.
    for original, nuevo in variables.items():
        gold_produccion_mensual = (
            gold_produccion_mensual
            .withColumnRenamed(original, nuevo)
        )

    gold_produccion_mensual = (
        gold_produccion_mensual
        .withColumn("anio", F.year("periodo"))
        .withColumn("mes_numero", F.month("periodo"))
    )

    print("PROGRESO: 75% - Tabla Gold construida")

    print("\n===== ESQUEMA GOLD =====")
    gold_produccion_mensual.printSchema()

    print("\n===== REGISTROS GOLD =====")
    print("Total:", gold_produccion_mensual.count())

    print("\n===== MUESTRA GOLD =====")
    gold_produccion_mensual.orderBy(
        "cultivo", "provincia", "distrito", "periodo"
    ).show(15, truncate=False)

    print("PROGRESO: 100% - Transformacion completada")

===== PASO 24: TRANSFORMACION GOLD =====
PROGRESO: 0% - Preparando datos
PROGRESO: 25% - Cultivos filtrados


Grupos con más de un registro: 0
PROGRESO: 50% - Claves verificadas
PROGRESO: 75% - Tabla Gold construida

===== ESQUEMA GOLD =====
root
 |-- provincia: string (nullable = true)
 |-- distrito: string (nullable = true)
 |-- cultivo: string (nullable = true)
 |-- periodo: date (nullable = true)
 |-- produccion_t: double (nullable = true)
 |-- precio_chacra_s_kg: double (nullable = true)
 |-- cosechas_ha: double (nullable = true)
 |-- siembras_ha: double (nullable = true)
 |-- superficie_verde_ha: double (nullable = true)
 |-- superficie_perdida_ha: double (nullable = true)
 |-- rendimiento_kg_ha: double (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes_numero: integer (nullable = true)


===== REGISTROS GOLD =====
Total: 561

===== MUESTRA GOLD =====
+---------+------------+---------+----------+------------+------------------+-----------+-----------+-------------------+---------------------+-----------------+----+----------+
|provincia|distrito    |cultivo  |periodo   |prod

In [5]:
from pyspark.sql import functions as F

print("===== PASO 25: VALIDACION GOLD PRODUCCION =====")

print("\nPROGRESO: 20% - Registros por cultivo")

gold_produccion_mensual.groupBy("cultivo") \
    .agg(
        F.count("*").alias("registros"),
        F.count("produccion_t").alias("meses_con_produccion"),
        F.count("precio_chacra_s_kg").alias("meses_con_precio")
    ) \
    .orderBy("cultivo") \
    .show(truncate=False)

print("\nPROGRESO: 40% - Valores negativos")

columnas = [
    "produccion_t",
    "precio_chacra_s_kg",
    "cosechas_ha",
    "siembras_ha",
    "superficie_verde_ha",
    "superficie_perdida_ha",
    "rendimiento_kg_ha"
]

for columna in columnas:
    negativos = gold_produccion_mensual.filter(
        F.col(columna) < 0
    ).count()

    print(f"{columna}: {negativos} negativos")

print("\nPROGRESO: 60% - Comprobar unicidad Gold")

duplicados_gold = (
    gold_produccion_mensual
    .groupBy("provincia", "distrito", "cultivo", "periodo")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Claves duplicadas Gold:", duplicados_gold)

print("\nPROGRESO: 80% - Comparar produccion Silver y Gold")

total_silver = (
    produccion_filtrada
    .filter(F.col("variable") == "Produccion (t.)")
    .agg(F.sum("valor").alias("total"))
    .first()["total"]
)

total_gold = (
    gold_produccion_mensual
    .agg(F.sum("produccion_t").alias("total"))
    .first()["total"]
)

print("Total produccion Silver (t):", total_silver)
print("Total produccion Gold (t):", total_gold)

diferencia = abs((total_silver or 0) - (total_gold or 0))

print("Diferencia:", diferencia)

if duplicados_gold == 0 and diferencia < 0.0001:
    print("\nVALIDACION PRINCIPAL: CORRECTA")
else:
    print("\nATENCION: REVISAR RESULTADOS")

print("\nPROGRESO: 100% - Validacion finalizada")
print("===== PASO 25 FINALIZADO =====")

===== PASO 25: VALIDACION GOLD PRODUCCION =====

PROGRESO: 20% - Registros por cultivo
+---------+---------+--------------------+----------------+
|cultivo  |registros|meses_con_produccion|meses_con_precio|
+---------+---------+--------------------+----------------+
|ALCACHOFA|62       |16                  |16              |
|ARANDANO |176      |73                  |73              |
|ESPARRAGO|323      |178                 |178             |
+---------+---------+--------------------+----------------+


PROGRESO: 40% - Valores negativos
produccion_t: 0 negativos
precio_chacra_s_kg: 0 negativos
cosechas_ha: 0 negativos
siembras_ha: 0 negativos
superficie_verde_ha: 0 negativos
superficie_perdida_ha: 0 negativos
rendimiento_kg_ha: 0 negativos

PROGRESO: 60% - Comprobar unicidad Gold
Claves duplicadas Gold: 0

PROGRESO: 80% - Comparar produccion Silver y Gold
Total produccion Silver (t): 380702.11100000003
Total produccion Gold (t): 380702.111
Diferencia: 5.820766091346741e-11

VALIDACION 

## Almacenamiento de la capa Gold

Después de validar la tabla de producción agrícola mensual,
se almacenan los resultados en Hadoop HDFS utilizando
el formato columnar Parquet.

La tabla conserva la granularidad de cultivo, provincia,
distrito y periodo mensual.

El almacenamiento se realiza en la ruta
/danper/gold/produccion_mensual.

No se modifican los datos originales de la capa Silver.

In [6]:
print("===== PASO 26: GUARDAR GOLD EN HDFS =====")
print("PROGRESO: 0% - Preparando almacenamiento")

ruta_gold = f"{BASE_GOLD}/produccion_mensual"

# Consultar HDFS antes de escribir.
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
ruta_java = spark._jvm.org.apache.hadoop.fs.Path(ruta_gold)
filesystem = ruta_java.getFileSystem(hadoop_conf)

if filesystem.exists(ruta_java):
    print("ATENCION: La ruta Gold ya existe.")
    print("No se sobrescribira ningun archivo.")
    print("Ruta:", ruta_gold)

else:
    print("PROGRESO: 25% - Ruta disponible")

    # Guardar la tabla validada.
    (
        gold_produccion_mensual
        .coalesce(1)
        .write
        .mode("errorifexists")
        .parquet(ruta_gold)
    )

    print("PROGRESO: 60% - Datos guardados")

    # Leer nuevamente desde HDFS.
    gold_verificacion = spark.read.parquet(ruta_gold)

    print("\n===== VERIFICACION DESDE HDFS =====")
    print("Registros guardados:", gold_verificacion.count())
    print("Columnas guardadas:", len(gold_verificacion.columns))

    gold_verificacion.groupBy("cultivo") \
        .count() \
        .orderBy("cultivo") \
        .show(truncate=False)

    print("PROGRESO: 100% - Almacenamiento completado")

print("===== PASO 26 FINALIZADO =====")

===== PASO 26: GUARDAR GOLD EN HDFS =====
PROGRESO: 0% - Preparando almacenamiento
PROGRESO: 25% - Ruta disponible


PROGRESO: 60% - Datos guardados

===== VERIFICACION DESDE HDFS =====
Registros guardados: 561
Columnas guardadas: 13
+---------+-----+
|cultivo  |count|
+---------+-----+
|ALCACHOFA|62   |
|ARANDANO |176  |
|ESPARRAGO|323  |
+---------+-----+

PROGRESO: 100% - Almacenamiento completado
===== PASO 26 FINALIZADO =====


## Transformación Gold: exportaciones regionales

Se transforma la información de exportaciones regionales de Silver
en una tabla analítica mensual por cultivo y periodo.

Se organizan tres indicadores: volumen exportado en toneladas,
valor FOB en miles de dólares y precio FOB en USD/kg.

Los indicadores mantienen sus unidades originales y no se
reemplazan valores faltantes por cero.

Esta tabla representa estadísticas regionales, no exportaciones
exclusivas de Danper.

In [7]:
from pyspark.sql import functions as F

print("===== PASO 27: GOLD EXPORTACIONES REGIONALES =====")
print("PROGRESO: 0% - Leyendo Silver")

exportaciones_regionales = spark.read.parquet(
    f"{BASE_SILVER}/exportaciones/regionales"
)

cultivos_prioritarios = [
    "ESPARRAGO", "ALCACHOFA", "ARANDANO", "PALTA", "UVA"
]

exportaciones_filtradas = (
    exportaciones_regionales
    .withColumn("cultivo", F.upper(F.trim("cultivo")))
    .filter(F.col("cultivo").isin(cultivos_prioritarios))
    .filter(F.col("anio").isNotNull())
    .filter(F.col("numero_mes").between(1, 12))
)

print("PROGRESO: 25% - Verificando indicadores")

# Las unidades deben corresponder a cada indicador.
unidades_validas = (
    ((F.col("indicador") == "VOLUMEN") &
     (F.col("unidad") == "TONELADAS")) |
    ((F.col("indicador") == "VALOR_FOB") &
     (F.col("unidad") == "MILES_USD")) |
    ((F.col("indicador") == "PRECIO_FOB") &
     (F.col("unidad") == "USD_KG"))
)

invalidos = exportaciones_filtradas.filter(
    ~unidades_validas | F.col("indicador").isNull()
)

print("Registros con indicador/unidad no compatible:",
      invalidos.count())

# Comprobar unicidad antes de convertir filas a columnas.
claves = ["cultivo", "anio", "numero_mes", "indicador"]

duplicados_exp = (
    exportaciones_filtradas
    .groupBy(*claves)
    .count()
    .filter(F.col("count") > 1)
)

cantidad_duplicados = duplicados_exp.count()
print("Grupos duplicados:", cantidad_duplicados)

# Comprobar valores negativos.
negativos_exp = exportaciones_filtradas.filter(
    F.col("valor") < 0
).count()

print("Valores negativos:", negativos_exp)

if invalidos.count() > 0 or cantidad_duplicados > 0 or negativos_exp > 0:
    print("\nATENCION: Revisar los datos antes de transformar.")
    invalidos.show(10, truncate=False)
    duplicados_exp.show(10, truncate=False)

else:
    print("PROGRESO: 50% - Construyendo tabla Gold")

    gold_exportaciones_regionales = (
        exportaciones_filtradas
        .groupBy("cultivo", "anio", "numero_mes")
        .pivot(
            "indicador",
            ["VOLUMEN", "VALOR_FOB", "PRECIO_FOB"]
        )
        .agg(F.first("valor", ignorenulls=True))
        .withColumnRenamed("VOLUMEN", "volumen_exportado_t")
        .withColumnRenamed("VALOR_FOB", "valor_fob_miles_usd")
        .withColumnRenamed("PRECIO_FOB", "precio_fob_usd_kg")
        .withColumn(
            "periodo",
            F.make_date(
                F.col("anio").cast("int"),
                F.col("numero_mes").cast("int"),
                F.lit(1)
            )
        )
    )

    print("PROGRESO: 75% - Validando estructura")

    print("\n===== ESQUEMA GOLD =====")
    gold_exportaciones_regionales.printSchema()

    print("\n===== REGISTROS GOLD =====")
    print("Total:", gold_exportaciones_regionales.count())

    print("\n===== REGISTROS POR CULTIVO =====")
    gold_exportaciones_regionales.groupBy("cultivo") \
        .count() \
        .orderBy("cultivo") \
        .show(truncate=False)

    print("\n===== MUESTRA =====")
    gold_exportaciones_regionales.orderBy(
        "cultivo", "anio", "numero_mes"
    ).show(15, truncate=False)

    print("PROGRESO: 100% - Transformacion completada")

print("===== PASO 27 FINALIZADO =====")

===== PASO 27: GOLD EXPORTACIONES REGIONALES =====
PROGRESO: 0% - Leyendo Silver
PROGRESO: 25% - Verificando indicadores
Registros con indicador/unidad no compatible: 0
Grupos duplicados: 0
Valores negativos: 0
PROGRESO: 50% - Construyendo tabla Gold
PROGRESO: 75% - Validando estructura

===== ESQUEMA GOLD =====
root
 |-- cultivo: string (nullable = true)
 |-- anio: long (nullable = true)
 |-- numero_mes: long (nullable = true)
 |-- volumen_exportado_t: double (nullable = true)
 |-- valor_fob_miles_usd: double (nullable = true)
 |-- precio_fob_usd_kg: double (nullable = true)
 |-- periodo: date (nullable = true)


===== REGISTROS GOLD =====
Total: 720

===== REGISTROS POR CULTIVO =====
+---------+-----+
|cultivo  |count|
+---------+-----+
|ALCACHOFA|144  |
|ARANDANO |144  |
|ESPARRAGO|144  |
|PALTA    |144  |
|UVA      |144  |
+---------+-----+


===== MUESTRA =====
+---------+----+----------+-------------------+-------------------+------------------+----------+
|cultivo  |anio|numero_

In [8]:
from pyspark.sql import functions as F

print("===== PASO 28: VALIDACION GOLD EXPORTACIONES =====")

print("\nPROGRESO: 20% - Comprobando registros")

total_gold = gold_exportaciones_regionales.count()

print("Registros Gold:", total_gold)

# Verificar que cada cultivo y mes sea unico
duplicados_gold = (
    gold_exportaciones_regionales
    .groupBy("cultivo", "anio", "numero_mes")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Claves duplicadas:", duplicados_gold)

print("\nPROGRESO: 40% - Validando volumen exportado")

volumen_silver = (
    exportaciones_filtradas
    .filter(F.col("indicador") == "VOLUMEN")
    .agg(F.sum("valor").alias("total"))
    .first()["total"]
)

volumen_gold = (
    gold_exportaciones_regionales
    .agg(F.sum("volumen_exportado_t").alias("total"))
    .first()["total"]
)

print("Volumen Silver (t):", volumen_silver)
print("Volumen Gold (t):", volumen_gold)

print("\nPROGRESO: 60% - Validando valor FOB")

fob_silver = (
    exportaciones_filtradas
    .filter(F.col("indicador") == "VALOR_FOB")
    .agg(F.sum("valor").alias("total"))
    .first()["total"]
)

fob_gold = (
    gold_exportaciones_regionales
    .agg(F.sum("valor_fob_miles_usd").alias("total"))
    .first()["total"]
)

print("Valor FOB Silver (miles USD):", fob_silver)
print("Valor FOB Gold (miles USD):", fob_gold)

print("\nPROGRESO: 80% - Validando precio FOB")

# El precio FOB no debe sumarse.
# Comparamos cada valor mensual de Silver con Gold.
precios_silver = (
    exportaciones_filtradas
    .filter(F.col("indicador") == "PRECIO_FOB")
    .select(
        "cultivo",
        "anio",
        "numero_mes",
        F.col("valor").alias("precio_silver")
    )
)

precios_gold = gold_exportaciones_regionales.select(
    "cultivo",
    "anio",
    "numero_mes",
    F.col("precio_fob_usd_kg").alias("precio_gold")
)

diferencias_precios = (
    precios_silver
    .join(
        precios_gold,
        ["cultivo", "anio", "numero_mes"],
        "full_outer"
    )
    .filter(
        F.col("precio_silver").isNull() |
        F.col("precio_gold").isNull() |
        (
            F.abs(
                F.col("precio_silver") - F.col("precio_gold")
            ) > 0.000001
        )
    )
    .count()
)

print("Precios FOB diferentes:", diferencias_precios)

# Comprobar totales con tolerancia numerica
diferencia_volumen = abs(
    (volumen_silver or 0) - (volumen_gold or 0)
)

diferencia_fob = abs(
    (fob_silver or 0) - (fob_gold or 0)
)

print("\n===== RESULTADO DE VALIDACION =====")

print("Diferencia volumen:", diferencia_volumen)
print("Diferencia valor FOB:", diferencia_fob)

if (
    duplicados_gold == 0
    and diferencia_volumen < 0.0001
    and diferencia_fob < 0.0001
    and diferencias_precios == 0
):
    print("VALIDACION PRINCIPAL: CORRECTA")
else:
    print("ATENCION: REVISAR RESULTADOS")

print("\nPROGRESO: 100% - Validacion finalizada")
print("===== PASO 28 FINALIZADO =====")

===== PASO 28: VALIDACION GOLD EXPORTACIONES =====

PROGRESO: 20% - Comprobando registros
Registros Gold: 720
Claves duplicadas: 0

PROGRESO: 40% - Validando volumen exportado
Volumen Silver (t): 3259152.4587402595
Volumen Gold (t): 3259152.4587402595

PROGRESO: 60% - Validando valor FOB
Valor FOB Silver (miles USD): 10555010.649790995
Valor FOB Gold (miles USD): 10555010.649790999

PROGRESO: 80% - Validando precio FOB
Precios FOB diferentes: 79

===== RESULTADO DE VALIDACION =====
Diferencia volumen: 0.0
Diferencia valor FOB: 3.725290298461914e-09
ATENCION: REVISAR RESULTADOS

PROGRESO: 100% - Validacion finalizada
===== PASO 28 FINALIZADO =====


In [9]:
from pyspark.sql import functions as F

print("===== PASO 28.1: INVESTIGAR PRECIOS FOB =====")
print("PROGRESO: 0% - Comparando precios")

comparacion_precios = (
    precios_gold
    .join(
        precios_silver,
        ["cultivo", "anio", "numero_mes"],
        "full_outer"
    )
)

print("\nPROGRESO: 30% - Identificando casos")

# Meses sin precio en ambas capas
sin_precio_ambas = comparacion_precios.filter(
    F.col("precio_silver").isNull() &
    F.col("precio_gold").isNull()
).count()

# Precio presente en Silver, ausente en Gold
perdidos_gold = comparacion_precios.filter(
    F.col("precio_silver").isNotNull() &
    F.col("precio_gold").isNull()
).count()

# Precio ausente en Silver, presente en Gold
nuevos_gold = comparacion_precios.filter(
    F.col("precio_silver").isNull() &
    F.col("precio_gold").isNotNull()
).count()

# Precios diferentes cuando ambos existen
diferentes_reales = comparacion_precios.filter(
    F.col("precio_silver").isNotNull() &
    F.col("precio_gold").isNotNull() &
    (
        F.abs(
            F.col("precio_silver") - F.col("precio_gold")
        ) > 0.000001
    )
).count()

print("\n===== RESULTADOS =====")
print("Sin precio en ambas capas:", sin_precio_ambas)
print("Precios perdidos en Gold:", perdidos_gold)
print("Precios nuevos en Gold:", nuevos_gold)
print("Precios realmente diferentes:", diferentes_reales)

print("\nPROGRESO: 70% - Mostrando ejemplos")

print("\n===== MESES SIN PRECIO FOB =====")
comparacion_precios.filter(
    F.col("precio_silver").isNull() &
    F.col("precio_gold").isNull()
).orderBy(
    "cultivo", "anio", "numero_mes"
).show(15, truncate=False)

print("\n===== PRECIOS REALMENTE DIFERENTES =====")
comparacion_precios.filter(
    F.col("precio_silver").isNotNull() &
    F.col("precio_gold").isNotNull() &
    (
        F.abs(
            F.col("precio_silver") - F.col("precio_gold")
        ) > 0.000001
    )
).show(15, truncate=False)

print("\nPROGRESO: 100% - Revision finalizada")
print("===== PASO 28.1 FINALIZADO =====")

===== PASO 28.1: INVESTIGAR PRECIOS FOB =====
PROGRESO: 0% - Comparando precios

PROGRESO: 30% - Identificando casos



===== RESULTADOS =====
Sin precio en ambas capas: 79
Precios perdidos en Gold: 0
Precios nuevos en Gold: 0
Precios realmente diferentes: 0

PROGRESO: 70% - Mostrando ejemplos

===== MESES SIN PRECIO FOB =====
+--------+----+----------+-----------+-------------+
|cultivo |anio|numero_mes|precio_gold|precio_silver|
+--------+----+----------+-----------+-------------+
|ARANDANO|2012|4         |NULL       |NULL         |
|ARANDANO|2012|5         |NULL       |NULL         |
|ARANDANO|2012|6         |NULL       |NULL         |
|ARANDANO|2012|7         |NULL       |NULL         |
|ARANDANO|2012|8         |NULL       |NULL         |
|ARANDANO|2017|4         |NULL       |NULL         |
|ARANDANO|2017|5         |NULL       |NULL         |
|ARANDANO|2020|4         |NULL       |NULL         |
|PALTA   |2012|1         |NULL       |NULL         |
|PALTA   |2012|3         |NULL       |NULL         |
|PALTA   |2012|10        |NULL       |NULL         |
|PALTA   |2012|12        |NULL       |NULL      

In [10]:
from pyspark.sql import functions as F

print("===== PASO 28.2: VALIDACION FINAL EXPORTACIONES =====")
print("PROGRESO: 0% - Preparando comprobaciones")

# Comparar cada indicador Silver con su equivalente Gold.
indicadores = {
    "VOLUMEN": "volumen_exportado_t",
    "VALOR_FOB": "valor_fob_miles_usd",
    "PRECIO_FOB": "precio_fob_usd_kg"
}

errores_totales = 0

for numero, (indicador, columna_gold) in enumerate(
    indicadores.items(), 1
):
    origen = (
        exportaciones_filtradas
        .filter(F.col("indicador") == indicador)
        .select(
            "cultivo",
            "anio",
            "numero_mes",
            F.col("valor").alias("valor_silver")
        )
    )

    destino = gold_exportaciones_regionales.select(
        "cultivo",
        "anio",
        "numero_mes",
        F.col(columna_gold).alias("valor_gold")
    )

    comparacion = origen.join(
        destino,
        ["cultivo", "anio", "numero_mes"],
        "full_outer"
    )

    diferencias = comparacion.filter(
        (
            F.col("valor_silver").isNull()
            & F.col("valor_gold").isNotNull()
        )
        |
        (
            F.col("valor_silver").isNotNull()
            & F.col("valor_gold").isNull()
        )
        |
        (
            F.col("valor_silver").isNotNull()
            & F.col("valor_gold").isNotNull()
            & (
                F.abs(
                    F.col("valor_silver") - F.col("valor_gold")
                ) > 0.000001
            )
        )
    ).count()

    errores_totales += diferencias

    print(f"{indicador}: {diferencias} diferencias")
    print(
        f"PROGRESO: {numero / len(indicadores) * 90:.0f}%",
        flush=True
    )

duplicados_finales = (
    gold_exportaciones_regionales
    .groupBy("cultivo", "anio", "numero_mes")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\nClaves duplicadas:", duplicados_finales)
print("Total de diferencias:", errores_totales)

if errores_totales == 0 and duplicados_finales == 0:
    exportaciones_gold_validada = True
    print("\nVALIDACION FINAL: CORRECTA")
else:
    exportaciones_gold_validada = False
    print("\nATENCION: VALIDACION NO SUPERADA")

print("PROGRESO: 100%")
print("===== PASO 28.2 FINALIZADO =====")

===== PASO 28.2: VALIDACION FINAL EXPORTACIONES =====
PROGRESO: 0% - Preparando comprobaciones
VOLUMEN: 0 diferencias
PROGRESO: 30%
VALOR_FOB: 0 diferencias
PROGRESO: 60%
PRECIO_FOB: 0 diferencias
PROGRESO: 90%

Claves duplicadas: 0
Total de diferencias: 0

VALIDACION FINAL: CORRECTA
PROGRESO: 100%
===== PASO 28.2 FINALIZADO =====


## Almacenamiento Gold: exportaciones regionales

Se almacena la tabla analítica de exportaciones regionales
en Hadoop HDFS, utilizando archivos Parquet.

La tabla contiene información mensual de espárrago,
alcachofa, arándano, palta y uva para el periodo 2012–2023.

Se conservan los indicadores de volumen exportado en toneladas,
valor FOB en miles de USD y precio FOB en USD/kg.

Los registros sin información disponible mantienen valores nulos.
La tabla representa exportaciones regionales y no exportaciones
exclusivas de Danper.

In [11]:
print("===== PASO 29: GUARDAR EXPORTACIONES GOLD =====")

if not globals().get("exportaciones_gold_validada", False):
    print("ERROR: Primero debe superarse la validacion del Paso 28.2.")

else:
    ruta_exportaciones_gold = (
        f"{BASE_GOLD}/exportaciones_regionales"
    )

    print("PROGRESO: 0% - Comprobando destino")

    # Acceder al sistema HDFS
    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_exportaciones_gold
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La tabla Gold ya existe.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_exportaciones_gold)

    else:
        print("PROGRESO: 25% - Guardando en formato Parquet")

        (
            gold_exportaciones_regionales
            .coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_exportaciones_gold)
        )

        print("PROGRESO: 60% - Leyendo datos guardados")

        # Verificar que la tabla pueda recuperarse de HDFS
        exportaciones_guardadas = spark.read.parquet(
            ruta_exportaciones_gold
        )

        registros_guardados = exportaciones_guardadas.count()

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print(
            "Columnas guardadas:",
            len(exportaciones_guardadas.columns)
        )

        print("\n===== REGISTROS POR CULTIVO =====")

        exportaciones_guardadas.groupBy("cultivo") \
            .count() \
            .orderBy("cultivo") \
            .show(truncate=False)

        if (
            registros_guardados == 720
            and len(exportaciones_guardadas.columns) == 7
        ):
            print("\nALMACENAMIENTO GOLD: CORRECTO")
        else:
            print("\nATENCION: Revisar los datos almacenados")

        print("PROGRESO: 100% - Verificacion finalizada")

print("===== PASO 29 FINALIZADO =====")

===== PASO 29: GUARDAR EXPORTACIONES GOLD =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando en formato Parquet
PROGRESO: 60% - Leyendo datos guardados

===== VERIFICACION HDFS =====
Registros guardados: 720
Columnas guardadas: 7

===== REGISTROS POR CULTIVO =====
+---------+-----+
|cultivo  |count|
+---------+-----+
|ALCACHOFA|144  |
|ARANDANO |144  |
|ESPARRAGO|144  |
|PALTA    |144  |
|UVA      |144  |
+---------+-----+


ALMACENAMIENTO GOLD: CORRECTO
PROGRESO: 100% - Verificacion finalizada
===== PASO 29 FINALIZADO =====


## Transformación Gold: exportaciones de Danper

Se agrupan los registros de exportaciones de Danper por país
de destino y periodo mensual.

Se calculan el valor FOB total en USD y el número de registros
aduaneros presentes en la fuente.

Los resultados no se desagregan por cultivo, porque los
archivos Silver utilizados no contienen esa información.

El año 2026 puede representar un periodo parcial, por lo
que debe interpretarse con precaución.

In [12]:
from pyspark.sql import functions as F

print("===== PASO 30: GOLD EXPORTACIONES DANPER =====")
print("PROGRESO: 0% - Leyendo datos Silver", flush=True)

exportaciones_danper = spark.read.parquet(
    f"{BASE_SILVER}/exportaciones/danper"
)

print("Registros Silver:", exportaciones_danper.count())

print("\nPROGRESO: 20% - Verificando campos", flush=True)

# Comprobar registros sin campos indispensables.
registros_invalidos = exportaciones_danper.filter(
    F.col("periodo").isNull()
    | F.col("pais").isNull()
    | (F.trim(F.col("pais")) == "")
    | F.col("fob_usd").isNull()
    | (F.col("fob_usd") < 0)
)

cantidad_invalidos = registros_invalidos.count()

print("Registros con campos invalidos:", cantidad_invalidos)

print("\nPROGRESO: 40% - Verificando exportadores", flush=True)

exportaciones_danper.groupBy("exportador") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(15, truncate=False)

print("\nPROGRESO: 60% - Revisando cobertura temporal", flush=True)

exportaciones_danper.groupBy("anio") \
    .agg(
        F.count("*").alias("registros"),
        F.sum("fob_usd").alias("fob_total_usd"),
        F.countDistinct("numero_mes").alias("meses_distintos")
    ) \
    .orderBy("anio") \
    .show(truncate=False)

if cantidad_invalidos > 0:
    print("\nATENCION: Revisar registros invalidos.")
    registros_invalidos.show(10, truncate=False)

else:
    print("\nPROGRESO: 75% - Construyendo Gold", flush=True)

    gold_exportaciones_danper = (
        exportaciones_danper
        .withColumn(
            "pais",
            F.upper(F.trim(F.col("pais")))
        )
        .groupBy("pais", "periodo")
        .agg(
            F.sum("fob_usd").alias("valor_fob_usd"),
            F.count("*").alias("cantidad_registros_aduaneros")
        )
        .withColumn("anio", F.year("periodo"))
        .withColumn("mes_numero", F.month("periodo"))
    )

    print("\n===== ESQUEMA GOLD =====")
    gold_exportaciones_danper.printSchema()

    print("\n===== TOTAL GOLD =====")
    print("Registros:", gold_exportaciones_danper.count())

    print("\n===== MUESTRA GOLD =====")
    gold_exportaciones_danper.orderBy(
        "periodo", "pais"
    ).show(15, truncate=False)

    print("\nPROGRESO: 100% - Transformacion completada")

print("===== PASO 30 FINALIZADO =====")

===== PASO 30: GOLD EXPORTACIONES DANPER =====
PROGRESO: 0% - Leyendo datos Silver
Registros Silver: 1124

PROGRESO: 20% - Verificando campos
Registros con campos invalidos: 0

PROGRESO: 40% - Verificando exportadores
+----------------------+-----+
|exportador            |count|
+----------------------+-----+
|DANPER TRUJILLO S.A.C.|1124 |
+----------------------+-----+


PROGRESO: 60% - Revisando cobertura temporal
+----+---------+--------------------+---------------+
|anio|registros|fob_total_usd       |meses_distintos|
+----+---------+--------------------+---------------+
|2024|444      |2.7603960728000003E8|12             |
|2025|542      |3.2994685796E8      |12             |
|2026|138      |5.243515171E7       |5              |
+----+---------+--------------------+---------------+


PROGRESO: 75% - Construyendo Gold

===== ESQUEMA GOLD =====
root
 |-- pais: string (nullable = true)
 |-- periodo: date (nullable = true)
 |-- valor_fob_usd: double (nullable = true)
 |-- cantidad_reg

In [13]:
from pyspark.sql import functions as F

print("===== PASO 31: VALIDACION GOLD EXPORTACIONES DANPER =====")
print("PROGRESO: 0% - Iniciando validaciones", flush=True)

# 1. Verificar cantidad de registros
print("\nPROGRESO: 20% - Comparando registros", flush=True)

registros_silver = exportaciones_danper.count()

registros_gold = (
    gold_exportaciones_danper
    .agg(F.sum("cantidad_registros_aduaneros").alias("total"))
    .first()["total"]
)

print("Registros Silver:", registros_silver)
print("Registros representados en Gold:", registros_gold)

# 2. Verificar valores FOB
print("\nPROGRESO: 40% - Comparando valores FOB", flush=True)

fob_silver = (
    exportaciones_danper
    .agg(F.sum("fob_usd").alias("total"))
    .first()["total"]
)

fob_gold = (
    gold_exportaciones_danper
    .agg(F.sum("valor_fob_usd").alias("total"))
    .first()["total"]
)

diferencia_fob = abs(fob_silver - fob_gold)

print("FOB Silver (USD):", fob_silver)
print("FOB Gold (USD):", fob_gold)
print("Diferencia absoluta:", diferencia_fob)

# 3. Verificar claves duplicadas
print("\nPROGRESO: 60% - Comprobando unicidad", flush=True)

duplicados_gold = (
    gold_exportaciones_danper
    .groupBy("pais", "periodo")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Claves duplicadas:", duplicados_gold)

# 4. Comprobar valores por año
print("\nPROGRESO: 80% - Revisando resultados anuales", flush=True)

gold_exportaciones_danper.groupBy("anio") \
    .agg(
        F.sum("valor_fob_usd").alias("fob_total_usd"),
        F.sum("cantidad_registros_aduaneros").alias("registros")
    ) \
    .orderBy("anio") \
    .show(truncate=False)

# 5. Resultado final
print("\n===== RESULTADO FINAL =====")

# Tolerancia relativa para manejar pequeñas diferencias
# de precisión en números decimales.
tolerancia_fob = max(0.01, abs(fob_silver) * 1e-10)

if (
    registros_silver == registros_gold
    and diferencia_fob <= tolerancia_fob
    and duplicados_gold == 0
):
    danper_gold_validada = True
    print("VALIDACION FINAL: CORRECTA")
else:
    danper_gold_validada = False
    print("ATENCION: REVISAR RESULTADOS")

print("PROGRESO: 100% - Validacion completada")
print("===== PASO 31 FINALIZADO =====")

===== PASO 31: VALIDACION GOLD EXPORTACIONES DANPER =====
PROGRESO: 0% - Iniciando validaciones

PROGRESO: 20% - Comparando registros
Registros Silver: 1124
Registros representados en Gold: 1124

PROGRESO: 40% - Comparando valores FOB
FOB Silver (USD): 658421616.95
FOB Gold (USD): 658421616.9499998
Diferencia absoluta: 2.384185791015625e-07

PROGRESO: 60% - Comprobando unicidad
Claves duplicadas: 0

PROGRESO: 80% - Revisando resultados anuales
+----+-------------------+---------+
|anio|fob_total_usd      |registros|
+----+-------------------+---------+
|2024|2.760396072799999E8|444      |
|2025|3.299468579599999E8|542      |
|2026|5.243515170999999E7|138      |
+----+-------------------+---------+


===== RESULTADO FINAL =====
VALIDACION FINAL: CORRECTA
PROGRESO: 100% - Validacion completada
===== PASO 31 FINALIZADO =====


## Almacenamiento Gold: exportaciones de Danper

Se almacena la información consolidada de exportaciones de
Danper Trujillo S.A.C. en formato Parquet dentro de Hadoop HDFS.

La tabla contiene el valor FOB en dólares y la cantidad de
registros aduaneros agrupados por país de destino y mes.

Se conserva la información de los años 2024, 2025 y 2026.
El año 2026 presenta información parcial, por lo que no debe
compararse directamente con los años completos.

Esta tabla no contiene información desagregada por cultivo.

In [14]:
print("===== PASO 32: GUARDAR EXPORTACIONES DANPER EN HDFS =====")

if not globals().get("danper_gold_validada", False):
    print("ERROR: Primero debe superarse la validacion del Paso 31.")

else:
    ruta_danper_gold = f"{BASE_GOLD}/exportaciones_danper"

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_danper_gold
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La ruta Gold ya existe.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_danper_gold)

    else:
        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            gold_exportaciones_danper
            .coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_danper_gold)
        )

        print("PROGRESO: 60% - Verificando HDFS", flush=True)

        danper_guardada = spark.read.parquet(ruta_danper_gold)

        registros_guardados = danper_guardada.count()
        columnas_guardadas = len(danper_guardada.columns)

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print("\n===== REGISTROS POR ANIO =====")

        danper_guardada.groupBy("anio") \
            .agg(
                F.count("*").alias("filas_gold"),
                F.sum("cantidad_registros_aduaneros").alias(
                    "registros_aduaneros"
                ),
                F.sum("valor_fob_usd").alias("fob_total_usd")
            ) \
            .orderBy("anio") \
            .show(truncate=False)

        registros_representados = (
            danper_guardada
            .agg(
                F.sum("cantidad_registros_aduaneros").alias("total")
            )
            .first()["total"]
        )

        if (
            registros_guardados == 495
            and columnas_guardadas == 6
            and registros_representados == 1124
        ):
            print("\nALMACENAMIENTO GOLD: CORRECTO")
        else:
            print("\nATENCION: Revisar resultados almacenados")

        print("PROGRESO: 100% - Verificacion finalizada", flush=True)

print("===== PASO 32 FINALIZADO =====")

===== PASO 32: GUARDAR EXPORTACIONES DANPER EN HDFS =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Verificando HDFS

===== VERIFICACION HDFS =====
Registros guardados: 495
Columnas guardadas: 6

===== REGISTROS POR ANIO =====
+----+----------+-------------------+-------------------+
|anio|filas_gold|registros_aduaneros|fob_total_usd      |
+----+----------+-------------------+-------------------+
|2024|204       |444                |2.760396072799999E8|
|2025|223       |542                |3.299468579599999E8|
|2026|68        |138                |5.243515170999999E7|
+----+----------+-------------------+-------------------+


ALMACENAMIENTO GOLD: CORRECTO
PROGRESO: 100% - Verificacion finalizada
===== PASO 32 FINALIZADO =====


## Transformación Gold: actividad agroindustrial

Se transforma la información histórica de actividad
agroindustrial de La Libertad para espárrago y alcachofa.

Los datos se organizan por cultivo, año, mes, proceso
y presentación, conservando la cantidad registrada
en toneladas.

Se mantienen separados los procesos de ingreso,
producción, utilización y venta para evitar duplicar
cantidades al realizar análisis.

Los resultados permiten evaluar las tendencias
agroindustriales y apoyar el análisis de abastecimiento
y comercialización del proyecto Danper.

In [15]:
from pyspark.sql import functions as F

print("===== PASO 33: GOLD AGROINDUSTRIA =====")
print("PROGRESO: 0% - Leyendo Silver", flush=True)

agroindustria = spark.read.parquet(
    f"{BASE_SILVER}/agroindustria"
)

print("Registros Silver:", agroindustria.count())

print("\nPROGRESO: 20% - Preparando datos", flush=True)

agroindustria_filtrada = (
    agroindustria
    .withColumn("cultivo", F.upper(F.trim("cultivo")))
    .withColumn("proceso", F.upper(F.trim("proceso")))
    .withColumn(
        "presentacion",
        F.upper(F.trim("presentacion"))
    )
    .filter(F.col("cultivo").isin("ESPARRAGO", "ALCACHOFA"))
)

print("Registros filtrados:", agroindustria_filtrada.count())

print("\nPROGRESO: 40% - Comprobando calidad", flush=True)

invalidos_agro = agroindustria_filtrada.filter(
    F.col("anio").isNull()
    | (~F.col("numero_mes").between(1, 12))
    | F.col("proceso").isNull()
    | F.col("presentacion").isNull()
    | F.col("cantidad_t").isNull()
    | (F.col("cantidad_t") < 0)
)

cantidad_invalidos_agro = invalidos_agro.count()

print("Registros invalidos:", cantidad_invalidos_agro)

claves_agro = [
    "cultivo",
    "anio",
    "numero_mes",
    "proceso",
    "presentacion"
]

duplicados_agro = (
    agroindustria_filtrada
    .groupBy(*claves_agro)
    .count()
    .filter(F.col("count") > 1)
)

cantidad_duplicados_agro = duplicados_agro.count()

print("Grupos con claves repetidas:", cantidad_duplicados_agro)

if cantidad_invalidos_agro > 0 or cantidad_duplicados_agro > 0:

    print("\nATENCION: Revisar registros antes de crear Gold.")

    print("\nRegistros invalidos:")
    invalidos_agro.show(10, truncate=False)

    print("\nClaves repetidas:")
    duplicados_agro.show(10, truncate=False)

else:

    print("\nPROGRESO: 60% - Construyendo Gold", flush=True)

    gold_agroindustria_mensual = (
        agroindustria_filtrada
        .groupBy(*claves_agro)
        .agg(
            F.sum("cantidad_t").alias("cantidad_t")
        )
        .withColumn(
            "periodo",
            F.make_date(
                F.col("anio").cast("int"),
                F.col("numero_mes").cast("int"),
                F.lit(1)
            )
        )
    )

    print("\nPROGRESO: 80% - Revisando Gold", flush=True)

    print("\n===== ESQUEMA GOLD =====")
    gold_agroindustria_mensual.printSchema()

    print("\n===== TOTAL GOLD =====")
    print("Registros:", gold_agroindustria_mensual.count())

    print("\n===== REGISTROS POR PROCESO =====")

    gold_agroindustria_mensual.groupBy(
        "cultivo", "proceso"
    ).count().orderBy(
        "cultivo", "proceso"
    ).show(truncate=False)

    print("\n===== MUESTRA GOLD =====")

    gold_agroindustria_mensual.orderBy(
        "cultivo",
        "anio",
        "numero_mes",
        "proceso",
        "presentacion"
    ).show(15, truncate=False)

    print("\nPROGRESO: 100% - Transformacion completada")

print("===== PASO 33 FINALIZADO =====")

===== PASO 33: GOLD AGROINDUSTRIA =====
PROGRESO: 0% - Leyendo Silver
Registros Silver: 3134

PROGRESO: 20% - Preparando datos
Registros filtrados: 3134

PROGRESO: 40% - Comprobando calidad
Registros invalidos: 0
Grupos con claves repetidas: 0

PROGRESO: 60% - Construyendo Gold

PROGRESO: 80% - Revisando Gold

===== ESQUEMA GOLD =====
root
 |-- cultivo: string (nullable = true)
 |-- anio: long (nullable = true)
 |-- numero_mes: long (nullable = true)
 |-- proceso: string (nullable = true)
 |-- presentacion: string (nullable = true)
 |-- cantidad_t: double (nullable = true)
 |-- periodo: date (nullable = true)


===== TOTAL GOLD =====
Registros: 3134

===== REGISTROS POR PROCESO =====
+---------+-----------+-----+
|cultivo  |proceso    |count|
+---------+-----------+-----+
|ALCACHOFA|INGRESO    |180  |
|ALCACHOFA|PRODUCCION |540  |
|ALCACHOFA|UTILIZACION|180  |
|ALCACHOFA|VENTA      |540  |
|ESPARRAGO|INGRESO    |180  |
|ESPARRAGO|PRODUCCION |662  |
|ESPARRAGO|UTILIZACION|180  |
|ESPARR

In [16]:
from pyspark.sql import functions as F

print("===== PASO 34: VALIDACION GOLD AGROINDUSTRIA =====")
print("PROGRESO: 0% - Iniciando validaciones", flush=True)

# 1. Comparar la cantidad de registros
print("\nPROGRESO: 20% - Comprobando registros", flush=True)

registros_silver_agro = agroindustria_filtrada.count()
registros_gold_agro = gold_agroindustria_mensual.count()

print("Registros Silver:", registros_silver_agro)
print("Registros Gold:", registros_gold_agro)

# 2. Comprobar unicidad
print("\nPROGRESO: 40% - Comprobando claves", flush=True)

claves_agro = [
    "cultivo", "anio", "numero_mes",
    "proceso", "presentacion"
]

duplicados_gold_agro = (
    gold_agroindustria_mensual
    .groupBy(*claves_agro)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Claves duplicadas:", duplicados_gold_agro)

# 3. Comparar toneladas por cultivo y proceso
print("\nPROGRESO: 60% - Comparando cantidades", flush=True)

totales_silver = (
    agroindustria_filtrada
    .groupBy("cultivo", "proceso")
    .agg(F.sum("cantidad_t").alias("total_silver_t"))
)

totales_gold = (
    gold_agroindustria_mensual
    .groupBy("cultivo", "proceso")
    .agg(F.sum("cantidad_t").alias("total_gold_t"))
)

comparacion = (
    totales_silver
    .join(totales_gold, ["cultivo", "proceso"], "full_outer")
    .withColumn(
        "diferencia",
        F.abs(
            F.col("total_silver_t") -
            F.col("total_gold_t")
        )
    )
)

comparacion.orderBy(
    "cultivo", "proceso"
).show(truncate=False)

# 4. Detectar diferencias significativas
diferencias_agro = comparacion.filter(
    F.col("total_silver_t").isNull()
    | F.col("total_gold_t").isNull()
    | (F.col("diferencia") > 0.0001)
).count()

print("\nPROGRESO: 80% - Resultado final", flush=True)

print("Grupos con diferencias:", diferencias_agro)

if (
    registros_silver_agro == registros_gold_agro
    and duplicados_gold_agro == 0
    and diferencias_agro == 0
):
    agroindustria_gold_validada = True
    print("\nVALIDACION FINAL: CORRECTA")
else:
    agroindustria_gold_validada = False
    print("\nATENCION: REVISAR RESULTADOS")

print("\nPROGRESO: 100% - Validacion completada")
print("===== PASO 34 FINALIZADO =====")

===== PASO 34: VALIDACION GOLD AGROINDUSTRIA =====
PROGRESO: 0% - Iniciando validaciones

PROGRESO: 20% - Comprobando registros
Registros Silver: 3134
Registros Gold: 3134

PROGRESO: 40% - Comprobando claves
Claves duplicadas: 0

PROGRESO: 60% - Comparando cantidades
+---------+-----------+------------------+------------------+----------------------+
|cultivo  |proceso    |total_silver_t    |total_gold_t      |diferencia            |
+---------+-----------+------------------+------------------+----------------------+
|ALCACHOFA|INGRESO    |1106802.274466382 |1106802.2744663819|2.3283064365386963E-10|
|ALCACHOFA|PRODUCCION |362601.865508683  |362601.86550868314|1.1641532182693481E-10|
|ALCACHOFA|UTILIZACION|1126123.0471030714|1126123.0471030723|9.313225746154785E-10 |
|ALCACHOFA|VENTA      |389810.0880407832 |389810.0880407833 |1.1641532182693481E-10|
|ESPARRAGO|INGRESO    |1177213.2361989398|1177213.2361989394|4.6566128730773926E-10|
|ESPARRAGO|PRODUCCION |600307.946127598  |600307.946

## Almacenamiento Gold: actividad agroindustrial

Se almacena la información agroindustrial de espárrago y
alcachofa en Hadoop HDFS, utilizando el formato Parquet.

La tabla contiene indicadores mensuales de ingreso,
producción, utilización y venta, diferenciados por
cultivo, proceso y presentación.

Se conserva la información histórica de 2010 a 2024.

Los procesos se mantienen separados porque representan
etapas diferentes de la actividad agroindustrial.

Antes del almacenamiento se verificaron 3 134 registros,
ausencia de claves duplicadas y conservación de cantidades.

In [17]:
print("===== PASO 35: GUARDAR GOLD AGROINDUSTRIA =====")

if not globals().get("agroindustria_gold_validada", False):
    print("ERROR: Primero debe superarse la validacion del Paso 34.")

else:
    ruta_agro_gold = f"{BASE_GOLD}/agroindustria_mensual"

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_agro_gold
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La ruta Gold ya existe.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_agro_gold)

    else:
        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            gold_agroindustria_mensual
            .coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_agro_gold)
        )

        print("PROGRESO: 60% - Verificando HDFS", flush=True)

        agro_guardada = spark.read.parquet(ruta_agro_gold)

        registros_guardados = agro_guardada.count()
        columnas_guardadas = len(agro_guardada.columns)

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print("\n===== REGISTROS POR CULTIVO Y PROCESO =====")

        agro_guardada.groupBy("cultivo", "proceso") \
            .count() \
            .orderBy("cultivo", "proceso") \
            .show(truncate=False)

        # Verificar que las cantidades almacenadas
        # coincidan con la tabla Gold original.
        total_original = (
            gold_agroindustria_mensual
            .agg(F.sum("cantidad_t").alias("total"))
            .first()["total"]
        )

        total_guardado = (
            agro_guardada
            .agg(F.sum("cantidad_t").alias("total"))
            .first()["total"]
        )

        diferencia = abs(total_original - total_guardado)

        print("\n===== COMPARACION DE CANTIDADES =====")
        print("Total original:", total_original)
        print("Total almacenado:", total_guardado)
        print("Diferencia:", diferencia)

        if (
            registros_guardados == 3134
            and columnas_guardadas == 7
            and diferencia < 0.0001
        ):
            print("\nALMACENAMIENTO GOLD: CORRECTO")
        else:
            print("\nATENCION: Revisar resultados almacenados")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 35 FINALIZADO =====")

===== PASO 35: GUARDAR GOLD AGROINDUSTRIA =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Verificando HDFS

===== VERIFICACION HDFS =====
Registros guardados: 3134
Columnas guardadas: 7

===== REGISTROS POR CULTIVO Y PROCESO =====
+---------+-----------+-----+
|cultivo  |proceso    |count|
+---------+-----------+-----+
|ALCACHOFA|INGRESO    |180  |
|ALCACHOFA|PRODUCCION |540  |
|ALCACHOFA|UTILIZACION|180  |
|ALCACHOFA|VENTA      |540  |
|ESPARRAGO|INGRESO    |180  |
|ESPARRAGO|PRODUCCION |662  |
|ESPARRAGO|UTILIZACION|180  |
|ESPARRAGO|VENTA      |672  |
+---------+-----------+-----+


===== COMPARACION DE CANTIDADES =====
Total original: 6449508.012731543
Total almacenado: 6449508.012731543
Diferencia: 0.0

ALMACENAMIENTO GOLD: CORRECTO
PROGRESO: 100% - Verificacion finalizada
===== PASO 35 FINALIZADO =====


## Transformación Gold: costos agrícolas mensuales

Se prepara una tabla analítica de costos agrícolas a partir
de la información procesada en Silver.

La tabla organiza los valores por región, tipo de costo
y periodo mensual.

Se consideran los conceptos de jornal agrícola, alquiler
de tractor y precio de yunta.

Debido a que la unidad de medida no se encuentra confirmada,
se conserva esta condición y no se realizan conversiones
ni sumas entre conceptos diferentes.

Los resultados permitirán analizar las tendencias de los
costos referenciales agrícolas y apoyar el análisis
agrocomercial del proyecto Danper.

In [18]:
from pyspark.sql import functions as F

print("===== PASO 36: GOLD COSTOS AGRICOLAS =====")
print("PROGRESO: 0% - Leyendo Silver", flush=True)

costos_silver = spark.read.parquet(
    f"{BASE_SILVER}/costos"
)

print("Registros Silver:", costos_silver.count())

print("\nPROGRESO: 20% - Normalizando categorias", flush=True)

costos_preparados = (
    costos_silver
    .withColumn("region", F.upper(F.trim("region")))
    .withColumn("tipo_costo", F.upper(F.trim("tipo_costo")))
    .withColumn("unidad", F.upper(F.trim("unidad")))
)

print("\n===== TIPOS DE COSTO =====")

costos_preparados.groupBy("tipo_costo", "unidad") \
    .count() \
    .orderBy("tipo_costo") \
    .show(truncate=False)

print("\nPROGRESO: 40% - Comprobando calidad", flush=True)

invalidos_costos = costos_preparados.filter(
    F.col("region").isNull()
    | (F.col("region") == "")
    | F.col("tipo_costo").isNull()
    | (F.col("tipo_costo") == "")
    | F.col("anio").isNull()
    | (~F.col("numero_mes").between(1, 12))
    | F.col("valor").isNull()
    | (F.col("valor") < 0)
)

cantidad_invalidos_costos = invalidos_costos.count()

print("Registros invalidos:", cantidad_invalidos_costos)

claves_costos = [
    "region",
    "tipo_costo",
    "anio",
    "numero_mes"
]

duplicados_costos = (
    costos_preparados
    .groupBy(*claves_costos)
    .count()
    .filter(F.col("count") > 1)
)

cantidad_duplicados_costos = duplicados_costos.count()

print("Grupos con claves repetidas:", cantidad_duplicados_costos)

print("\nPROGRESO: 60% - Evaluando resultados", flush=True)

if cantidad_invalidos_costos > 0 or cantidad_duplicados_costos > 0:

    print("\nATENCION: Revisar los datos antes de construir Gold.")

    print("\n===== REGISTROS INVALIDOS =====")
    invalidos_costos.show(10, truncate=False)

    print("\n===== CLAVES REPETIDAS =====")
    duplicados_costos.show(10, truncate=False)

else:

    print("PROGRESO: 75% - Construyendo Gold", flush=True)

    gold_costos_mensuales = (
        costos_preparados
        .select(
            "region",
            "tipo_costo",
            "anio",
            "numero_mes",
            F.col("valor").alias("valor_costo"),
            "unidad"
        )
        .withColumn(
            "periodo",
            F.make_date(
                F.col("anio"),
                F.col("numero_mes"),
                F.lit(1)
            )
        )
    )

    print("\n===== ESQUEMA GOLD =====")
    gold_costos_mensuales.printSchema()

    print("\n===== REGISTROS GOLD =====")
    print("Total:", gold_costos_mensuales.count())

    print("\n===== REGISTROS POR TIPO DE COSTO =====")

    gold_costos_mensuales.groupBy("tipo_costo") \
        .count() \
        .orderBy("tipo_costo") \
        .show(truncate=False)

    print("\n===== MUESTRA GOLD =====")

    gold_costos_mensuales.orderBy(
        "region",
        "tipo_costo",
        "anio",
        "numero_mes"
    ).show(15, truncate=False)

    print("\nPROGRESO: 100% - Transformacion completada")

print("===== PASO 36 FINALIZADO =====")

===== PASO 36: GOLD COSTOS AGRICOLAS =====
PROGRESO: 0% - Leyendo Silver
Registros Silver: 6292

PROGRESO: 20% - Normalizando categorias

===== TIPOS DE COSTO =====
+----------------+-------------+-----+
|tipo_costo      |unidad       |count|
+----------------+-------------+-----+
|ALQUILER_TRACTOR|NO_CONFIRMADA|2275 |
|PRECIO_YUNTA    |NO_CONFIRMADA|1890 |
|VALOR_JORNAL    |NO_CONFIRMADA|2127 |
+----------------+-------------+-----+


PROGRESO: 40% - Comprobando calidad
Registros invalidos: 0
Grupos con claves repetidas: 0

PROGRESO: 60% - Evaluando resultados
PROGRESO: 75% - Construyendo Gold

===== ESQUEMA GOLD =====
root
 |-- region: string (nullable = true)
 |-- tipo_costo: string (nullable = true)
 |-- anio: integer (nullable = true)
 |-- numero_mes: integer (nullable = true)
 |-- valor_costo: double (nullable = true)
 |-- unidad: string (nullable = true)
 |-- periodo: date (nullable = true)


===== REGISTROS GOLD =====
Total: 6292

===== REGISTROS POR TIPO DE COSTO =====
+------

## Validación Gold: costos agrícolas

Se verifica la conservación de los 6 292 registros de costos
agrícolas al pasar de Silver a Gold.

La validación comprueba la unicidad de las claves, la
correspondencia de los valores por región, tipo de costo
y periodo, y la conservación de las unidades originales.

Los conceptos de jornal, tractor y yunta se mantienen
independientes debido a sus distintas características
y unidades aún no confirmadas.

In [19]:
from pyspark.sql import functions as F

print("===== PASO 37: VALIDACION GOLD COSTOS =====")
print("PROGRESO: 0% - Iniciando validacion", flush=True)

# 1. Verificar registros
print("\nPROGRESO: 20% - Comparando registros", flush=True)

registros_silver_costos = costos_preparados.count()
registros_gold_costos = gold_costos_mensuales.count()

print("Registros Silver:", registros_silver_costos)
print("Registros Gold:", registros_gold_costos)

# 2. Verificar claves duplicadas
print("\nPROGRESO: 40% - Verificando unicidad", flush=True)

claves_costos = [
    "region",
    "tipo_costo",
    "anio",
    "numero_mes"
]

duplicados_gold_costos = (
    gold_costos_mensuales
    .groupBy(*claves_costos)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Claves duplicadas:", duplicados_gold_costos)

# 3. Comparar valores individuales de Silver con Gold
print("\nPROGRESO: 60% - Comparando valores", flush=True)

silver_comparacion = costos_preparados.select(
    *claves_costos,
    F.col("valor").alias("valor_silver"),
    F.col("unidad").alias("unidad_silver")
)

gold_comparacion = gold_costos_mensuales.select(
    *claves_costos,
    F.col("valor_costo").alias("valor_gold"),
    F.col("unidad").alias("unidad_gold")
)

comparacion_costos = (
    silver_comparacion
    .join(
        gold_comparacion,
        claves_costos,
        "full_outer"
    )
)

# Detectar diferencias de valor o de unidad.
# eqNullSafe considera equivalentes dos valores NULL.
diferencias_costos = (
    comparacion_costos
    .filter(
        ~F.col("valor_silver").eqNullSafe(F.col("valor_gold"))
        |
        ~F.col("unidad_silver").eqNullSafe(F.col("unidad_gold"))
    )
    .count()
)

print("Valores o unidades diferentes:", diferencias_costos)

# 4. Comprobar que las unidades sigan sin confirmarse
print("\nPROGRESO: 80% - Verificando unidades", flush=True)

gold_costos_mensuales.groupBy("tipo_costo", "unidad") \
    .count() \
    .orderBy("tipo_costo") \
    .show(truncate=False)

# 5. Resultado final
print("\n===== RESULTADO FINAL =====")

if (
    registros_silver_costos == registros_gold_costos
    and duplicados_gold_costos == 0
    and diferencias_costos == 0
):
    costos_gold_validada = True
    print("VALIDACION FINAL: CORRECTA")
else:
    costos_gold_validada = False
    print("ATENCION: REVISAR RESULTADOS")

print("\nPROGRESO: 100% - Validacion completada")
print("===== PASO 37 FINALIZADO =====")

===== PASO 37: VALIDACION GOLD COSTOS =====
PROGRESO: 0% - Iniciando validacion

PROGRESO: 20% - Comparando registros
Registros Silver: 6292
Registros Gold: 6292

PROGRESO: 40% - Verificando unicidad
Claves duplicadas: 0

PROGRESO: 60% - Comparando valores
Valores o unidades diferentes: 0

PROGRESO: 80% - Verificando unidades
+----------------+-------------+-----+
|tipo_costo      |unidad       |count|
+----------------+-------------+-----+
|ALQUILER_TRACTOR|NO_CONFIRMADA|2275 |
|PRECIO_YUNTA    |NO_CONFIRMADA|1890 |
|VALOR_JORNAL    |NO_CONFIRMADA|2127 |
+----------------+-------------+-----+


===== RESULTADO FINAL =====
VALIDACION FINAL: CORRECTA

PROGRESO: 100% - Validacion completada
===== PASO 37 FINALIZADO =====


## Almacenamiento Gold: costos agrícolas

Se almacena la tabla de costos agrícolas mensuales en
Hadoop HDFS utilizando el formato Parquet.

Los datos se organizan por región, tipo de costo y periodo,
conservando los conceptos de jornal agrícola, alquiler
de tractor y precio de yunta.

La tabla contiene 6 292 registros validados y mantiene
la unidad NO_CONFIRMADA, debido a que todavía no se ha
verificado su unidad de medida original.

Los tipos de costos se conservan separados para evitar
comparaciones o agregaciones incorrectas.

In [20]:
from pyspark.sql import functions as F

print("===== PASO 38: GUARDAR GOLD COSTOS =====")

if not globals().get("costos_gold_validada", False):
    print("ERROR: Primero debes superar la validacion del Paso 37.")

else:
    ruta_costos_gold = f"{BASE_GOLD}/costos_mensuales"

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_costos_gold
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La tabla Gold ya existe.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_costos_gold)

    else:
        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            gold_costos_mensuales
            .coalesce(2)
            .write
            .mode("errorifexists")
            .parquet(ruta_costos_gold)
        )

        print("PROGRESO: 60% - Leyendo desde HDFS", flush=True)

        costos_guardados = spark.read.parquet(
            ruta_costos_gold
        )

        registros_guardados = costos_guardados.count()
        columnas_guardadas = len(costos_guardados.columns)

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print("\n===== REGISTROS POR TIPO DE COSTO =====")

        costos_guardados.groupBy("tipo_costo", "unidad") \
            .count() \
            .orderBy("tipo_costo") \
            .show(truncate=False)

        # Comparar valores por cada tipo de costo
        original = (
            gold_costos_mensuales
            .groupBy("tipo_costo")
            .agg(F.sum("valor_costo").alias("total_original"))
        )

        guardado = (
            costos_guardados
            .groupBy("tipo_costo")
            .agg(F.sum("valor_costo").alias("total_guardado"))
        )

        comparacion = (
            original
            .join(guardado, "tipo_costo", "full_outer")
            .withColumn(
                "diferencia",
                F.abs(
                    F.col("total_original") -
                    F.col("total_guardado")
                )
            )
        )

        print("\n===== COMPARACION POR TIPO DE COSTO =====")
        comparacion.show(truncate=False)

        diferencias = comparacion.filter(
            F.col("total_original").isNull()
            | F.col("total_guardado").isNull()
            | (F.col("diferencia") > 0.0001)
        ).count()

        if (
            registros_guardados == 6292
            and columnas_guardadas == 7
            and diferencias == 0
        ):
            print("\nALMACENAMIENTO GOLD: CORRECTO")
        else:
            print("\nATENCION: Revisar resultados almacenados")

        print("PROGRESO: 100% - Verificacion finalizada", flush=True)

print("===== PASO 38 FINALIZADO =====")

===== PASO 38: GUARDAR GOLD COSTOS =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Leyendo desde HDFS



===== VERIFICACION HDFS =====
Registros guardados: 6292
Columnas guardadas: 7

===== REGISTROS POR TIPO DE COSTO =====
+----------------+-------------+-----+
|tipo_costo      |unidad       |count|
+----------------+-------------+-----+
|ALQUILER_TRACTOR|NO_CONFIRMADA|2275 |
|PRECIO_YUNTA    |NO_CONFIRMADA|1890 |
|VALOR_JORNAL    |NO_CONFIRMADA|2127 |
+----------------+-------------+-----+


===== COMPARACION POR TIPO DE COSTO =====
+----------------+------------------+------------------+----------+
|tipo_costo      |total_original    |total_guardado    |diferencia|
+----------------+------------------+------------------+----------+
|ALQUILER_TRACTOR|230610.42999999996|230610.42999999996|0.0       |
|PRECIO_YUNTA    |188832.09999999998|188832.09999999998|0.0       |
|VALOR_JORNAL    |107036.27         |107036.27         |0.0       |
+----------------+------------------+------------------+----------+


ALMACENAMIENTO GOLD: CORRECTO
PROGRESO: 100% - Verificacion finalizada
===== PASO 38 

## Exploración Gold: abastecimiento y precios mayoristas

Se analizan las fuentes Silver de abastecimiento y mercado
de productores para identificar productos, mercados,
fechas y unidades disponibles.

El objetivo es preparar indicadores de ingreso de productos
en toneladas y evolución de precios mayoristas.

Antes de integrar las fuentes, se verificará que las unidades,
productos y periodos sean compatibles.

La ausencia de información se conservará como valor nulo,
sin asumir que representa un ingreso o precio igual a cero.

In [21]:
from pyspark.sql import functions as F

print("===== PASO 39: EXPLORACION DE ABASTECIMIENTO =====")
print("PROGRESO: 0% - Leyendo fuentes Silver", flush=True)

abastecimiento_silver = spark.read.parquet(
    f"{BASE_SILVER}/abastecimiento"
)

mercado_productores_silver = spark.read.parquet(
    f"{BASE_SILVER}/mercado_productores"
)

print("\n===== REGISTROS DISPONIBLES =====")
print("Abastecimiento:", abastecimiento_silver.count())
print("Mercado productores:", mercado_productores_silver.count())

print("\nPROGRESO: 20% - Revisando fechas y mercados", flush=True)

print("\n===== ABASTECIMIENTO: FECHAS Y MERCADOS =====")

abastecimiento_silver.groupBy(
    "fecha_reporte", "mercado"
).count().orderBy(
    "fecha_reporte", "mercado"
).show(30, truncate=False)

print("\n===== MERCADO PRODUCTORES: FECHAS Y MERCADOS =====")

mercado_productores_silver.groupBy("mercado") \
    .agg(
        F.count("*").alias("registros"),
        F.min("fecha_reporte").alias("desde"),
        F.max("fecha_reporte").alias("hasta")
    ).show(truncate=False)

print("\nPROGRESO: 40% - Revisando unidades", flush=True)

print("\n===== UNIDADES DE ABASTECIMIENTO =====")

abastecimiento_silver.groupBy(
    "unidad_medida", "equivalencia_kg"
).count().orderBy(
    "unidad_medida", "equivalencia_kg"
).show(30, truncate=False)

print("\n===== UNIDADES DEL MERCADO DE PRODUCTORES =====")

mercado_productores_silver.groupBy(
    "unidad_medida"
).count().orderBy(
    "unidad_medida"
).show(30, truncate=False)

print("\nPROGRESO: 60% - Revisando productos", flush=True)

print("\n===== PRODUCTOS DE ABASTECIMIENTO =====")

abastecimiento_silver.groupBy("producto") \
    .count().orderBy("producto") \
    .show(100, truncate=False)

print("\n===== PRODUCTOS DEL MERCADO DE PRODUCTORES =====")

mercado_productores_silver.groupBy("categoria") \
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("producto").alias("productos_distintos")
    ).orderBy("categoria").show(50, truncate=False)

print("\nPROGRESO: 80% - Revisando valores faltantes", flush=True)

campos_abastecimiento = [
    "ingreso_hoy_t",
    "precio_hoy_soles",
    "precio_7dias_soles",
    "equivalencia_kg"
]

for campo in campos_abastecimiento:
    nulos = abastecimiento_silver.filter(
        F.col(campo).isNull()
    ).count()
    print(f"Abastecimiento - {campo}: {nulos} nulos")

campos_mercado = [
    "precio_hoy_soles",
    "precio_7dias_soles",
    "variacion_porcentaje"
]

for campo in campos_mercado:
    nulos = mercado_productores_silver.filter(
        F.col(campo).isNull()
    ).count()
    print(f"Mercado productores - {campo}: {nulos} nulos")

print("\nPROGRESO: 100% - Exploracion completada")
print("===== PASO 39 FINALIZADO =====")

===== PASO 39: EXPLORACION DE ABASTECIMIENTO =====
PROGRESO: 0% - Leyendo fuentes Silver

===== REGISTROS DISPONIBLES =====
Abastecimiento: 66
Mercado productores: 756

PROGRESO: 20% - Revisando fechas y mercados

===== ABASTECIMIENTO: FECHAS Y MERCADOS =====
+-------------+---------+-----+
|fecha_reporte|mercado  |count|
+-------------+---------+-----+
|2024-01-31   |GMML_LIMA|2    |
|2024-02-29   |GMML_LIMA|2    |
|2024-03-28   |GMML_LIMA|2    |
|2024-04-30   |GMML_LIMA|2    |
|2024-05-31   |GMML_LIMA|2    |
|2024-06-28   |GMML_LIMA|2    |
|2024-07-31   |GMML_LIMA|2    |
|2024-08-29   |GMML_LIMA|2    |
|2024-09-30   |GMML_LIMA|2    |
|2024-10-31   |GMML_LIMA|2    |
|2024-11-29   |GMML_LIMA|2    |
|2024-12-31   |GMML_LIMA|2    |
|2025-01-31   |GMML_LIMA|2    |
|2025-02-28   |GMML_LIMA|2    |
|2025-03-31   |GMML_LIMA|2    |
|2025-04-30   |GMML_LIMA|2    |
|2025-05-30   |GMML_LIMA|2    |
|2025-06-30   |GMML_LIMA|2    |
|2025-07-31   |GMML_LIMA|2    |
|2025-08-29   |GMML_LIMA|2    |
|202

## Transformación Gold: abastecimiento mayorista

Se prepara una tabla analítica de abastecimiento y precios
mayoristas de espárrago y alcachofa a partir de Silver.

Los registros se organizan por mercado, producto y fecha
del reporte, conservando los indicadores de ingreso
y los precios reportados.

Se mantiene la unidad original de comercialización y
se identifica la disponibilidad de información de ingreso.

Los valores nulos no se reemplazan por cero, debido a
que representan información no disponible en la fuente.

La tabla permitirá analizar el comportamiento del mercado
mayorista como información de apoyo para la planificación
agrocomercial de Danper.

In [22]:
from pyspark.sql import functions as F

print("===== PASO 40: GOLD ABASTECIMIENTO MAYORISTA =====")
print("PROGRESO: 0% - Preparando Silver", flush=True)

abastecimiento_preparado = (
    abastecimiento_silver
    .withColumn("mercado", F.upper(F.trim("mercado")))
    .withColumn("producto", F.upper(F.trim("producto")))
    .withColumn("unidad_medida", F.upper(F.trim("unidad_medida")))
    .withColumn(
        "fecha",
        F.to_date(F.col("fecha_reporte"), "yyyy-MM-dd")
    )
)

print("PROGRESO: 20% - Verificando registros", flush=True)

invalidos_abastecimiento = abastecimiento_preparado.filter(
    F.col("fecha").isNull()
    | F.col("mercado").isNull()
    | (F.col("mercado") == "")
    | F.col("producto").isNull()
    | (F.col("producto") == "")
    | F.col("unidad_medida").isNull()
    | (F.col("unidad_medida") == "")
    | (F.col("precio_hoy_soles") < 0)
    | (F.col("precio_7dias_soles") < 0)
    | (F.col("ingreso_hoy_t") < 0)
    | (F.col("equivalencia_kg") <= 0)
)

cantidad_invalidos = invalidos_abastecimiento.count()

print("Registros invalidos:", cantidad_invalidos)

claves_abastecimiento = [
    "mercado",
    "producto",
    "fecha"
]

duplicados_abastecimiento = (
    abastecimiento_preparado
    .groupBy(*claves_abastecimiento)
    .count()
    .filter(F.col("count") > 1)
)

cantidad_duplicados = duplicados_abastecimiento.count()

print("Grupos con claves duplicadas:", cantidad_duplicados)

print("PROGRESO: 40% - Revisando calidad", flush=True)

abastecimiento_preparado.groupBy(
    "producto", "unidad_medida", "equivalencia_kg"
).count().orderBy(
    "producto"
).show(truncate=False)

if cantidad_invalidos > 0 or cantidad_duplicados > 0:

    print("\nATENCION: Se necesita revisar Silver.")

    invalidos_abastecimiento.show(10, truncate=False)
    duplicados_abastecimiento.show(10, truncate=False)

else:

    print("PROGRESO: 60% - Construyendo Gold", flush=True)

    gold_abastecimiento_mayorista = (
        abastecimiento_preparado
        .select(
            "mercado",
            "producto",
            F.col("fecha").alias("fecha_reporte"),
            "unidad_medida",
            "equivalencia_kg",
            "ingreso_ayer_t",
            "ingreso_hoy_t",
            "promedio_ingreso_7dias_t",
            "promedio_ingreso_4semanas_t",
            "precio_ayer_soles",
            "precio_hoy_soles",
            "precio_7dias_soles",
            "tendencia"
        )
        .withColumn(
            "tiene_dato_ingreso",
            F.col("ingreso_hoy_t").isNotNull()
        )
        .withColumn("anio", F.year("fecha_reporte"))
        .withColumn("mes_numero", F.month("fecha_reporte"))
    )

    print("PROGRESO: 80% - Revisando resultados", flush=True)

    print("\n===== ESQUEMA GOLD =====")
    gold_abastecimiento_mayorista.printSchema()

    print("\n===== REGISTROS GOLD =====")
    print("Total:", gold_abastecimiento_mayorista.count())

    print("\n===== DISPONIBILIDAD DE INGRESOS =====")

    gold_abastecimiento_mayorista.groupBy(
        "producto", "tiene_dato_ingreso"
    ).count().orderBy(
        "producto", "tiene_dato_ingreso"
    ).show(truncate=False)

    print("\n===== MUESTRA =====")
    gold_abastecimiento_mayorista.orderBy(
        "fecha_reporte", "producto"
    ).show(12, truncate=False)

    print("PROGRESO: 100% - Transformacion completada")

print("===== PASO 40 FINALIZADO =====")

===== PASO 40: GOLD ABASTECIMIENTO MAYORISTA =====
PROGRESO: 0% - Preparando Silver
PROGRESO: 20% - Verificando registros
Registros invalidos: 0
Grupos con claves duplicadas: 0
PROGRESO: 40% - Revisando calidad
+---------+-------------+---------------+-----+
|producto |unidad_medida|equivalencia_kg|count|
+---------+-------------+---------------+-----+
|ALCACHOFA|DOCENA       |4.0            |33   |
|ESPARRAGO|PAQUETE      |0.5            |33   |
+---------+-------------+---------------+-----+

PROGRESO: 60% - Construyendo Gold
PROGRESO: 80% - Revisando resultados

===== ESQUEMA GOLD =====
root
 |-- mercado: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- fecha_reporte: date (nullable = true)
 |-- unidad_medida: string (nullable = true)
 |-- equivalencia_kg: double (nullable = true)
 |-- ingreso_ayer_t: double (nullable = true)
 |-- ingreso_hoy_t: double (nullable = true)
 |-- promedio_ingreso_7dias_t: double (nullable = true)
 |-- promedio_ingreso_4semanas_t: dou

## Validación Gold: abastecimiento mayorista

Se verifica la conservación de los 66 registros de
abastecimiento de espárrago y alcachofa.

La validación comprueba las claves por mercado, producto
y fecha, así como los precios, volúmenes de ingreso,
unidades de comercialización y equivalencias en kilogramos.

También se comprueba que los registros sin ingreso
reportado mantengan valores nulos y estén identificados
mediante el campo tiene_dato_ingreso.

Las fechas representan reportes puntuales y no
necesariamente totales mensuales.

In [24]:
from pyspark.sql import functions as F

print("===== PASO 41: VALIDACION GOLD ABASTECIMIENTO =====")
print("PROGRESO: 0% - Iniciando validacion", flush=True)

# 1. Comparar cantidad de registros
print("\nPROGRESO: 20% - Verificando registros", flush=True)

registros_silver = abastecimiento_preparado.count()
registros_gold = gold_abastecimiento_mayorista.count()

print("Registros Silver:", registros_silver)
print("Registros Gold:", registros_gold)

# 2. Comprobar unicidad
print("\nPROGRESO: 40% - Verificando claves", flush=True)

claves = ["mercado", "producto", "fecha_reporte"]

duplicados_gold = (
    gold_abastecimiento_mayorista
    .groupBy(*claves)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Claves duplicadas:", duplicados_gold)

# 3. Comparar las columnas originales, incluyendo NULL
print("\nPROGRESO: 60% - Comparando valores", flush=True)

columnas_comparar = [
    "unidad_medida",
    "equivalencia_kg",
    "ingreso_ayer_t",
    "ingreso_hoy_t",
    "promedio_ingreso_7dias_t",
    "promedio_ingreso_4semanas_t",
    "precio_ayer_soles",
    "precio_hoy_soles",
    "precio_7dias_soles",
    "tendencia"
]

silver_comparacion = abastecimiento_preparado.select(
    "mercado",
    "producto",
    F.col("fecha").alias("fecha_reporte"),
    *columnas_comparar
)

gold_comparacion = gold_abastecimiento_mayorista.select(
    *claves,
    *columnas_comparar
)

# Comparar ambas tablas sin confundir NULL con cero
diferencias = silver_comparacion.exceptAll(gold_comparacion).count()
diferencias += gold_comparacion.exceptAll(silver_comparacion).count()

print("Filas diferentes entre Silver y Gold:", diferencias)

# 4. Verificar el indicador de disponibilidad
print("\nPROGRESO: 80% - Verificando ingresos faltantes", flush=True)

errores_indicador = gold_abastecimiento_mayorista.filter(
    ~F.col("tiene_dato_ingreso").eqNullSafe(
        F.col("ingreso_hoy_t").isNotNull()
    )
).count()

nulos_silver = abastecimiento_preparado.filter(
    F.col("ingreso_hoy_t").isNull()
).count()

nulos_gold = gold_abastecimiento_mayorista.filter(
    F.col("ingreso_hoy_t").isNull()
).count()

print("Ingresos nulos Silver:", nulos_silver)
print("Ingresos nulos Gold:", nulos_gold)
print("Errores en tiene_dato_ingreso:", errores_indicador)

# 5. Resultado final
print("\n===== RESULTADO FINAL =====")

if (
    registros_silver == registros_gold == 66
    and duplicados_gold == 0
    and diferencias == 0
    and nulos_silver == nulos_gold == 28
    and errores_indicador == 0
):
    abastecimiento_gold_validada = True
    print("VALIDACION FINAL: CORRECTA")
else:
    abastecimiento_gold_validada = False
    print("ATENCION: REVISAR RESULTADOS")

print("\nPROGRESO: 100% - Validacion completada")
print("===== PASO 41 FINALIZADO =====")

===== PASO 41: VALIDACION GOLD ABASTECIMIENTO =====
PROGRESO: 0% - Iniciando validacion

PROGRESO: 20% - Verificando registros
Registros Silver: 66
Registros Gold: 66

PROGRESO: 40% - Verificando claves
Claves duplicadas: 0

PROGRESO: 60% - Comparando valores
Filas diferentes entre Silver y Gold: 0

PROGRESO: 80% - Verificando ingresos faltantes
Ingresos nulos Silver: 28
Ingresos nulos Gold: 28
Errores en tiene_dato_ingreso: 0

===== RESULTADO FINAL =====
VALIDACION FINAL: CORRECTA

PROGRESO: 100% - Validacion completada
===== PASO 41 FINALIZADO =====


## Almacenamiento Gold: abastecimiento mayorista

Se almacena la tabla analítica de abastecimiento mayorista
en Hadoop HDFS, utilizando archivos Parquet.

La información corresponde a espárrago y alcachofa del
Gran Mercado Mayorista de Lima y se organiza por producto,
mercado y fecha del reporte.

Se conservan los precios, las unidades de comercialización
y los indicadores de ingreso disponibles.

Los 28 registros sin ingreso reportado mantienen sus
valores nulos. Las fechas representan reportes puntuales,
no acumulados mensuales.

La tabla fue validada antes de su almacenamiento:
66 registros, cero duplicados y cero diferencias
respecto a los valores originales de Silver.

In [25]:
from pyspark.sql import functions as F

print("===== PASO 42: GUARDAR GOLD ABASTECIMIENTO =====")

if not globals().get("abastecimiento_gold_validada", False):
    print("ERROR: Primero debes superar la validacion del Paso 41.")

else:
    ruta_abastecimiento_gold = (
        f"{BASE_GOLD}/abastecimiento_mayorista"
    )

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_abastecimiento_gold
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La ruta Gold ya existe.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_abastecimiento_gold)

    else:
        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            gold_abastecimiento_mayorista
            .coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_abastecimiento_gold)
        )

        print("PROGRESO: 60% - Leyendo desde HDFS", flush=True)

        abastecimiento_guardado = spark.read.parquet(
            ruta_abastecimiento_gold
        )

        registros_guardados = abastecimiento_guardado.count()
        columnas_guardadas = len(abastecimiento_guardado.columns)

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print("\n===== REGISTROS POR PRODUCTO =====")

        abastecimiento_guardado.groupBy("producto") \
            .agg(
                F.count("*").alias("registros"),
                F.sum(
                    F.when(
                        F.col("tiene_dato_ingreso"), 1
                    ).otherwise(0)
                ).alias("con_ingreso"),
                F.sum(
                    F.when(
                        ~F.col("tiene_dato_ingreso"), 1
                    ).otherwise(0)
                ).alias("sin_ingreso")
            ) \
            .orderBy("producto") \
            .show(truncate=False)

        print("\nPROGRESO: 80% - Comparando con Gold original")

        columnas = gold_abastecimiento_mayorista.columns

        diferencias = (
            gold_abastecimiento_mayorista
            .select(*columnas)
            .exceptAll(
                abastecimiento_guardado.select(*columnas)
            )
            .count()
        )

        diferencias += (
            abastecimiento_guardado
            .select(*columnas)
            .exceptAll(
                gold_abastecimiento_mayorista.select(*columnas)
            )
            .count()
        )

        print("Filas diferentes:", diferencias)

        nulos_guardados = abastecimiento_guardado.filter(
            F.col("ingreso_hoy_t").isNull()
        ).count()

        print("Ingresos nulos guardados:", nulos_guardados)

        if (
            registros_guardados == 66
            and columnas_guardadas == 16
            and diferencias == 0
            and nulos_guardados == 28
        ):
            print("\nALMACENAMIENTO GOLD: CORRECTO")
        else:
            print("\nATENCION: Revisar resultados almacenados")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 42 FINALIZADO =====")

===== PASO 42: GUARDAR GOLD ABASTECIMIENTO =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Leyendo desde HDFS

===== VERIFICACION HDFS =====
Registros guardados: 66
Columnas guardadas: 16

===== REGISTROS POR PRODUCTO =====
+---------+---------+-----------+-----------+
|producto |registros|con_ingreso|sin_ingreso|
+---------+---------+-----------+-----------+
|ALCACHOFA|33       |15         |18         |
|ESPARRAGO|33       |23         |10         |
+---------+---------+-----------+-----------+


PROGRESO: 80% - Comparando con Gold original
Filas diferentes: 0
Ingresos nulos guardados: 28

ALMACENAMIENTO GOLD: CORRECTO
PROGRESO: 100% - Verificacion finalizada
===== PASO 42 FINALIZADO =====


## Transformación Gold: precios del Mercado de Productores

Se transforma la información de precios del Mercado de
Productores de Santa Anita correspondiente a 2024–2025.

Los datos se organizan por mercado, categoría, producto,
unidad de medida y fecha del reporte.

Se conservan los precios reportados y sus variaciones
porcentuales, manteniendo las unidades comerciales originales.

Esta fuente se utilizará como información complementaria
del mercado interno y no se integrará directamente con
las series de espárrago y alcachofa.

In [26]:
from pyspark.sql import functions as F

print("===== PASO 43: GOLD MERCADO DE PRODUCTORES =====")
print("PROGRESO: 0% - Leyendo y preparando Silver", flush=True)

mercado_preparado = (
    mercado_productores_silver
    .withColumn("mercado", F.upper(F.trim("mercado")))
    .withColumn("categoria", F.upper(F.trim("categoria")))
    .withColumn("producto", F.upper(F.trim("producto")))
    .withColumn("unidad_medida", F.upper(F.trim("unidad_medida")))
)

print("Registros Silver:", mercado_preparado.count())

print("\nPROGRESO: 20% - Verificando calidad", flush=True)

# Revisar que las claves indispensables estén presentes.
invalidos_mercado = mercado_preparado.filter(
    F.col("fecha_reporte").isNull()
    | F.col("mercado").isNull()
    | (F.col("mercado") == "")
    | F.col("categoria").isNull()
    | (F.col("categoria") == "")
    | F.col("producto").isNull()
    | (F.col("producto") == "")
    | F.col("unidad_medida").isNull()
    | (F.col("unidad_medida") == "")
    | F.col("precio_hoy_soles").isNull()
    | (F.col("precio_hoy_soles") < 0)
    | (F.col("precio_7dias_soles") < 0)
    | (F.col("precio_ayer_soles") < 0)
)

cantidad_invalidos = invalidos_mercado.count()

print("Registros invalidos:", cantidad_invalidos)

print("\nPROGRESO: 40% - Verificando unicidad", flush=True)

claves_mercado = [
    "mercado",
    "categoria",
    "producto",
    "unidad_medida",
    "fecha_reporte"
]

duplicados_mercado = (
    mercado_preparado
    .groupBy(*claves_mercado)
    .count()
    .filter(F.col("count") > 1)
)

cantidad_duplicados = duplicados_mercado.count()

print("Grupos con claves repetidas:", cantidad_duplicados)

if cantidad_invalidos > 0 or cantidad_duplicados > 0:

    print("\nATENCION: Revisar los datos antes de construir Gold.")

    invalidos_mercado.show(10, truncate=False)
    duplicados_mercado.show(10, truncate=False)

else:

    print("\nPROGRESO: 60% - Construyendo Gold", flush=True)

    gold_precios_mercado_productores = (
        mercado_preparado
        .select(
            "mercado",
            "categoria",
            "producto",
            "unidad_medida",
            "fecha_reporte",
            "precio_ayer_soles",
            "precio_hoy_soles",
            "precio_7dias_soles",
            "variacion_porcentaje"
        )
        .withColumn("anio", F.year("fecha_reporte"))
        .withColumn("mes_numero", F.month("fecha_reporte"))
    )

    print("\nPROGRESO: 80% - Revisando resultados", flush=True)

    print("\n===== ESQUEMA GOLD =====")
    gold_precios_mercado_productores.printSchema()

    print("\n===== REGISTROS GOLD =====")
    print("Total:", gold_precios_mercado_productores.count())

    print("\n===== REGISTROS POR CATEGORIA =====")
    gold_precios_mercado_productores.groupBy("categoria") \
        .count() \
        .orderBy("categoria") \
        .show(truncate=False)

    print("\n===== MUESTRA GOLD =====")
    gold_precios_mercado_productores.orderBy(
        "fecha_reporte", "categoria", "producto"
    ).show(12, truncate=False)

    print("\nPROGRESO: 100% - Transformacion completada")

print("===== PASO 43 FINALIZADO =====")

===== PASO 43: GOLD MERCADO DE PRODUCTORES =====
PROGRESO: 0% - Leyendo y preparando Silver
Registros Silver: 756

PROGRESO: 20% - Verificando calidad
Registros invalidos: 0

PROGRESO: 40% - Verificando unicidad
Grupos con claves repetidas: 0

PROGRESO: 60% - Construyendo Gold

PROGRESO: 80% - Revisando resultados

===== ESQUEMA GOLD =====
root
 |-- mercado: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- unidad_medida: string (nullable = true)
 |-- fecha_reporte: date (nullable = true)
 |-- precio_ayer_soles: double (nullable = true)
 |-- precio_hoy_soles: double (nullable = true)
 |-- precio_7dias_soles: double (nullable = true)
 |-- variacion_porcentaje: double (nullable = true)
 |-- anio: integer (nullable = true)
 |-- mes_numero: integer (nullable = true)


===== REGISTROS GOLD =====
Total: 756

===== REGISTROS POR CATEGORIA =====
+-----------------+-----+
|categoria        |count|
+-----------------+-----+
|ACEITE COM

## Validación Gold: precios del Mercado de Productores

Se verifica la conservación de los registros de precios
del Mercado de Productores de Santa Anita.

La validación comprueba la cantidad de registros, la
unicidad de las claves y la correspondencia entre
los valores de Silver y Gold.

Se conservan los precios en soles, las variaciones
porcentuales y las unidades comerciales de cada producto.

Esta tabla representa información complementaria del
mercado interno y permanece separada de los indicadores
específicos de espárrago y alcachofa.

In [27]:
from pyspark.sql import functions as F

print("===== PASO 44: VALIDACION GOLD MERCADO PRODUCTORES =====")
print("PROGRESO: 0% - Iniciando validacion", flush=True)

print("\nPROGRESO: 20% - Comparando registros", flush=True)

registros_silver_mercado = mercado_preparado.count()
registros_gold_mercado = gold_precios_mercado_productores.count()

print("Registros Silver:", registros_silver_mercado)
print("Registros Gold:", registros_gold_mercado)

print("\nPROGRESO: 40% - Verificando claves", flush=True)

claves_mercado = [
    "mercado",
    "categoria",
    "producto",
    "unidad_medida",
    "fecha_reporte"
]

duplicados_gold_mercado = (
    gold_precios_mercado_productores
    .groupBy(*claves_mercado)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Claves duplicadas:", duplicados_gold_mercado)

print("\nPROGRESO: 60% - Comparando valores", flush=True)

columnas_comparar = claves_mercado + [
    "precio_ayer_soles",
    "precio_hoy_soles",
    "precio_7dias_soles",
    "variacion_porcentaje"
]

silver_comparacion_mercado = mercado_preparado.select(
    *columnas_comparar
)

gold_comparacion_mercado = (
    gold_precios_mercado_productores
    .select(*columnas_comparar)
)

# exceptAll conserva las repeticiones y compara también NULL.
diferencias_silver_gold = (
    silver_comparacion_mercado
    .exceptAll(gold_comparacion_mercado)
    .count()
)

diferencias_gold_silver = (
    gold_comparacion_mercado
    .exceptAll(silver_comparacion_mercado)
    .count()
)

diferencias_mercado = (
    diferencias_silver_gold + diferencias_gold_silver
)

print("Filas diferentes:", diferencias_mercado)

print("\nPROGRESO: 80% - Verificando unidades", flush=True)

gold_precios_mercado_productores.groupBy("categoria") \
    .agg(
        F.count("*").alias("registros"),
        F.countDistinct("unidad_medida").alias("unidades_distintas")
    ) \
    .orderBy("categoria") \
    .show(truncate=False)

print("\n===== RESULTADO FINAL =====")

if (
    registros_silver_mercado == registros_gold_mercado == 756
    and duplicados_gold_mercado == 0
    and diferencias_mercado == 0
):
    mercado_gold_validada = True
    print("VALIDACION FINAL: CORRECTA")
else:
    mercado_gold_validada = False
    print("ATENCION: REVISAR RESULTADOS")

print("\nPROGRESO: 100% - Validacion completada")
print("===== PASO 44 FINALIZADO =====")

===== PASO 44: VALIDACION GOLD MERCADO PRODUCTORES =====
PROGRESO: 0% - Iniciando validacion

PROGRESO: 20% - Comparando registros
Registros Silver: 756
Registros Gold: 756

PROGRESO: 40% - Verificando claves
Claves duplicadas: 0

PROGRESO: 60% - Comparando valores
Filas diferentes: 0

PROGRESO: 80% - Verificando unidades
+-----------------+---------+------------------+
|categoria        |registros|unidades_distintas|
+-----------------+---------+------------------+
|ACEITE COMESTIBLE|72       |2                 |
|ARROZ            |119      |2                 |
|AZUCAR           |255      |1                 |
|FIDEOS           |47       |1                 |
|HARINA           |71       |1                 |
|LECHE            |96       |3                 |
|MENESTRAS        |96       |1                 |
+-----------------+---------+------------------+


===== RESULTADO FINAL =====
VALIDACION FINAL: CORRECTA

PROGRESO: 100% - Validacion completada
===== PASO 44 FINALIZADO =====


## Almacenamiento Gold: precios del Mercado de Productores

Se almacena en Hadoop HDFS la tabla analítica de precios
del Mercado de Productores de Santa Anita, utilizando
archivos Parquet.

La tabla contiene 756 registros de los años 2024 y 2025,
organizados por mercado, categoría, producto, unidad
comercial y fecha del reporte.

Se conservan los precios originales y sus variaciones,
sin mezclar unidades de medida diferentes.

Antes del almacenamiento se verificaron la unicidad
de los registros y la conservación de los valores
respecto a la capa Silver.

Esta fuente constituye información complementaria
del mercado interno y no representa directamente
los precios de espárrago y alcachofa.

In [28]:
from pyspark.sql import functions as F

print("===== PASO 45: GUARDAR GOLD MERCADO PRODUCTORES =====")

if not globals().get("mercado_gold_validada", False):
    print("ERROR: Primero debe superarse la validacion del Paso 44.")

else:
    ruta_mercado_gold = (
        f"{BASE_GOLD}/precios_mercado_productores"
    )

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_mercado_gold
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La tabla Gold ya existe.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_mercado_gold)

    else:
        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            gold_precios_mercado_productores
            .coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_mercado_gold)
        )

        print("PROGRESO: 60% - Verificando HDFS", flush=True)

        mercado_guardado = spark.read.parquet(
            ruta_mercado_gold
        )

        registros_guardados = mercado_guardado.count()
        columnas_guardadas = len(mercado_guardado.columns)

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print("\n===== REGISTROS POR CATEGORIA =====")

        mercado_guardado.groupBy("categoria") \
            .count() \
            .orderBy("categoria") \
            .show(truncate=False)

        print("\nPROGRESO: 80% - Comparando datos originales")

        columnas = gold_precios_mercado_productores.columns

        diferencias = (
            gold_precios_mercado_productores
            .select(*columnas)
            .exceptAll(mercado_guardado.select(*columnas))
            .count()
        )

        diferencias += (
            mercado_guardado
            .select(*columnas)
            .exceptAll(
                gold_precios_mercado_productores.select(*columnas)
            )
            .count()
        )

        print("Filas diferentes:", diferencias)

        if (
            registros_guardados == 756
            and columnas_guardadas == 11
            and diferencias == 0
        ):
            print("\nALMACENAMIENTO GOLD: CORRECTO")
        else:
            print("\nATENCION: Revisar resultados almacenados")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 45 FINALIZADO =====")

===== PASO 45: GUARDAR GOLD MERCADO PRODUCTORES =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Verificando HDFS

===== VERIFICACION HDFS =====
Registros guardados: 756
Columnas guardadas: 11

===== REGISTROS POR CATEGORIA =====
+-----------------+-----+
|categoria        |count|
+-----------------+-----+
|ACEITE COMESTIBLE|72   |
|ARROZ            |119  |
|AZUCAR           |255  |
|FIDEOS           |47   |
|HARINA           |71   |
|LECHE            |96   |
|MENESTRAS        |96   |
+-----------------+-----+


PROGRESO: 80% - Comparando datos originales
Filas diferentes: 0

ALMACENAMIENTO GOLD: CORRECTO
PROGRESO: 100% - Verificacion finalizada
===== PASO 45 FINALIZADO =====


## Análisis de compatibilidad para la integración Gold

Se revisan las siete tablas analíticas almacenadas en HDFS
para identificar sus periodos, dimensiones y niveles de detalle.

El análisis permitirá determinar qué fuentes pueden
relacionarse mediante dimensiones compartidas, como
tiempo, cultivo y ubicación.

Se evitarán uniones que produzcan duplicación de registros
o mezclen datos de diferentes periodos, mercados y unidades.

Este análisis servirá como base para diseñar el modelo
dimensional de la capa Gold del proyecto Danper.

In [29]:
from pyspark.sql import functions as F

print("===== PASO 46: ANALISIS DE COMPATIBILIDAD GOLD =====")
print("PROGRESO: 0% - Leyendo tablas almacenadas", flush=True)

tablas_gold = {
    "produccion_mensual": spark.read.parquet(
        f"{BASE_GOLD}/produccion_mensual"
    ),
    "exportaciones_regionales": spark.read.parquet(
        f"{BASE_GOLD}/exportaciones_regionales"
    ),
    "exportaciones_danper": spark.read.parquet(
        f"{BASE_GOLD}/exportaciones_danper"
    ),
    "agroindustria_mensual": spark.read.parquet(
        f"{BASE_GOLD}/agroindustria_mensual"
    ),
    "costos_mensuales": spark.read.parquet(
        f"{BASE_GOLD}/costos_mensuales"
    ),
    "abastecimiento_mayorista": spark.read.parquet(
        f"{BASE_GOLD}/abastecimiento_mayorista"
    ),
    "precios_mercado_productores": spark.read.parquet(
        f"{BASE_GOLD}/precios_mercado_productores"
    )
}

print("\nPROGRESO: 25% - Identificando periodos", flush=True)

resumen_gold = []

for nombre, df in tablas_gold.items():

    total = df.count()

    if "anio" in df.columns:
        periodo = df.agg(
            F.min("anio").alias("desde"),
            F.max("anio").alias("hasta")
        ).first()
        desde = periodo["desde"]
        hasta = periodo["hasta"]
    else:
        desde = None
        hasta = None

    resumen_gold.append((
        nombre,
        total,
        len(df.columns),
        desde,
        hasta
    ))

resumen_gold_df = spark.createDataFrame(
    resumen_gold,
    [
        "tabla",
        "registros",
        "columnas",
        "anio_desde",
        "anio_hasta"
    ]
)

print("\n===== COBERTURA DE LAS TABLAS GOLD =====")

resumen_gold_df.show(20, truncate=False)

print("\nPROGRESO: 50% - Revisando dimensiones", flush=True)

dimensiones_analisis = [
    "cultivo",
    "producto",
    "provincia",
    "distrito",
    "region",
    "pais",
    "mercado",
    "categoria",
    "tipo_costo",
    "proceso",
    "presentacion",
    "periodo",
    "fecha_reporte",
    "anio",
    "mes_numero"
]

print("\n===== DIMENSIONES DISPONIBLES POR TABLA =====")

for nombre, df in tablas_gold.items():

    presentes = [
        columna
        for columna in dimensiones_analisis
        if columna in df.columns
    ]

    print(f"\n{nombre}:")
    print(", ".join(presentes))

print("\nPROGRESO: 70% - Identificando cultivos", flush=True)

print("\n===== CULTIVOS DISPONIBLES =====")

for nombre in [
    "produccion_mensual",
    "exportaciones_regionales",
    "agroindustria_mensual"
]:

    df = tablas_gold[nombre]

    print(f"\n{nombre}")

    df.select("cultivo") \
        .distinct() \
        .orderBy("cultivo") \
        .show(20, truncate=False)

print("\n===== PRODUCTOS DE ABASTECIMIENTO =====")

tablas_gold["abastecimiento_mayorista"] \
    .select("producto") \
    .distinct() \
    .orderBy("producto") \
    .show(20, truncate=False)

print("\nPROGRESO: 85% - Revisando cobertura temporal", flush=True)

print("\n===== REGISTROS POR ANIO Y TABLA =====")

for nombre, df in tablas_gold.items():

    print(f"\n{nombre}")

    df.groupBy("anio") \
        .count() \
        .orderBy("anio") \
        .show(30, truncate=False)

print("\nPROGRESO: 100% - Analisis completado", flush=True)
print("===== PASO 46 FINALIZADO =====")

===== PASO 46: ANALISIS DE COMPATIBILIDAD GOLD =====
PROGRESO: 0% - Leyendo tablas almacenadas

PROGRESO: 25% - Identificando periodos

===== COBERTURA DE LAS TABLAS GOLD =====


[Stage 799:>                                                        (0 + 1) / 1]

+---------------------------+---------+--------+----------+----------+
|tabla                      |registros|columnas|anio_desde|anio_hasta|
+---------------------------+---------+--------+----------+----------+
|produccion_mensual         |561      |13      |2024      |2025      |
|exportaciones_regionales   |720      |7       |2012      |2023      |
|exportaciones_danper       |495      |6       |2024      |2026      |
|agroindustria_mensual      |3134     |7       |2010      |2024      |
|costos_mensuales           |6292     |7       |2018      |2026      |
|abastecimiento_mayorista   |66       |16      |2024      |2026      |
|precios_mercado_productores|756      |11      |2024      |2025      |
+---------------------------+---------+--------+----------+----------+


PROGRESO: 50% - Revisando dimensiones

===== DIMENSIONES DISPONIBLES POR TABLA =====

produccion_mensual:
cultivo, provincia, distrito, periodo, anio, mes_numero

exportaciones_regionales:
cultivo, periodo, anio

expo


===== CULTIVOS DISPONIBLES =====

produccion_mensual
+---------+
|cultivo  |
+---------+
|ALCACHOFA|
|ARANDANO |
|ESPARRAGO|
+---------+


exportaciones_regionales
+---------+
|cultivo  |
+---------+
|ALCACHOFA|
|ARANDANO |
|ESPARRAGO|
|PALTA    |
|UVA      |
+---------+


agroindustria_mensual
+---------+
|cultivo  |
+---------+
|ALCACHOFA|
|ESPARRAGO|
+---------+


===== PRODUCTOS DE ABASTECIMIENTO =====
+---------+
|producto |
+---------+
|ALCACHOFA|
|ESPARRAGO|
+---------+


PROGRESO: 85% - Revisando cobertura temporal

===== REGISTROS POR ANIO Y TABLA =====

produccion_mensual
+----+-----+
|anio|count|
+----+-----+
|2024|165  |
|2025|396  |
+----+-----+


exportaciones_regionales
+----+-----+
|anio|count|
+----+-----+
|2012|60   |
|2013|60   |
|2014|60   |
|2015|60   |
|2016|60   |
|2017|60   |
|2018|60   |
|2019|60   |
|2020|60   |
|2021|60   |
|2022|60   |
|2023|60   |
+----+-----+


exportaciones_danper
+----+-----+
|anio|count|
+----+-----+
|2024|204  |
|2025|223  |
|2026|68 

## Modelo dimensional Gold: dimensión tiempo

Se construye la dimensión de tiempo del modelo analítico
agrocomercial para organizar los indicadores por año,
trimestre y mes.

La dimensión se genera a partir de los periodos disponibles
en las siete tablas Gold almacenadas en HDFS.

Cada registro representa un mes y contiene una clave
temporal única, el año, el número de mes, el trimestre
y la identificación del periodo.

La dimensión permitirá consultar distintas tablas de hechos
a través de una referencia temporal común.

La coincidencia temporal no implica que los indicadores
de diferentes fuentes sean directamente comparables.

In [30]:
from pyspark.sql import functions as F

print("===== PASO 47: CONSTRUIR DIMENSION TIEMPO =====")
print("PROGRESO: 0% - Preparando periodos", flush=True)

# Usamos las siete tablas Gold leidas en el Paso 46.
periodos_gold = []

for nombre, df in tablas_gold.items():

    print(f"Revisando: {nombre}", flush=True)

    if "periodo" in df.columns:

        fechas = df.select(
            F.trunc(F.col("periodo"), "month").alias("periodo")
        )

    elif "fecha_reporte" in df.columns:

        fechas = df.select(
            F.trunc(F.col("fecha_reporte"), "month").alias("periodo")
        )

    else:
        raise ValueError(
            f"La tabla {nombre} no tiene columna de fecha."
        )

    periodos_gold.append(fechas)

print("\nPROGRESO: 30% - Integrando periodos", flush=True)

# Unimos solo fechas, nunca los indicadores de distintas fuentes.
todos_periodos = periodos_gold[0]

for fechas in periodos_gold[1:]:
    todos_periodos = todos_periodos.unionByName(fechas)

# Cada mes se representa una sola vez.
periodos_unicos = (
    todos_periodos
    .filter(F.col("periodo").isNotNull())
    .distinct()
)

print("\nPROGRESO: 60% - Construyendo dimension", flush=True)

dim_tiempo = (
    periodos_unicos
    .withColumn("anio", F.year("periodo"))
    .withColumn("mes_numero", F.month("periodo"))
    .withColumn("trimestre", F.quarter("periodo"))
    .withColumn(
        "tiempo_id",
        F.year("periodo") * 100 + F.month("periodo")
    )
    .withColumn(
        "anio_mes",
        F.date_format("periodo", "yyyy-MM")
    )
    .select(
        "tiempo_id",
        "periodo",
        "anio",
        "trimestre",
        "mes_numero",
        "anio_mes"
    )
)

print("\nPROGRESO: 80% - Verificando resultados", flush=True)

print("\n===== ESQUEMA DIMENSION TIEMPO =====")
dim_tiempo.printSchema()

print("\n===== COBERTURA TEMPORAL =====")

dim_tiempo.agg(
    F.count("*").alias("meses_disponibles"),
    F.min("periodo").alias("primer_periodo"),
    F.max("periodo").alias("ultimo_periodo")
).show(truncate=False)

print("\n===== REGISTROS POR ANIO =====")

dim_tiempo.groupBy("anio") \
    .count() \
    .orderBy("anio") \
    .show(25, truncate=False)

print("\n===== MUESTRA DE LA DIMENSION =====")

dim_tiempo.orderBy("periodo") \
    .show(15, truncate=False)

print("\n===== VERIFICACION DE CLAVES =====")

duplicados_tiempo = (
    dim_tiempo
    .groupBy("tiempo_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

periodos_nulos = dim_tiempo.filter(
    F.col("periodo").isNull()
).count()

print("Claves duplicadas:", duplicados_tiempo)
print("Periodos nulos:", periodos_nulos)

if duplicados_tiempo == 0 and periodos_nulos == 0:
    dim_tiempo_validada = True
    print("\nDIMENSION TIEMPO: CONSTRUIDA CORRECTAMENTE")
else:
    dim_tiempo_validada = False
    print("\nATENCION: Revisar dimension tiempo")

print("\nPROGRESO: 100% - Construccion finalizada")
print("===== PASO 47 FINALIZADO =====")

===== PASO 47: CONSTRUIR DIMENSION TIEMPO =====
PROGRESO: 0% - Preparando periodos
Revisando: produccion_mensual
Revisando: exportaciones_regionales
Revisando: exportaciones_danper
Revisando: agroindustria_mensual
Revisando: costos_mensuales
Revisando: abastecimiento_mayorista
Revisando: precios_mercado_productores

PROGRESO: 30% - Integrando periodos

PROGRESO: 60% - Construyendo dimension

PROGRESO: 80% - Verificando resultados

===== ESQUEMA DIMENSION TIEMPO =====
root
 |-- tiempo_id: integer (nullable = true)
 |-- periodo: date (nullable = true)
 |-- anio: integer (nullable = true)
 |-- trimestre: integer (nullable = true)
 |-- mes_numero: integer (nullable = true)
 |-- anio_mes: string (nullable = true)


===== COBERTURA TEMPORAL =====
+-----------------+--------------+--------------+
|meses_disponibles|primer_periodo|ultimo_periodo|
+-----------------+--------------+--------------+
|201              |2010-01-01    |2026-09-01    |
+-----------------+--------------+--------------+

## Almacenamiento Gold: dimensión tiempo

Se almacena la dimensión tiempo del modelo dimensional
agrocomercial en Hadoop HDFS, utilizando formato Parquet.

La dimensión contiene 201 periodos mensuales únicos,
comprendidos entre enero de 2010 y septiembre de 2026.

Cada periodo posee una clave única en formato AAAAMM,
así como atributos de año, trimestre y mes.

Esta dimensión podrá ser utilizada por las diferentes
tablas de hechos para organizar los análisis temporales.

La disponibilidad de un mes en la dimensión no implica
que todas las fuentes contengan información para ese mes.

In [32]:
print("===== VERIFICACION DIMENSION TIEMPO =====")

ruta = f"{BASE_GOLD}/dim_tiempo"

df = spark.read.parquet(ruta)

print("Registros:", df.count())
print("Columnas:", len(df.columns))

df.orderBy("tiempo_id").show(5, truncate=False)

print("===== VERIFICACION FINALIZADA =====")

===== VERIFICACION DIMENSION TIEMPO =====


Registros: 201
Columnas: 6
+---------+----------+----+---------+----------+--------+
|tiempo_id|periodo   |anio|trimestre|mes_numero|anio_mes|
+---------+----------+----+---------+----------+--------+
|201001   |2010-01-01|2010|1        |1         |2010-01 |
|201002   |2010-02-01|2010|1        |2         |2010-02 |
|201003   |2010-03-01|2010|1        |3         |2010-03 |
|201004   |2010-04-01|2010|2        |4         |2010-04 |
|201005   |2010-05-01|2010|2        |5         |2010-05 |
+---------+----------+----+---------+----------+--------+
only showing top 5 rows

===== VERIFICACION FINALIZADA =====


In [33]:
from pyspark.sql import functions as F

print("===== PASO 48.1: VALIDACION FINAL DIM_TIEMPO =====")

dim_tiempo_guardada = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
)

total = dim_tiempo_guardada.count()

duplicados = (
    dim_tiempo_guardada
    .groupBy("tiempo_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

nulos = dim_tiempo_guardada.filter(
    F.col("tiempo_id").isNull()
    | F.col("periodo").isNull()
).count()

claves_incorrectas = dim_tiempo_guardada.filter(
    F.col("tiempo_id") != (
        F.year("periodo") * 100 + F.month("periodo")
    )
).count()

print("Registros:", total)
print("Claves duplicadas:", duplicados)
print("Claves o fechas nulas:", nulos)
print("Claves temporales incorrectas:", claves_incorrectas)

if (
    total == 201
    and duplicados == 0
    and nulos == 0
    and claves_incorrectas == 0
):
    dim_tiempo_validada = True
    print("\nDIMENSION TIEMPO: VALIDACION CORRECTA")
else:
    dim_tiempo_validada = False
    print("\nATENCION: REVISAR DIMENSION TIEMPO")

print("===== PASO 48.1 FINALIZADO =====")

===== PASO 48.1: VALIDACION FINAL DIM_TIEMPO =====
Registros: 201
Claves duplicadas: 0
Claves o fechas nulas: 0
Claves temporales incorrectas: 0

DIMENSION TIEMPO: VALIDACION CORRECTA
===== PASO 48.1 FINALIZADO =====


## Modelo dimensional Gold: dimensión cultivo

Se construye la dimensión cultivo a partir de los cultivos
disponibles en las fuentes analíticas del proyecto Danper.

Esta dimensión identifica los cultivos presentes en los
registros de producción agrícola, exportaciones regionales,
agroindustria y abastecimiento mayorista.

Cada cultivo contará con una clave única y atributos que
indican la disponibilidad de información en cada fuente.

La dimensión permitirá relacionar las tablas de hechos
que contienen información por cultivo, sin duplicar
registros ni asumir coincidencias entre periodos distintos.

Las exportaciones propias de Danper no se asociarán
a cultivos específicos porque su fuente no contiene
esa identificación.

In [34]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("===== PASO 49: CONSTRUIR DIMENSION CULTIVO =====")
print("PROGRESO: 0% - Leyendo cultivos de Gold", flush=True)

# Fuente 1: produccion agricola
cultivos_produccion = (
    tablas_gold["produccion_mensual"]
    .select(F.upper(F.trim("cultivo")).alias("cultivo"))
    .distinct()
    .withColumn("tiene_produccion", F.lit(True))
)

# Fuente 2: exportaciones regionales
cultivos_exportaciones = (
    tablas_gold["exportaciones_regionales"]
    .select(F.upper(F.trim("cultivo")).alias("cultivo"))
    .distinct()
    .withColumn("tiene_exportaciones_regionales", F.lit(True))
)

# Fuente 3: agroindustria
cultivos_agroindustria = (
    tablas_gold["agroindustria_mensual"]
    .select(F.upper(F.trim("cultivo")).alias("cultivo"))
    .distinct()
    .withColumn("tiene_agroindustria", F.lit(True))
)

# Fuente 4: abastecimiento mayorista
cultivos_abastecimiento = (
    tablas_gold["abastecimiento_mayorista"]
    .select(F.upper(F.trim("producto")).alias("cultivo"))
    .distinct()
    .withColumn("tiene_abastecimiento", F.lit(True))
)

print("PROGRESO: 30% - Integrando cultivos", flush=True)

# Crear lista unica de cultivos.
todos_cultivos = (
    cultivos_produccion.select("cultivo")
    .unionByName(cultivos_exportaciones.select("cultivo"))
    .unionByName(cultivos_agroindustria.select("cultivo"))
    .unionByName(cultivos_abastecimiento.select("cultivo"))
    .filter(
        F.col("cultivo").isNotNull()
        & (F.col("cultivo") != "")
    )
    .distinct()
)

print("PROGRESO: 55% - Construyendo atributos", flush=True)

dim_cultivo_base = (
    todos_cultivos
    .join(cultivos_produccion, "cultivo", "left")
    .join(cultivos_exportaciones, "cultivo", "left")
    .join(cultivos_agroindustria, "cultivo", "left")
    .join(cultivos_abastecimiento, "cultivo", "left")
    .fillna({
        "tiene_produccion": False,
        "tiene_exportaciones_regionales": False,
        "tiene_agroindustria": False,
        "tiene_abastecimiento": False
    })
)

# Asignar identificadores estables ordenando alfabeticamente.
ventana_cultivos = Window.orderBy("cultivo")

dim_cultivo = (
    dim_cultivo_base
    .withColumn(
        "cultivo_id",
        F.row_number().over(ventana_cultivos)
    )
    .select(
        "cultivo_id",
        "cultivo",
        "tiene_produccion",
        "tiene_exportaciones_regionales",
        "tiene_agroindustria",
        "tiene_abastecimiento"
    )
)

print("PROGRESO: 80% - Verificando resultados", flush=True)

print("\n===== ESQUEMA DIMENSION CULTIVO =====")
dim_cultivo.printSchema()

print("\n===== CULTIVOS IDENTIFICADOS =====")
dim_cultivo.orderBy("cultivo_id").show(
    20, truncate=False
)

print("\n===== VALIDACION =====")

total_cultivos = dim_cultivo.count()

duplicados = (
    dim_cultivo
    .groupBy("cultivo_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

nombres_duplicados = (
    dim_cultivo
    .groupBy("cultivo")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

nulos = dim_cultivo.filter(
    F.col("cultivo_id").isNull()
    | F.col("cultivo").isNull()
).count()

print("Total cultivos:", total_cultivos)
print("Identificadores duplicados:", duplicados)
print("Nombres duplicados:", nombres_duplicados)
print("Claves o nombres nulos:", nulos)

if (
    total_cultivos == 5
    and duplicados == 0
    and nombres_duplicados == 0
    and nulos == 0
):
    dim_cultivo_validada = True
    print("\nDIMENSION CULTIVO: CONSTRUIDA CORRECTAMENTE")
else:
    dim_cultivo_validada = False
    print("\nATENCION: Revisar dimension cultivo")

print("\nPROGRESO: 100% - Construccion finalizada")
print("===== PASO 49 FINALIZADO =====")

===== PASO 49: CONSTRUIR DIMENSION CULTIVO =====
PROGRESO: 0% - Leyendo cultivos de Gold
PROGRESO: 30% - Integrando cultivos
PROGRESO: 55% - Construyendo atributos
PROGRESO: 80% - Verificando resultados

===== ESQUEMA DIMENSION CULTIVO =====
root
 |-- cultivo_id: integer (nullable = false)
 |-- cultivo: string (nullable = true)
 |-- tiene_produccion: boolean (nullable = false)
 |-- tiene_exportaciones_regionales: boolean (nullable = false)
 |-- tiene_agroindustria: boolean (nullable = false)
 |-- tiene_abastecimiento: boolean (nullable = false)


===== CULTIVOS IDENTIFICADOS =====


+----------+---------+----------------+------------------------------+-------------------+--------------------+
|cultivo_id|cultivo  |tiene_produccion|tiene_exportaciones_regionales|tiene_agroindustria|tiene_abastecimiento|
+----------+---------+----------------+------------------------------+-------------------+--------------------+
|1         |ALCACHOFA|true            |true                          |true               |true                |
|2         |ARANDANO |true            |true                          |false              |false               |
|3         |ESPARRAGO|true            |true                          |true               |true                |
|4         |PALTA    |false           |true                          |false              |false               |
|5         |UVA      |false           |true                          |false              |false               |
+----------+---------+----------------+------------------------------+-------------------+--------------

## Almacenamiento Gold: dimensión cultivo

Se almacena en Hadoop HDFS la dimensión cultivo del modelo
dimensional agrocomercial utilizando el formato Parquet.

La dimensión contiene cinco cultivos: alcachofa, arándano,
espárrago, palta y uva.

Cada cultivo cuenta con una clave única y atributos que
identifican en qué fuentes existen registros disponibles.

Esta dimensión permitirá organizar el análisis de
producción agrícola, exportaciones regionales,
agroindustria y abastecimiento mayorista.

Las exportaciones propias de Danper permanecerán separadas,
debido a que su fuente no identifica el cultivo exportado.

In [35]:
from pyspark.sql import functions as F

print("===== PASO 50: GUARDAR DIMENSION CULTIVO =====")

if not globals().get("dim_cultivo_validada", False):

    print("ERROR: Primero debes validar la dimension del Paso 49.")

else:

    ruta_dim_cultivo = f"{BASE_GOLD}/dim_cultivo"

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_dim_cultivo
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):

        print("\nATENCION: La dimension ya existe en HDFS.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_dim_cultivo)

    else:

        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            dim_cultivo
            .coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_dim_cultivo)
        )

        print("PROGRESO: 60% - Verificando HDFS", flush=True)

        dim_cultivo_guardada = spark.read.parquet(
            ruta_dim_cultivo
        )

        registros_guardados = dim_cultivo_guardada.count()
        columnas_guardadas = len(dim_cultivo_guardada.columns)

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print("\n===== CULTIVOS ALMACENADOS =====")

        dim_cultivo_guardada.orderBy("cultivo_id") \
            .show(10, truncate=False)

        print("\nPROGRESO: 80% - Verificando claves", flush=True)

        duplicados_id = (
            dim_cultivo_guardada
            .groupBy("cultivo_id")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        duplicados_nombre = (
            dim_cultivo_guardada
            .groupBy("cultivo")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        nulos = dim_cultivo_guardada.filter(
            F.col("cultivo_id").isNull()
            | F.col("cultivo").isNull()
        ).count()

        print("Identificadores duplicados:", duplicados_id)
        print("Nombres duplicados:", duplicados_nombre)
        print("Claves o nombres nulos:", nulos)

        if (
            registros_guardados == 5
            and columnas_guardadas == 6
            and duplicados_id == 0
            and duplicados_nombre == 0
            and nulos == 0
        ):
            print("\nALMACENAMIENTO DIMENSION CULTIVO: CORRECTO")
        else:
            print("\nATENCION: Revisar dimension almacenada")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 50 FINALIZADO =====")

===== PASO 50: GUARDAR DIMENSION CULTIVO =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Verificando HDFS

===== VERIFICACION HDFS =====
Registros guardados: 5
Columnas guardadas: 6

===== CULTIVOS ALMACENADOS =====
+----------+---------+----------------+------------------------------+-------------------+--------------------+
|cultivo_id|cultivo  |tiene_produccion|tiene_exportaciones_regionales|tiene_agroindustria|tiene_abastecimiento|
+----------+---------+----------------+------------------------------+-------------------+--------------------+
|1         |ALCACHOFA|true            |true                          |true               |true                |
|2         |ARANDANO |true            |true                          |false              |false               |
|3         |ESPARRAGO|true            |true                          |true               |true                |
|4         |PALTA    |false           |true                          

## Modelo dimensional Gold: dimensiones de ubicación

Se construyen dimensiones independientes de ubicación
para las fuentes agrocomerciales del proyecto Danper.

Se identifican las provincias y distritos de producción,
las regiones de costos agrícolas, los países de destino
de exportaciones y los mercados mayoristas.

Cada dimensión contiene una clave única y atributos
descriptivos de su correspondiente nivel geográfico.

Las ubicaciones no se combinan en una sola dimensión,
debido a que representan conceptos diferentes.

Estas dimensiones servirán para organizar las tablas
de hechos y facilitar la consulta de indicadores
por territorio, destino comercial o mercado.

In [36]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("===== PASO 51: DIMENSIONES DE UBICACION =====")
print("PROGRESO: 0% - Preparando fuentes", flush=True)

# 1. Ubicaciones de produccion agricola
produccion_ubicaciones = (
    tablas_gold["produccion_mensual"]
    .select(
        F.upper(F.trim("provincia")).alias("provincia"),
        F.upper(F.trim("distrito")).alias("distrito")
    )
    .distinct()
)

dim_ubicacion_produccion = (
    produccion_ubicaciones
    .withColumn(
        "ubicacion_produccion_id",
        F.row_number().over(
            Window.orderBy("provincia", "distrito")
        )
    )
    .select(
        "ubicacion_produccion_id",
        "provincia",
        "distrito"
    )
)

print("PROGRESO: 25% - Regiones de costos", flush=True)

# 2. Regiones de los costos agricolas
regiones = (
    tablas_gold["costos_mensuales"]
    .select(
        F.upper(F.trim("region")).alias("region")
    )
    .distinct()
)

dim_region_costos = (
    regiones
    .withColumn(
        "region_id",
        F.row_number().over(Window.orderBy("region"))
    )
    .select("region_id", "region")
)

print("PROGRESO: 50% - Paises de exportacion", flush=True)

# 3. Paises destino de las exportaciones Danper
paises = (
    tablas_gold["exportaciones_danper"]
    .select(
        F.upper(F.trim("pais")).alias("pais")
    )
    .distinct()
)

dim_pais_exportacion = (
    paises
    .withColumn(
        "pais_id",
        F.row_number().over(Window.orderBy("pais"))
    )
    .select("pais_id", "pais")
)

print("PROGRESO: 70% - Mercados mayoristas", flush=True)

# 4. Mercados de abastecimiento y precios
mercados_abastecimiento = (
    tablas_gold["abastecimiento_mayorista"]
    .select(
        F.upper(F.trim("mercado")).alias("mercado")
    )
)

mercados_productores = (
    tablas_gold["precios_mercado_productores"]
    .select(
        F.upper(F.trim("mercado")).alias("mercado")
    )
)

mercados = (
    mercados_abastecimiento
    .unionByName(mercados_productores)
    .distinct()
)

dim_mercado = (
    mercados
    .withColumn(
        "mercado_id",
        F.row_number().over(Window.orderBy("mercado"))
    )
    .select("mercado_id", "mercado")
)

print("PROGRESO: 85% - Mostrando resultados", flush=True)

dimensiones_ubicacion = {
    "dim_ubicacion_produccion": (
        dim_ubicacion_produccion,
        ["provincia", "distrito"],
        "ubicacion_produccion_id"
    ),
    "dim_region_costos": (
        dim_region_costos,
        ["region"],
        "region_id"
    ),
    "dim_pais_exportacion": (
        dim_pais_exportacion,
        ["pais"],
        "pais_id"
    ),
    "dim_mercado": (
        dim_mercado,
        ["mercado"],
        "mercado_id"
    )
}

ubicaciones_validadas = True

for nombre, (df, claves_naturales, clave_id) in (
    dimensiones_ubicacion.items()
):
    print(f"\n===== {nombre.upper()} =====")

    total = df.count()

    duplicados = (
        df.groupBy(*claves_naturales)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    nulos = df.filter(
        F.col(clave_id).isNull()
        | F.expr(
            " OR ".join(
                f"`{campo}` IS NULL OR TRIM(`{campo}`) = ''"
                for campo in claves_naturales
            )
        )
    ).count()

    ids_duplicados = (
        df.groupBy(clave_id)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print("Registros:", total)
    print("Claves naturales duplicadas:", duplicados)
    print("Identificadores duplicados:", ids_duplicados)
    print("Valores nulos o vacios:", nulos)

    df.orderBy(clave_id).show(10, truncate=False)

    if (
        total == 0
        or duplicados > 0
        or ids_duplicados > 0
        or nulos > 0
    ):
        ubicaciones_validadas = False

print("\n===== RESULTADO FINAL =====")

if ubicaciones_validadas:
    print("DIMENSIONES DE UBICACION: CONSTRUIDAS CORRECTAMENTE")
else:
    print("ATENCION: REVISAR DIMENSIONES DE UBICACION")

print("PROGRESO: 100% - Construccion finalizada")
print("===== PASO 51 FINALIZADO =====")

===== PASO 51: DIMENSIONES DE UBICACION =====
PROGRESO: 0% - Preparando fuentes
PROGRESO: 25% - Regiones de costos
PROGRESO: 50% - Paises de exportacion
PROGRESO: 70% - Mercados mayoristas
PROGRESO: 85% - Mostrando resultados

===== DIM_UBICACION_PRODUCCION =====
Registros: 22
Claves naturales duplicadas: 0
Identificadores duplicados: 0
Valores nulos o vacios: 0
+-----------------------+---------+----------------+
|ubicacion_produccion_id|provincia|distrito        |
+-----------------------+---------+----------------+
|1                      |ASCOPE   |CASA GRANDE     |
|2                      |ASCOPE   |CHICAMA         |
|3                      |ASCOPE   |CHOCOPE         |
|4                      |ASCOPE   |MAGDALENA DE CAO|
|5                      |ASCOPE   |PAIJAN          |
|6                      |ASCOPE   |RAZURI          |
|7                      |ASCOPE   |SANTIAGO DE CAO |
|8                      |CHEPEN   |CHEPEN          |
|9                      |CHEPEN   |PACANGA         |

## Almacenamiento Gold: dimensiones de ubicación

Se almacenan cuatro dimensiones de ubicación en Hadoop HDFS
utilizando el formato Parquet.

Las dimensiones representan distintos niveles geográficos
y comerciales: provincia y distrito de producción, región
de costos, país de exportación y mercado mayorista.

Cada dimensión mantiene identificadores únicos y atributos
descriptivos previamente validados.

Se almacenan por separado para evitar relaciones incorrectas
entre ubicaciones de naturaleza diferente.

Estas dimensiones servirán como referencias para las
futuras tablas de hechos del modelo agrocomercial Danper.

In [37]:
from pyspark.sql import functions as F

print("===== PASO 52: GUARDAR DIMENSIONES DE UBICACION =====")

if not globals().get("ubicaciones_validadas", False):
    print("ERROR: Primero debes validar las dimensiones del Paso 51.")

else:
    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    print("PROGRESO: 0% - Preparando almacenamiento", flush=True)

    resultados_ubicaciones = []

    total_dimensiones = len(dimensiones_ubicacion)

    for indice, (nombre, datos) in enumerate(
        dimensiones_ubicacion.items(), start=1
    ):

        df, claves_naturales, clave_id = datos

        ruta_destino = f"{BASE_GOLD}/{nombre}"

        print(f"\n===== {nombre.upper()} =====", flush=True)
        print(f"Dimension {indice} de {total_dimensiones}")
        print("Ruta:", ruta_destino)

        ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
            ruta_destino
        )

        filesystem = ruta_java.getFileSystem(hadoop_conf)

        # No modificar una ruta que ya existe.
        if filesystem.exists(ruta_java):
            print("ATENCION: La ruta ya existe. No se sobrescribira.")
            resultados_ubicaciones.append(
                (nombre, "EXISTENTE - PENDIENTE DE VERIFICAR")
            )
            continue

        print("Guardando Parquet...", flush=True)

        (
            df.coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_destino)
        )

        print("Leyendo datos guardados...", flush=True)

        df_guardado = spark.read.parquet(ruta_destino)

        registros_originales = df.count()
        registros_guardados = df_guardado.count()

        columnas_originales = len(df.columns)
        columnas_guardadas = len(df_guardado.columns)

        duplicados_id = (
            df_guardado
            .groupBy(clave_id)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        duplicados_naturales = (
            df_guardado
            .groupBy(*claves_naturales)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        valores_invalidos = df_guardado.filter(
            F.col(clave_id).isNull()
            | F.expr(
                " OR ".join(
                    f"`{campo}` IS NULL OR TRIM(`{campo}`) = ''"
                    for campo in claves_naturales
                )
            )
        ).count()

        print("Registros originales:", registros_originales)
        print("Registros guardados:", registros_guardados)
        print("Columnas originales:", columnas_originales)
        print("Columnas guardadas:", columnas_guardadas)
        print("Identificadores duplicados:", duplicados_id)
        print("Ubicaciones duplicadas:", duplicados_naturales)
        print("Valores invalidos:", valores_invalidos)

        if (
            registros_originales == registros_guardados
            and columnas_originales == columnas_guardadas
            and duplicados_id == 0
            and duplicados_naturales == 0
            and valores_invalidos == 0
        ):
            estado = "CORRECTO"
        else:
            estado = "REVISAR"

        resultados_ubicaciones.append((nombre, estado))

        progreso = int(
            (indice / total_dimensiones) * 100
        )

        print(f"PROGRESO: {progreso}%", flush=True)

    print("\n===== RESUMEN FINAL =====")

    for nombre, estado in resultados_ubicaciones:
        print(f"{nombre}: {estado}")

    if (
        len(resultados_ubicaciones) == total_dimensiones
        and all(
            estado == "CORRECTO"
            for _, estado in resultados_ubicaciones
        )
    ):
        print("\nALMACENAMIENTO DIMENSIONES: CORRECTO")
    else:
        print(
            "\nATENCION: Hay dimensiones que requieren "
            "verificacion o revision."
        )

print("===== PASO 52 FINALIZADO =====")

===== PASO 52: GUARDAR DIMENSIONES DE UBICACION =====
PROGRESO: 0% - Preparando almacenamiento

===== DIM_UBICACION_PRODUCCION =====
Dimension 1 de 4
Ruta: hdfs://localhost:9000/danper/gold/dim_ubicacion_produccion
Guardando Parquet...
Leyendo datos guardados...
Registros originales: 22
Registros guardados: 22
Columnas originales: 3
Columnas guardadas: 3
Identificadores duplicados: 0
Ubicaciones duplicadas: 0
Valores invalidos: 0
PROGRESO: 25%

===== DIM_REGION_COSTOS =====
Dimension 2 de 4
Ruta: hdfs://localhost:9000/danper/gold/dim_region_costos
Guardando Parquet...
Leyendo datos guardados...


Registros originales: 25
Registros guardados: 25
Columnas originales: 2
Columnas guardadas: 2
Identificadores duplicados: 0
Ubicaciones duplicadas: 0
Valores invalidos: 0
PROGRESO: 50%

===== DIM_PAIS_EXPORTACION =====
Dimension 3 de 4
Ruta: hdfs://localhost:9000/danper/gold/dim_pais_exportacion
Guardando Parquet...
Leyendo datos guardados...
Registros originales: 35
Registros guardados: 35
Columnas originales: 2
Columnas guardadas: 2
Identificadores duplicados: 0
Ubicaciones duplicadas: 0
Valores invalidos: 0
PROGRESO: 75%

===== DIM_MERCADO =====
Dimension 4 de 4
Ruta: hdfs://localhost:9000/danper/gold/dim_mercado
Guardando Parquet...
Leyendo datos guardados...
Registros originales: 2
Registros guardados: 2
Columnas originales: 2
Columnas guardadas: 2
Identificadores duplicados: 0
Ubicaciones duplicadas: 0
Valores invalidos: 0
PROGRESO: 100%

===== RESUMEN FINAL =====
dim_ubicacion_produccion: CORRECTO
dim_region_costos: CORRECTO
dim_pais_exportacion: CORRECTO
dim_mercado: CORRECTO



## Modelo dimensional Gold: tabla de hechos de producción

Se construye la tabla de hechos fact_produccion_mensual
a partir de la información validada de producción agrícola.

La tabla se relaciona con las dimensiones de tiempo,
cultivo y ubicación de producción mediante claves foráneas.

Se conservan los indicadores de producción, precio en
chacra, cosechas, siembras, superficie y rendimiento.

El nivel de detalle corresponde a cada combinación
de mes, cultivo, provincia y distrito.

Se verifica que las relaciones no generen registros
duplicados ni pérdida de información.

Esta tabla facilitará los análisis productivos del
proyecto Danper mediante el modelo dimensional Gold.

In [38]:
from pyspark.sql import functions as F

print("===== PASO 53: TABLA DE HECHOS PRODUCCION =====")
print("PROGRESO: 0% - Leyendo tablas Gold", flush=True)

# Leer las tablas directamente desde HDFS.
produccion_fact_origen = spark.read.parquet(
    f"{BASE_GOLD}/produccion_mensual"
)

tiempo_fact = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
)

cultivo_fact = spark.read.parquet(
    f"{BASE_GOLD}/dim_cultivo"
)

ubicacion_fact = spark.read.parquet(
    f"{BASE_GOLD}/dim_ubicacion_produccion"
)

total_origen = produccion_fact_origen.count()

print("Registros de produccion originales:", total_origen)

print("\nPROGRESO: 25% - Relacionando dimensiones", flush=True)

# Relacionar por las claves naturales originales.
# Utilizamos LEFT JOIN para poder detectar claves sin relacion.
produccion_relacionada = (
    produccion_fact_origen.alias("p")
    .join(
        tiempo_fact.alias("t"),
        F.col("p.periodo") == F.col("t.periodo"),
        "left"
    )
    .join(
        cultivo_fact.alias("c"),
        F.col("p.cultivo") == F.col("c.cultivo"),
        "left"
    )
    .join(
        ubicacion_fact.alias("u"),
        (
            (F.col("p.provincia") == F.col("u.provincia"))
            &
            (F.col("p.distrito") == F.col("u.distrito"))
        ),
        "left"
    )
)

print("\nPROGRESO: 50% - Construyendo tabla de hechos", flush=True)

fact_produccion_mensual = produccion_relacionada.select(
    F.col("t.tiempo_id").alias("tiempo_id"),
    F.col("c.cultivo_id").alias("cultivo_id"),
    F.col("u.ubicacion_produccion_id").alias(
        "ubicacion_produccion_id"
    ),
    F.col("p.produccion_t").alias("produccion_t"),
    F.col("p.precio_chacra_s_kg").alias("precio_chacra_s_kg"),
    F.col("p.cosechas_ha").alias("cosechas_ha"),
    F.col("p.siembras_ha").alias("siembras_ha"),
    F.col("p.superficie_verde_ha").alias("superficie_verde_ha"),
    F.col("p.superficie_perdida_ha").alias(
        "superficie_perdida_ha"
    ),
    F.col("p.rendimiento_kg_ha").alias("rendimiento_kg_ha")
)

print("\n===== ESQUEMA DE LA TABLA DE HECHOS =====")
fact_produccion_mensual.printSchema()

print("\nPROGRESO: 75% - Verificando integridad", flush=True)

total_fact = fact_produccion_mensual.count()

# Detectar relaciones sin una dimension correspondiente.
claves_sin_relacion = fact_produccion_mensual.filter(
    F.col("tiempo_id").isNull()
    | F.col("cultivo_id").isNull()
    | F.col("ubicacion_produccion_id").isNull()
).count()

# Comprobar que el nivel de detalle sea unico.
duplicados_fact = (
    fact_produccion_mensual
    .groupBy(
        "tiempo_id",
        "cultivo_id",
        "ubicacion_produccion_id"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\n===== VALIDACION =====")
print("Registros originales:", total_origen)
print("Registros tabla de hechos:", total_fact)
print("Registros sin relacion dimensional:", claves_sin_relacion)
print("Claves de hechos duplicadas:", duplicados_fact)

print("\n===== MUESTRA DE LA TABLA =====")

fact_produccion_mensual.orderBy(
    "tiempo_id",
    "cultivo_id",
    "ubicacion_produccion_id"
).show(10, truncate=False)

print("\n===== RESULTADO FINAL =====")

if (
    total_origen == 561
    and total_fact == total_origen
    and claves_sin_relacion == 0
    and duplicados_fact == 0
):
    fact_produccion_validada = True
    print("TABLA DE HECHOS PRODUCCION: CONSTRUIDA CORRECTAMENTE")
else:
    fact_produccion_validada = False
    print("ATENCION: REVISAR RELACIONES DIMENSIONALES")

print("\nPROGRESO: 100% - Construccion finalizada")
print("===== PASO 53 FINALIZADO =====")

===== PASO 53: TABLA DE HECHOS PRODUCCION =====
PROGRESO: 0% - Leyendo tablas Gold
Registros de produccion originales: 561

PROGRESO: 25% - Relacionando dimensiones

PROGRESO: 50% - Construyendo tabla de hechos

===== ESQUEMA DE LA TABLA DE HECHOS =====
root
 |-- tiempo_id: integer (nullable = true)
 |-- cultivo_id: integer (nullable = true)
 |-- ubicacion_produccion_id: integer (nullable = true)
 |-- produccion_t: double (nullable = true)
 |-- precio_chacra_s_kg: double (nullable = true)
 |-- cosechas_ha: double (nullable = true)
 |-- siembras_ha: double (nullable = true)
 |-- superficie_verde_ha: double (nullable = true)
 |-- superficie_perdida_ha: double (nullable = true)
 |-- rendimiento_kg_ha: double (nullable = true)


PROGRESO: 75% - Verificando integridad

===== VALIDACION =====
Registros originales: 561
Registros tabla de hechos: 561
Registros sin relacion dimensional: 0
Claves de hechos duplicadas: 0

===== MUESTRA DE LA TABLA =====
+---------+----------+---------------------

## Validación de medidas: tabla de hechos de producción

Se valida la correspondencia entre los indicadores de
producción agrícola de la tabla Gold original y la tabla
de hechos dimensional.

La comparación utiliza las claves de tiempo, cultivo
y ubicación para verificar individualmente las medidas
de producción, precio en chacra, cosechas, siembras,
superficie y rendimiento.

Los valores nulos se conservan sin reemplazarlos por cero.

La validación garantiza que la incorporación de claves
dimensionales no altere los valores productivos originales.

In [39]:
from pyspark.sql import functions as F

print("===== PASO 54: VALIDACION MEDIDAS PRODUCCION =====")
print("PROGRESO: 0% - Preparando comparacion", flush=True)

# Columnas numericas que deben conservarse.
medidas_produccion = [
    "produccion_t",
    "precio_chacra_s_kg",
    "cosechas_ha",
    "siembras_ha",
    "superficie_verde_ha",
    "superficie_perdida_ha",
    "rendimiento_kg_ha"
]

# Crear claves dimensionales desde los datos originales.
origen_comparacion = (
    produccion_fact_origen
    .withColumn(
        "tiempo_id",
        F.year("periodo") * 100 + F.month("periodo")
    )
    .join(
        cultivo_fact.select("cultivo_id", "cultivo"),
        "cultivo",
        "left"
    )
    .join(
        ubicacion_fact.select(
            "ubicacion_produccion_id",
            "provincia",
            "distrito"
        ),
        ["provincia", "distrito"],
        "left"
    )
    .select(
        "tiempo_id",
        "cultivo_id",
        "ubicacion_produccion_id",
        *medidas_produccion
    )
)

print("PROGRESO: 30% - Comparando registros", flush=True)

columnas_comparar = [
    "tiempo_id",
    "cultivo_id",
    "ubicacion_produccion_id"
] + medidas_produccion

# Comparacion exacta, incluyendo valores NULL.
diferencias_origen_fact = (
    origen_comparacion
    .select(*columnas_comparar)
    .exceptAll(
        fact_produccion_mensual.select(*columnas_comparar)
    )
    .count()
)

diferencias_fact_origen = (
    fact_produccion_mensual
    .select(*columnas_comparar)
    .exceptAll(
        origen_comparacion.select(*columnas_comparar)
    )
    .count()
)

total_diferencias = (
    diferencias_origen_fact + diferencias_fact_origen
)

print("Filas diferentes:", total_diferencias)

print("\nPROGRESO: 60% - Verificando valores faltantes", flush=True)

print("\n===== VALORES NULOS POR INDICADOR =====")

resultados_nulos = []

for medida in medidas_produccion:

    nulos_origen = produccion_fact_origen.filter(
        F.col(medida).isNull()
    ).count()

    nulos_fact = fact_produccion_mensual.filter(
        F.col(medida).isNull()
    ).count()

    resultados_nulos.append(
        (medida, nulos_origen, nulos_fact)
    )

    print(
        f"{medida}: "
        f"origen={nulos_origen}, "
        f"fact={nulos_fact}"
    )

print("\nPROGRESO: 85% - Evaluando integridad", flush=True)

nulos_iguales = all(
    origen == fact
    for _, origen, fact in resultados_nulos
)

registros_iguales = (
    origen_comparacion.count()
    == fact_produccion_mensual.count()
    == 561
)

print("\n===== RESULTADO FINAL =====")

if (
    total_diferencias == 0
    and nulos_iguales
    and registros_iguales
):
    fact_produccion_medidas_validada = True
    print("VALIDACION MEDIDAS PRODUCCION: CORRECTA")
else:
    fact_produccion_medidas_validada = False
    print("ATENCION: REVISAR MEDIDAS DE PRODUCCION")

print("\nPROGRESO: 100% - Validacion finalizada")
print("===== PASO 54 FINALIZADO =====")

===== PASO 54: VALIDACION MEDIDAS PRODUCCION =====
PROGRESO: 0% - Preparando comparacion
PROGRESO: 30% - Comparando registros
Filas diferentes: 0

PROGRESO: 60% - Verificando valores faltantes

===== VALORES NULOS POR INDICADOR =====
produccion_t: origen=294, fact=294
precio_chacra_s_kg: origen=294, fact=294
cosechas_ha: origen=527, fact=527
siembras_ha: origen=525, fact=525
superficie_verde_ha: origen=0, fact=0
superficie_perdida_ha: origen=559, fact=559
rendimiento_kg_ha: origen=561, fact=561

PROGRESO: 85% - Evaluando integridad

===== RESULTADO FINAL =====
VALIDACION MEDIDAS PRODUCCION: CORRECTA

PROGRESO: 100% - Validacion finalizada
===== PASO 54 FINALIZADO =====


## Almacenamiento Gold: tabla de hechos de producción

Se almacena en Hadoop HDFS la tabla de hechos de producción
agrícola del modelo dimensional agrocomercial Danper.

La tabla contiene 561 registros y se relaciona mediante
claves foráneas con las dimensiones de tiempo, cultivo
y ubicación de producción.

Se conservan las medidas originales de producción,
precio en chacra, superficie, cosechas, siembras y rendimiento,
incluyendo sus valores nulos.

La validación previa confirmó que no existen diferencias
entre las medidas de origen y las de la tabla dimensional.

El almacenamiento se realiza en formato Parquet sin
sobrescribir las tablas Gold existentes.

In [40]:
from pyspark.sql import functions as F

print("===== PASO 55: GUARDAR FACT PRODUCCION =====")

if not (
    globals().get("fact_produccion_validada", False)
    and globals().get("fact_produccion_medidas_validada", False)
):
    print("ERROR: Primero deben completarse los Pasos 53 y 54.")

else:
    ruta_fact_produccion = f"{BASE_GOLD}/fact_produccion_mensual"

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_fact_produccion
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La tabla de hechos ya existe.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_fact_produccion)

    else:
        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            fact_produccion_mensual
            .coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_fact_produccion)
        )

        print("PROGRESO: 60% - Leyendo desde HDFS", flush=True)

        fact_produccion_guardada = spark.read.parquet(
            ruta_fact_produccion
        )

        registros_guardados = fact_produccion_guardada.count()
        columnas_guardadas = len(fact_produccion_guardada.columns)

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print("\nPROGRESO: 80% - Verificando integridad", flush=True)

        claves_nulas = fact_produccion_guardada.filter(
            F.col("tiempo_id").isNull()
            | F.col("cultivo_id").isNull()
            | F.col("ubicacion_produccion_id").isNull()
        ).count()

        claves_duplicadas = (
            fact_produccion_guardada
            .groupBy(
                "tiempo_id",
                "cultivo_id",
                "ubicacion_produccion_id"
            )
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        print("Claves foraneas nulas:", claves_nulas)
        print("Claves de hechos duplicadas:", claves_duplicadas)

        print("\n===== VALORES DISPONIBLES POR INDICADOR =====")

        medidas = [
            "produccion_t",
            "precio_chacra_s_kg",
            "cosechas_ha",
            "siembras_ha",
            "superficie_verde_ha",
            "superficie_perdida_ha",
            "rendimiento_kg_ha"
        ]

        resumen = fact_produccion_guardada.agg(
            *[
                F.count(F.col(medida)).alias(medida)
                for medida in medidas
            ]
        )

        resumen.show(truncate=False)

        if (
            registros_guardados == 561
            and columnas_guardadas == 10
            and claves_nulas == 0
            and claves_duplicadas == 0
        ):
            print("\nALMACENAMIENTO FACT PRODUCCION: CORRECTO")
        else:
            print("\nATENCION: Revisar tabla de hechos almacenada")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 55 FINALIZADO =====")

===== PASO 55: GUARDAR FACT PRODUCCION =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Leyendo desde HDFS

===== VERIFICACION HDFS =====
Registros guardados: 561
Columnas guardadas: 10

PROGRESO: 80% - Verificando integridad
Claves foraneas nulas: 0
Claves de hechos duplicadas: 0

===== VALORES DISPONIBLES POR INDICADOR =====
+------------+------------------+-----------+-----------+-------------------+---------------------+-----------------+
|produccion_t|precio_chacra_s_kg|cosechas_ha|siembras_ha|superficie_verde_ha|superficie_perdida_ha|rendimiento_kg_ha|
+------------+------------------+-----------+-----------+-------------------+---------------------+-----------------+
|267         |267               |34         |36         |561                |2                    |0                |
+------------+------------------+-----------+-----------+-------------------+---------------------+-----------------+


ALMACENAMIENTO FACT PRODUCCION: CORRE

## Modelo dimensional Gold: hechos de exportaciones regionales

Se construye la tabla de hechos de exportaciones regionales
a partir de los registros históricos disponibles en Gold.

La tabla contiene información mensual por cultivo durante
el periodo 2012–2023.

Se relaciona con las dimensiones de tiempo y cultivo
mediante claves foráneas.

Se conservan tres indicadores: volumen exportado en
toneladas, valor FOB en miles de dólares estadounidenses
y precio FOB en dólares por kilogramo.

Los precios no disponibles permanecen como valores nulos
y no se reemplazan por cero.

Esta información corresponde a exportaciones regionales
y se mantiene separada de las exportaciones propias de Danper.

In [41]:
from pyspark.sql import functions as F

print("===== PASO 56: FACT EXPORTACIONES REGIONALES =====")
print("PROGRESO: 0% - Leyendo tablas Gold", flush=True)

# 1. Leer las tablas previamente almacenadas
exportaciones_regionales_origen = spark.read.parquet(
    f"{BASE_GOLD}/exportaciones_regionales"
)

tiempo_exp = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
)

cultivo_exp = spark.read.parquet(
    f"{BASE_GOLD}/dim_cultivo"
)

total_origen = exportaciones_regionales_origen.count()

print("Registros originales:", total_origen)

print("\nPROGRESO: 25% - Relacionando dimensiones", flush=True)

# 2. Asociar cada exportacion con su mes y cultivo.
# LEFT JOIN permite detectar relaciones faltantes.
exportaciones_relacionadas = (
    exportaciones_regionales_origen.alias("e")
    .join(
        tiempo_exp.alias("t"),
        F.col("e.periodo") == F.col("t.periodo"),
        "left"
    )
    .join(
        cultivo_exp.alias("c"),
        F.col("e.cultivo") == F.col("c.cultivo"),
        "left"
    )
)

print("\nPROGRESO: 50% - Construyendo tabla de hechos", flush=True)

# 3. Seleccionar claves dimensionales y medidas
fact_exportaciones_regionales = (
    exportaciones_relacionadas
    .select(
        F.col("t.tiempo_id").alias("tiempo_id"),
        F.col("c.cultivo_id").alias("cultivo_id"),
        F.col("e.volumen_exportado_t").alias(
            "volumen_exportado_t"
        ),
        F.col("e.valor_fob_miles_usd").alias(
            "valor_fob_miles_usd"
        ),
        F.col("e.precio_fob_usd_kg").alias(
            "precio_fob_usd_kg"
        )
    )
)

print("\n===== ESQUEMA FACT EXPORTACIONES REGIONALES =====")
fact_exportaciones_regionales.printSchema()

print("\nPROGRESO: 70% - Validando integridad", flush=True)

total_fact = fact_exportaciones_regionales.count()

claves_nulas = fact_exportaciones_regionales.filter(
    F.col("tiempo_id").isNull()
    | F.col("cultivo_id").isNull()
).count()

duplicados = (
    fact_exportaciones_regionales
    .groupBy("tiempo_id", "cultivo_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

precios_nulos_origen = exportaciones_regionales_origen.filter(
    F.col("precio_fob_usd_kg").isNull()
).count()

precios_nulos_fact = fact_exportaciones_regionales.filter(
    F.col("precio_fob_usd_kg").isNull()
).count()

print("\n===== VALIDACION =====")
print("Registros originales:", total_origen)
print("Registros tabla de hechos:", total_fact)
print("Claves foraneas nulas:", claves_nulas)
print("Claves duplicadas:", duplicados)
print("Precios FOB nulos originales:", precios_nulos_origen)
print("Precios FOB nulos en fact:", precios_nulos_fact)

print("\nPROGRESO: 85% - Revisando resultados", flush=True)

print("\n===== REGISTROS POR CULTIVO =====")

fact_exportaciones_regionales.alias("f").join(
    cultivo_exp.alias("c"),
    F.col("f.cultivo_id") == F.col("c.cultivo_id"),
    "left"
).groupBy(
    F.col("c.cultivo")
).count().orderBy(
    "cultivo"
).show(truncate=False)

print("\n===== MUESTRA DE LA TABLA =====")

fact_exportaciones_regionales.orderBy(
    "tiempo_id",
    "cultivo_id"
).show(12, truncate=False)

print("\n===== RESULTADO FINAL =====")

if (
    total_origen == 720
    and total_fact == total_origen
    and claves_nulas == 0
    and duplicados == 0
    and precios_nulos_origen == precios_nulos_fact == 79
):
    fact_exportaciones_regionales_validada = True
    print(
        "TABLA DE HECHOS EXPORTACIONES REGIONALES: "
        "CONSTRUIDA CORRECTAMENTE"
    )
else:
    fact_exportaciones_regionales_validada = False
    print("ATENCION: REVISAR TABLA DE HECHOS")

print("\nPROGRESO: 100% - Construccion finalizada")
print("===== PASO 56 FINALIZADO =====")

===== PASO 56: FACT EXPORTACIONES REGIONALES =====
PROGRESO: 0% - Leyendo tablas Gold
Registros originales: 720

PROGRESO: 25% - Relacionando dimensiones

PROGRESO: 50% - Construyendo tabla de hechos

===== ESQUEMA FACT EXPORTACIONES REGIONALES =====
root
 |-- tiempo_id: integer (nullable = true)
 |-- cultivo_id: integer (nullable = true)
 |-- volumen_exportado_t: double (nullable = true)
 |-- valor_fob_miles_usd: double (nullable = true)
 |-- precio_fob_usd_kg: double (nullable = true)


PROGRESO: 70% - Validando integridad

===== VALIDACION =====
Registros originales: 720
Registros tabla de hechos: 720
Claves foraneas nulas: 0
Claves duplicadas: 0
Precios FOB nulos originales: 79
Precios FOB nulos en fact: 79

PROGRESO: 85% - Revisando resultados

===== REGISTROS POR CULTIVO =====
+---------+-----+
|cultivo  |count|
+---------+-----+
|ALCACHOFA|144  |
|ARANDANO |144  |
|ESPARRAGO|144  |
|PALTA    |144  |
|UVA      |144  |
+---------+-----+


===== MUESTRA DE LA TABLA =====
+---------

## Validación de medidas: hechos de exportaciones regionales

Se verifica que la tabla de hechos de exportaciones regionales
conserve los 720 registros y las medidas originales de Gold.

Se comparan individualmente el volumen exportado en toneladas,
el valor FOB en miles de dólares y el precio FOB en USD/kg.

La comparación utiliza las claves de tiempo y cultivo,
conservando los valores nulos y evitando modificaciones
en los indicadores originales.

Los datos corresponden al periodo histórico 2012–2023
y no representan exportaciones propias de Danper.

In [42]:
from pyspark.sql import functions as F

print("===== PASO 57: VALIDACION MEDIDAS EXPORTACIONES =====")
print("PROGRESO: 0% - Preparando validacion", flush=True)

medidas_exportaciones = [
    "volumen_exportado_t",
    "valor_fob_miles_usd",
    "precio_fob_usd_kg"
]

claves_exportaciones = [
    "tiempo_id",
    "cultivo_id"
]

# 1. Preparar origen con las mismas claves de la tabla de hechos
origen_comparacion_exp = (
    exportaciones_regionales_origen
    .withColumn(
        "tiempo_id",
        F.year("periodo") * 100 + F.month("periodo")
    )
    .join(
        cultivo_exp.select("cultivo_id", "cultivo"),
        "cultivo",
        "left"
    )
    .select(
        *claves_exportaciones,
        *medidas_exportaciones
    )
)

print("PROGRESO: 25% - Revisando registros", flush=True)

registros_origen = origen_comparacion_exp.count()
registros_fact = fact_exportaciones_regionales.count()

print("Registros origen:", registros_origen)
print("Registros fact:", registros_fact)

# 2. Comparar valores, incluyendo NULL
print("\nPROGRESO: 50% - Comparando medidas", flush=True)

columnas = claves_exportaciones + medidas_exportaciones

diferencias_origen = (
    origen_comparacion_exp
    .select(*columnas)
    .exceptAll(
        fact_exportaciones_regionales.select(*columnas)
    )
    .count()
)

diferencias_fact = (
    fact_exportaciones_regionales
    .select(*columnas)
    .exceptAll(
        origen_comparacion_exp.select(*columnas)
    )
    .count()
)

total_diferencias = diferencias_origen + diferencias_fact

print("Filas diferentes:", total_diferencias)

# 3. Verificar disponibilidad de cada indicador
print("\nPROGRESO: 75% - Verificando valores nulos", flush=True)

nulos_iguales = True

print("\n===== VALORES NULOS POR INDICADOR =====")

for medida in medidas_exportaciones:

    nulos_origen = origen_comparacion_exp.filter(
        F.col(medida).isNull()
    ).count()

    nulos_fact = fact_exportaciones_regionales.filter(
        F.col(medida).isNull()
    ).count()

    print(
        f"{medida}: origen={nulos_origen}, fact={nulos_fact}"
    )

    if nulos_origen != nulos_fact:
        nulos_iguales = False

# 4. Resultado final
print("\n===== RESULTADO FINAL =====")

if (
    registros_origen == registros_fact == 720
    and total_diferencias == 0
    and nulos_iguales
    and globals().get(
        "fact_exportaciones_regionales_validada", False
    )
):
    fact_exportaciones_medidas_validada = True
    print("VALIDACION MEDIDAS EXPORTACIONES: CORRECTA")
else:
    fact_exportaciones_medidas_validada = False
    print("ATENCION: REVISAR MEDIDAS DE EXPORTACIONES")

print("\nPROGRESO: 100% - Validacion finalizada")
print("===== PASO 57 FINALIZADO =====")

===== PASO 57: VALIDACION MEDIDAS EXPORTACIONES =====
PROGRESO: 0% - Preparando validacion
PROGRESO: 25% - Revisando registros
Registros origen: 720
Registros fact: 720

PROGRESO: 50% - Comparando medidas
Filas diferentes: 0

PROGRESO: 75% - Verificando valores nulos

===== VALORES NULOS POR INDICADOR =====
volumen_exportado_t: origen=38, fact=38
valor_fob_miles_usd: origen=0, fact=0
precio_fob_usd_kg: origen=79, fact=79

===== RESULTADO FINAL =====
VALIDACION MEDIDAS EXPORTACIONES: CORRECTA

PROGRESO: 100% - Validacion finalizada
===== PASO 57 FINALIZADO =====


## Almacenamiento Gold: tabla de hechos de exportaciones regionales

Se almacena la tabla de hechos de exportaciones regionales
del modelo dimensional agrocomercial Danper en Hadoop HDFS.

La tabla contiene 720 registros correspondientes al periodo
2012–2023 y se relaciona con las dimensiones de tiempo
y cultivo mediante claves foráneas.

Se conservan el volumen exportado en toneladas, el valor FOB
en miles de dólares y el precio FOB en dólares por kilogramo.

Los valores faltantes permanecen como nulos y las medidas
fueron validadas previamente respecto a la tabla Gold original.

La información representa exportaciones regionales históricas
y no exportaciones propias de Danper.

In [43]:
from pyspark.sql import functions as F

print("===== PASO 58: GUARDAR FACT EXPORTACIONES REGIONALES =====")

if not (
    globals().get("fact_exportaciones_regionales_validada", False)
    and globals().get("fact_exportaciones_medidas_validada", False)
):
    print("ERROR: Primero debes validar los Pasos 56 y 57.")

else:
    ruta_fact_exportaciones = (
        f"{BASE_GOLD}/fact_exportaciones_regionales"
    )

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_fact_exportaciones
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La tabla ya existe en HDFS.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_fact_exportaciones)

    else:
        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            fact_exportaciones_regionales
            .coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_fact_exportaciones)
        )

        print("PROGRESO: 60% - Leyendo desde HDFS", flush=True)

        fact_exportaciones_guardada = spark.read.parquet(
            ruta_fact_exportaciones
        )

        registros_guardados = fact_exportaciones_guardada.count()
        columnas_guardadas = len(fact_exportaciones_guardada.columns)

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print("\nPROGRESO: 80% - Verificando integridad", flush=True)

        claves_nulas = fact_exportaciones_guardada.filter(
            F.col("tiempo_id").isNull()
            | F.col("cultivo_id").isNull()
        ).count()

        claves_duplicadas = (
            fact_exportaciones_guardada
            .groupBy("tiempo_id", "cultivo_id")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        precios_nulos = fact_exportaciones_guardada.filter(
            F.col("precio_fob_usd_kg").isNull()
        ).count()

        volumenes_nulos = fact_exportaciones_guardada.filter(
            F.col("volumen_exportado_t").isNull()
        ).count()

        print("Claves foraneas nulas:", claves_nulas)
        print("Claves duplicadas:", claves_duplicadas)
        print("Precios FOB nulos:", precios_nulos)
        print("Volumenes exportados nulos:", volumenes_nulos)

        print("\n===== REGISTROS POR CULTIVO =====")

        (
            fact_exportaciones_guardada.alias("f")
            .join(
                cultivo_exp.select(
                    "cultivo_id", "cultivo"
                ).alias("c"),
                "cultivo_id",
                "left"
            )
            .groupBy("cultivo")
            .count()
            .orderBy("cultivo")
            .show(truncate=False)
        )

        if (
            registros_guardados == 720
            and columnas_guardadas == 5
            and claves_nulas == 0
            and claves_duplicadas == 0
            and precios_nulos == 79
            and volumenes_nulos == 38
        ):
            print("\nALMACENAMIENTO FACT EXPORTACIONES: CORRECTO")
        else:
            print("\nATENCION: Revisar tabla almacenada")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 58 FINALIZADO =====")

===== PASO 58: GUARDAR FACT EXPORTACIONES REGIONALES =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Leyendo desde HDFS

===== VERIFICACION HDFS =====
Registros guardados: 720
Columnas guardadas: 5

PROGRESO: 80% - Verificando integridad
Claves foraneas nulas: 0
Claves duplicadas: 0
Precios FOB nulos: 79
Volumenes exportados nulos: 38

===== REGISTROS POR CULTIVO =====
+---------+-----+
|cultivo  |count|
+---------+-----+
|ALCACHOFA|144  |
|ARANDANO |144  |
|ESPARRAGO|144  |
|PALTA    |144  |
|UVA      |144  |
+---------+-----+


ALMACENAMIENTO FACT EXPORTACIONES: CORRECTO
PROGRESO: 100% - Verificacion finalizada
===== PASO 58 FINALIZADO =====


## Modelo dimensional Gold: hechos de exportaciones Danper

Se construye la tabla de hechos de exportaciones propias
de Danper a partir de los registros analíticos disponibles
en la capa Gold.

Los datos se organizan por país de destino y periodo
mensual, conservando el valor FOB en dólares estadounidenses
y la cantidad de registros aduaneros.

La tabla se relaciona con las dimensiones de tiempo y país
de exportación mediante identificadores únicos.

Los registros de 2026 representan información parcial.

No se asignan cultivos a estas exportaciones porque la
fuente original no contiene esa identificación.

In [44]:
from pyspark.sql import functions as F

print("===== PASO 59: FACT EXPORTACIONES DANPER =====")
print("PROGRESO: 0% - Leyendo tablas Gold", flush=True)

# Leer fuentes desde HDFS
danper_fact_origen = spark.read.parquet(
    f"{BASE_GOLD}/exportaciones_danper"
)

tiempo_danper = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
)

paises_danper = spark.read.parquet(
    f"{BASE_GOLD}/dim_pais_exportacion"
)

total_origen_danper = danper_fact_origen.count()

print("Registros originales:", total_origen_danper)

print("\nPROGRESO: 25% - Relacionando dimensiones", flush=True)

# Asociar mes y pais sin perder registros originales
danper_relacionada = (
    danper_fact_origen.alias("e")
    .join(
        tiempo_danper.alias("t"),
        F.col("e.periodo") == F.col("t.periodo"),
        "left"
    )
    .join(
        paises_danper.alias("p"),
        F.upper(F.trim(F.col("e.pais"))) == F.col("p.pais"),
        "left"
    )
)

print("\nPROGRESO: 50% - Construyendo tabla de hechos", flush=True)

fact_exportaciones_danper = danper_relacionada.select(
    F.col("t.tiempo_id").alias("tiempo_id"),
    F.col("p.pais_id").alias("pais_id"),
    F.col("e.valor_fob_usd").alias("valor_fob_usd"),
    F.col("e.cantidad_registros_aduaneros").alias(
        "cantidad_registros_aduaneros"
    )
)

print("\n===== ESQUEMA FACT EXPORTACIONES DANPER =====")
fact_exportaciones_danper.printSchema()

print("\nPROGRESO: 75% - Verificando integridad", flush=True)

total_fact_danper = fact_exportaciones_danper.count()

claves_nulas_danper = fact_exportaciones_danper.filter(
    F.col("tiempo_id").isNull()
    | F.col("pais_id").isNull()
).count()

duplicados_fact_danper = (
    fact_exportaciones_danper
    .groupBy("tiempo_id", "pais_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\n===== VALIDACION =====")
print("Registros originales:", total_origen_danper)
print("Registros fact:", total_fact_danper)
print("Claves foraneas nulas:", claves_nulas_danper)
print("Claves duplicadas:", duplicados_fact_danper)

print("\n===== REGISTROS POR ANIO =====")

fact_exportaciones_danper.withColumn(
    "anio",
    F.floor(F.col("tiempo_id") / 100)
).groupBy("anio") \
    .count() \
    .orderBy("anio") \
    .show(truncate=False)

print("\n===== MUESTRA FACT =====")

fact_exportaciones_danper.orderBy(
    "tiempo_id", "pais_id"
).show(12, truncate=False)

print("\n===== RESULTADO FINAL =====")

if (
    total_origen_danper == 495
    and total_fact_danper == total_origen_danper
    and claves_nulas_danper == 0
    and duplicados_fact_danper == 0
):
    fact_danper_validada = True
    print(
        "TABLA DE HECHOS EXPORTACIONES DANPER: "
        "CONSTRUIDA CORRECTAMENTE"
    )
else:
    fact_danper_validada = False
    print("ATENCION: REVISAR RELACIONES DIMENSIONALES")

print("\nPROGRESO: 100% - Construccion finalizada")
print("===== PASO 59 FINALIZADO =====")

===== PASO 59: FACT EXPORTACIONES DANPER =====
PROGRESO: 0% - Leyendo tablas Gold
Registros originales: 495

PROGRESO: 25% - Relacionando dimensiones

PROGRESO: 50% - Construyendo tabla de hechos

===== ESQUEMA FACT EXPORTACIONES DANPER =====
root
 |-- tiempo_id: integer (nullable = true)
 |-- pais_id: integer (nullable = true)
 |-- valor_fob_usd: double (nullable = true)
 |-- cantidad_registros_aduaneros: long (nullable = true)


PROGRESO: 75% - Verificando integridad

===== VALIDACION =====
Registros originales: 495
Registros fact: 495
Claves foraneas nulas: 0
Claves duplicadas: 0

===== REGISTROS POR ANIO =====
+----+-----+
|anio|count|
+----+-----+
|2024|204  |
|2025|223  |
|2026|68   |
+----+-----+


===== MUESTRA FACT =====
+---------+-------+------------------+----------------------------+
|tiempo_id|pais_id|valor_fob_usd     |cantidad_registros_aduaneros|
+---------+-------+------------------+----------------------------+
|202401   |3      |68720.0           |1                 

## Validación de medidas: hechos de exportaciones Danper

Se verifica que la tabla de hechos de exportaciones Danper
conserve los registros y las medidas originales de Gold.

Se comparan individualmente el valor FOB en dólares
estadounidenses y la cantidad de registros aduaneros
por país de destino y periodo mensual.

La comparación utiliza las claves dimensionales de tiempo
y país, conservando los valores originales y sus posibles nulos.

Esta validación garantiza que las relaciones dimensionales
no alteren los indicadores de exportación de Danper.

Los datos de 2026 representan información parcial.

In [45]:
from pyspark.sql import functions as F

print("===== PASO 60: VALIDACION MEDIDAS EXPORTACIONES DANPER =====")
print("PROGRESO: 0% - Preparando validacion", flush=True)

medidas_danper = [
    "valor_fob_usd",
    "cantidad_registros_aduaneros"
]

claves_danper = [
    "tiempo_id",
    "pais_id"
]

# 1. Preparar el origen con las claves dimensionales
origen_comparacion_danper = (
    danper_fact_origen.alias("e")
    .join(
        paises_danper.select("pais_id", "pais").alias("p"),
        F.upper(F.trim(F.col("e.pais"))) == F.col("p.pais"),
        "left"
    )
    .select(
        (
            F.year(F.col("e.periodo")) * 100
            + F.month(F.col("e.periodo"))
        ).alias("tiempo_id"),
        F.col("p.pais_id").alias("pais_id"),
        F.col("e.valor_fob_usd").alias("valor_fob_usd"),
        F.col("e.cantidad_registros_aduaneros").alias(
            "cantidad_registros_aduaneros"
        )
    )
)

print("\nPROGRESO: 25% - Comparando registros", flush=True)

registros_origen = origen_comparacion_danper.count()
registros_fact = fact_exportaciones_danper.count()

print("Registros origen:", registros_origen)
print("Registros fact:", registros_fact)

# 2. Comparar las medidas registro por registro
print("\nPROGRESO: 50% - Comparando valores", flush=True)

columnas = claves_danper + medidas_danper

diferencias_origen = (
    origen_comparacion_danper
    .select(*columnas)
    .exceptAll(
        fact_exportaciones_danper.select(*columnas)
    )
    .count()
)

diferencias_fact = (
    fact_exportaciones_danper
    .select(*columnas)
    .exceptAll(
        origen_comparacion_danper.select(*columnas)
    )
    .count()
)

total_diferencias_danper = (
    diferencias_origen + diferencias_fact
)

print("Filas diferentes:", total_diferencias_danper)

# 3. Verificar la conservación de valores nulos
print("\nPROGRESO: 75% - Verificando valores faltantes", flush=True)

nulos_iguales_danper = True

print("\n===== VALORES NULOS POR INDICADOR =====")

for medida in medidas_danper:

    nulos_origen = origen_comparacion_danper.filter(
        F.col(medida).isNull()
    ).count()

    nulos_fact = fact_exportaciones_danper.filter(
        F.col(medida).isNull()
    ).count()

    print(
        f"{medida}: origen={nulos_origen}, fact={nulos_fact}"
    )

    if nulos_origen != nulos_fact:
        nulos_iguales_danper = False

# 4. Verificación final
print("\n===== RESULTADO FINAL =====")

if (
    registros_origen == registros_fact == 495
    and total_diferencias_danper == 0
    and nulos_iguales_danper
    and globals().get("fact_danper_validada", False)
):
    fact_danper_medidas_validada = True
    print("VALIDACION MEDIDAS DANPER: CORRECTA")
else:
    fact_danper_medidas_validada = False
    print("ATENCION: REVISAR MEDIDAS DE EXPORTACIONES DANPER")

print("\nPROGRESO: 100% - Validacion finalizada")
print("===== PASO 60 FINALIZADO =====")

===== PASO 60: VALIDACION MEDIDAS EXPORTACIONES DANPER =====
PROGRESO: 0% - Preparando validacion

PROGRESO: 25% - Comparando registros
Registros origen: 495
Registros fact: 495

PROGRESO: 50% - Comparando valores
Filas diferentes: 0

PROGRESO: 75% - Verificando valores faltantes

===== VALORES NULOS POR INDICADOR =====
valor_fob_usd: origen=0, fact=0
cantidad_registros_aduaneros: origen=0, fact=0

===== RESULTADO FINAL =====
VALIDACION MEDIDAS DANPER: CORRECTA

PROGRESO: 100% - Validacion finalizada
===== PASO 60 FINALIZADO =====


## Almacenamiento Gold: tabla de hechos de exportaciones Danper

Se almacena en Hadoop HDFS la tabla de hechos de exportaciones
propias de Danper, utilizando el formato Parquet.

La tabla contiene 495 registros correspondientes al periodo
2024–2026 y se relaciona con las dimensiones de tiempo
y país de exportación mediante claves foráneas.

Se conservan el valor FOB en dólares estadounidenses y
la cantidad de registros aduaneros por país y periodo mensual.

Las medidas fueron validadas previamente, comprobándose
que no existen diferencias respecto a la tabla Gold original.

Los datos de 2026 son parciales y la fuente no identifica
el cultivo exportado, por lo que no se atribuyen los valores
FOB a productos específicos.

In [46]:
from pyspark.sql import functions as F

print("===== PASO 61: GUARDAR FACT EXPORTACIONES DANPER =====")

if not (
    globals().get("fact_danper_validada", False)
    and globals().get("fact_danper_medidas_validada", False)
):
    print("ERROR: Primero debes validar los Pasos 59 y 60.")

else:
    ruta_fact_danper = (
        f"{BASE_GOLD}/fact_exportaciones_danper"
    )

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_fact_danper
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La tabla ya existe en HDFS.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_fact_danper)

    else:
        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            fact_exportaciones_danper
            .coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_fact_danper)
        )

        print("PROGRESO: 60% - Leyendo desde HDFS", flush=True)

        fact_danper_guardada = spark.read.parquet(
            ruta_fact_danper
        )

        registros_guardados = fact_danper_guardada.count()
        columnas_guardadas = len(fact_danper_guardada.columns)

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print("\nPROGRESO: 80% - Verificando integridad", flush=True)

        claves_nulas = fact_danper_guardada.filter(
            F.col("tiempo_id").isNull()
            | F.col("pais_id").isNull()
        ).count()

        claves_duplicadas = (
            fact_danper_guardada
            .groupBy("tiempo_id", "pais_id")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        fob_nulos = fact_danper_guardada.filter(
            F.col("valor_fob_usd").isNull()
        ).count()

        registros_aduaneros_nulos = fact_danper_guardada.filter(
            F.col("cantidad_registros_aduaneros").isNull()
        ).count()

        print("Claves foraneas nulas:", claves_nulas)
        print("Claves duplicadas:", claves_duplicadas)
        print("Valores FOB nulos:", fob_nulos)
        print(
            "Cantidades aduaneras nulas:",
            registros_aduaneros_nulos
        )

        print("\n===== REGISTROS POR ANIO =====")

        (
            fact_danper_guardada
            .withColumn(
                "anio",
                F.floor(F.col("tiempo_id") / 100)
            )
            .groupBy("anio")
            .agg(
                F.count("*").alias("registros"),
                F.sum("cantidad_registros_aduaneros").alias(
                    "registros_aduaneros"
                )
            )
            .orderBy("anio")
            .show(truncate=False)
        )

        if (
            registros_guardados == 495
            and columnas_guardadas == 4
            and claves_nulas == 0
            and claves_duplicadas == 0
            and fob_nulos == 0
            and registros_aduaneros_nulos == 0
        ):
            print("\nALMACENAMIENTO FACT DANPER: CORRECTO")
        else:
            print("\nATENCION: Revisar tabla almacenada")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 61 FINALIZADO =====")

===== PASO 61: GUARDAR FACT EXPORTACIONES DANPER =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Leyendo desde HDFS

===== VERIFICACION HDFS =====
Registros guardados: 495
Columnas guardadas: 4

PROGRESO: 80% - Verificando integridad
Claves foraneas nulas: 0
Claves duplicadas: 0
Valores FOB nulos: 0
Cantidades aduaneras nulas: 0

===== REGISTROS POR ANIO =====
+----+---------+-------------------+
|anio|registros|registros_aduaneros|
+----+---------+-------------------+
|2024|204      |444                |
|2025|223      |542                |
|2026|68       |138                |
+----+---------+-------------------+


ALMACENAMIENTO FACT DANPER: CORRECTO
PROGRESO: 100% - Verificacion finalizada
===== PASO 61 FINALIZADO =====


## Modelo dimensional Gold: tabla de hechos de agroindustria

Se construye la tabla de hechos de agroindustria a partir
de los registros analíticos disponibles en la capa Gold.

La información corresponde a espárrago y alcachofa,
con datos históricos del periodo 2010–2024.

La tabla se relaciona con las dimensiones de tiempo y
cultivo mediante claves foráneas.

Se conservan los procesos y las presentaciones originales,
así como las cantidades expresadas en toneladas.

Los procesos INGRESO, PRODUCCION, UTILIZACION y VENTA
representan operaciones diferentes y no deben agregarse
indiscriminadamente.

Se comprueba que la transformación no genere duplicados,
relaciones faltantes ni modificaciones en las cantidades.

In [47]:
from pyspark.sql import functions as F

print("===== PASO 62: FACT AGROINDUSTRIA MENSUAL =====")
print("PROGRESO: 0% - Leyendo tablas Gold", flush=True)

# 1. Leer los datos y las dimensiones desde HDFS.
agro_fact_origen = spark.read.parquet(
    f"{BASE_GOLD}/agroindustria_mensual"
)

tiempo_agro = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
)

cultivo_agro = spark.read.parquet(
    f"{BASE_GOLD}/dim_cultivo"
)

total_origen_agro = agro_fact_origen.count()

print("Registros originales:", total_origen_agro)

print("\nPROGRESO: 25% - Relacionando dimensiones", flush=True)

# 2. Relacionar tiempo y cultivo mediante LEFT JOIN.
agro_relacionada = (
    agro_fact_origen.alias("a")
    .join(
        tiempo_agro.select(
            "periodo", "tiempo_id"
        ).alias("t"),
        F.col("a.periodo") == F.col("t.periodo"),
        "left"
    )
    .join(
        cultivo_agro.select(
            "cultivo", "cultivo_id"
        ).alias("c"),
        F.col("a.cultivo") == F.col("c.cultivo"),
        "left"
    )
)

print("\nPROGRESO: 50% - Construyendo tabla de hechos", flush=True)

# 3. Mantener el nivel de detalle original.
fact_agroindustria_mensual = agro_relacionada.select(
    F.col("t.tiempo_id").alias("tiempo_id"),
    F.col("c.cultivo_id").alias("cultivo_id"),
    F.col("a.proceso").alias("proceso"),
    F.col("a.presentacion").alias("presentacion"),
    F.col("a.cantidad_t").alias("cantidad_t")
)

print("\n===== ESQUEMA FACT AGROINDUSTRIA =====")
fact_agroindustria_mensual.printSchema()

print("\nPROGRESO: 70% - Verificando integridad", flush=True)

total_fact_agro = fact_agroindustria_mensual.count()

claves_nulas_agro = fact_agroindustria_mensual.filter(
    F.col("tiempo_id").isNull()
    | F.col("cultivo_id").isNull()
).count()

claves_agro = [
    "tiempo_id",
    "cultivo_id",
    "proceso",
    "presentacion"
]

duplicados_fact_agro = (
    fact_agroindustria_mensual
    .groupBy(*claves_agro)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\n===== VALIDACION =====")
print("Registros originales:", total_origen_agro)
print("Registros tabla de hechos:", total_fact_agro)
print("Claves foraneas nulas:", claves_nulas_agro)
print("Claves de hechos duplicadas:", duplicados_fact_agro)

print("\nPROGRESO: 85% - Revisando resultados", flush=True)

print("\n===== REGISTROS POR CULTIVO Y PROCESO =====")

(
    fact_agroindustria_mensual.alias("f")
    .join(
        cultivo_agro.select(
            "cultivo_id", "cultivo"
        ).alias("c"),
        "cultivo_id",
        "left"
    )
    .groupBy("cultivo", "proceso")
    .count()
    .orderBy("cultivo", "proceso")
    .show(truncate=False)
)

print("\n===== MUESTRA FACT =====")

fact_agroindustria_mensual.orderBy(
    "tiempo_id",
    "cultivo_id",
    "proceso",
    "presentacion"
).show(12, truncate=False)

print("\n===== RESULTADO FINAL =====")

if (
    total_origen_agro == 3134
    and total_fact_agro == total_origen_agro
    and claves_nulas_agro == 0
    and duplicados_fact_agro == 0
):
    fact_agroindustria_validada = True

    print(
        "TABLA DE HECHOS AGROINDUSTRIA: "
        "CONSTRUIDA CORRECTAMENTE"
    )
else:
    fact_agroindustria_validada = False

    print(
        "ATENCION: REVISAR RELACIONES "
        "DE AGROINDUSTRIA"
    )

print("\nPROGRESO: 100% - Construccion finalizada")
print("===== PASO 62 FINALIZADO =====")

===== PASO 62: FACT AGROINDUSTRIA MENSUAL =====
PROGRESO: 0% - Leyendo tablas Gold
Registros originales: 3134

PROGRESO: 25% - Relacionando dimensiones

PROGRESO: 50% - Construyendo tabla de hechos

===== ESQUEMA FACT AGROINDUSTRIA =====
root
 |-- tiempo_id: integer (nullable = true)
 |-- cultivo_id: integer (nullable = true)
 |-- proceso: string (nullable = true)
 |-- presentacion: string (nullable = true)
 |-- cantidad_t: double (nullable = true)


PROGRESO: 70% - Verificando integridad

===== VALIDACION =====
Registros originales: 3134
Registros tabla de hechos: 3134
Claves foraneas nulas: 0
Claves de hechos duplicadas: 0

PROGRESO: 85% - Revisando resultados

===== REGISTROS POR CULTIVO Y PROCESO =====
+---------+-----------+-----+
|cultivo  |proceso    |count|
+---------+-----------+-----+
|ALCACHOFA|INGRESO    |180  |
|ALCACHOFA|PRODUCCION |540  |
|ALCACHOFA|UTILIZACION|180  |
|ALCACHOFA|VENTA      |540  |
|ESPARRAGO|INGRESO    |180  |
|ESPARRAGO|PRODUCCION |662  |
|ESPARRAGO|UTI

## Validación de medidas: hechos de agroindustria

Se verifica que la tabla de hechos de agroindustria
conserve los 3 134 registros y las cantidades originales
de la capa Gold.

La comparación se realiza considerando el mes, cultivo,
proceso y presentación agroindustrial.

Se comprueba individualmente la conservación de las
cantidades en toneladas, incluyendo valores nulos
y valores iguales a cero.

Los procesos INGRESO, PRODUCCION, UTILIZACION y VENTA
se mantienen independientes para evitar agregaciones
que mezclen operaciones diferentes.

Esta validación garantiza que la transformación
dimensional no altere la información original.

In [48]:
from pyspark.sql import functions as F

print("===== PASO 63: VALIDACION MEDIDAS AGROINDUSTRIA =====")
print("PROGRESO: 0% - Preparando validacion", flush=True)

# 1. Preparar datos originales con claves dimensionales
origen_comparacion_agro = (
    agro_fact_origen.alias("a")
    .join(
        cultivo_agro.select(
            "cultivo_id", "cultivo"
        ).alias("c"),
        F.col("a.cultivo") == F.col("c.cultivo"),
        "left"
    )
    .select(
        (
            F.year(F.col("a.periodo")) * 100
            + F.month(F.col("a.periodo"))
        ).alias("tiempo_id"),
        F.col("c.cultivo_id").alias("cultivo_id"),
        F.col("a.proceso").alias("proceso"),
        F.col("a.presentacion").alias("presentacion"),
        F.col("a.cantidad_t").alias("cantidad_t")
    )
)

print("\nPROGRESO: 25% - Comparando registros", flush=True)

registros_origen_agro = origen_comparacion_agro.count()
registros_fact_agro = fact_agroindustria_mensual.count()

print("Registros origen:", registros_origen_agro)
print("Registros fact:", registros_fact_agro)

# 2. Comparar valores individuales, incluyendo NULL y cero
print("\nPROGRESO: 50% - Comparando cantidades", flush=True)

columnas_agro = [
    "tiempo_id",
    "cultivo_id",
    "proceso",
    "presentacion",
    "cantidad_t"
]

diferencias_origen_agro = (
    origen_comparacion_agro
    .select(*columnas_agro)
    .exceptAll(
        fact_agroindustria_mensual.select(*columnas_agro)
    )
    .count()
)

diferencias_fact_agro = (
    fact_agroindustria_mensual
    .select(*columnas_agro)
    .exceptAll(
        origen_comparacion_agro.select(*columnas_agro)
    )
    .count()
)

total_diferencias_agro = (
    diferencias_origen_agro + diferencias_fact_agro
)

print("Filas diferentes:", total_diferencias_agro)

# 3. Verificar valores nulos y ceros
print("\nPROGRESO: 75% - Verificando valores faltantes", flush=True)

nulos_origen_agro = origen_comparacion_agro.filter(
    F.col("cantidad_t").isNull()
).count()

nulos_fact_agro = fact_agroindustria_mensual.filter(
    F.col("cantidad_t").isNull()
).count()

ceros_origen_agro = origen_comparacion_agro.filter(
    F.col("cantidad_t") == 0
).count()

ceros_fact_agro = fact_agroindustria_mensual.filter(
    F.col("cantidad_t") == 0
).count()

print("Cantidades nulas origen:", nulos_origen_agro)
print("Cantidades nulas fact:", nulos_fact_agro)
print("Cantidades iguales a cero origen:", ceros_origen_agro)
print("Cantidades iguales a cero fact:", ceros_fact_agro)

# 4. Resultado final
print("\n===== RESULTADO FINAL =====")

if (
    registros_origen_agro == registros_fact_agro == 3134
    and total_diferencias_agro == 0
    and nulos_origen_agro == nulos_fact_agro
    and ceros_origen_agro == ceros_fact_agro
    and globals().get("fact_agroindustria_validada", False)
):
    fact_agroindustria_medidas_validada = True
    print("VALIDACION MEDIDAS AGROINDUSTRIA: CORRECTA")
else:
    fact_agroindustria_medidas_validada = False
    print("ATENCION: REVISAR MEDIDAS DE AGROINDUSTRIA")

print("\nPROGRESO: 100% - Validacion finalizada")
print("===== PASO 63 FINALIZADO =====")

===== PASO 63: VALIDACION MEDIDAS AGROINDUSTRIA =====
PROGRESO: 0% - Preparando validacion

PROGRESO: 25% - Comparando registros
Registros origen: 3134
Registros fact: 3134

PROGRESO: 50% - Comparando cantidades
Filas diferentes: 0

PROGRESO: 75% - Verificando valores faltantes
Cantidades nulas origen: 0
Cantidades nulas fact: 0
Cantidades iguales a cero origen: 978
Cantidades iguales a cero fact: 978

===== RESULTADO FINAL =====
VALIDACION MEDIDAS AGROINDUSTRIA: CORRECTA

PROGRESO: 100% - Validacion finalizada
===== PASO 63 FINALIZADO =====


## Almacenamiento Gold: tabla de hechos de agroindustria

Se almacena en Hadoop HDFS la tabla de hechos de agroindustria
del modelo dimensional agrocomercial Danper, utilizando
el formato Parquet.

La tabla contiene 3 134 registros correspondientes al
periodo histórico 2010–2024, organizados por tiempo,
cultivo, proceso y presentación agroindustrial.

Se conservan las cantidades en toneladas, incluyendo
978 valores iguales a cero y ningún valor nulo.

La validación previa confirmó que no existen diferencias
respecto a los registros originales de Gold.

Los procesos INGRESO, PRODUCCION, UTILIZACION y VENTA
se mantienen separados, debido a que representan
operaciones agroindustriales diferentes.

In [49]:
from pyspark.sql import functions as F

print("===== PASO 64: GUARDAR FACT AGROINDUSTRIA =====")

if not (
    globals().get("fact_agroindustria_validada", False)
    and globals().get("fact_agroindustria_medidas_validada", False)
):
    print("ERROR: Primero debes validar los Pasos 62 y 63.")

else:
    ruta_fact_agro = f"{BASE_GOLD}/fact_agroindustria_mensual"

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_fact_agro
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La tabla ya existe en HDFS.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_fact_agro)

    else:
        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            fact_agroindustria_mensual
            .coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_fact_agro)
        )

        print("PROGRESO: 60% - Verificando HDFS", flush=True)

        fact_agro_guardada = spark.read.parquet(
            ruta_fact_agro
        )

        registros_guardados = fact_agro_guardada.count()
        columnas_guardadas = len(fact_agro_guardada.columns)

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print("\nPROGRESO: 80% - Verificando integridad", flush=True)

        claves_nulas = fact_agro_guardada.filter(
            F.col("tiempo_id").isNull()
            | F.col("cultivo_id").isNull()
        ).count()

        claves_duplicadas = (
            fact_agro_guardada
            .groupBy(
                "tiempo_id",
                "cultivo_id",
                "proceso",
                "presentacion"
            )
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        cantidades_nulas = fact_agro_guardada.filter(
            F.col("cantidad_t").isNull()
        ).count()

        cantidades_cero = fact_agro_guardada.filter(
            F.col("cantidad_t") == 0
        ).count()

        print("Claves foraneas nulas:", claves_nulas)
        print("Claves duplicadas:", claves_duplicadas)
        print("Cantidades nulas:", cantidades_nulas)
        print("Cantidades iguales a cero:", cantidades_cero)

        print("\n===== REGISTROS POR PROCESO =====")

        (
            fact_agro_guardada
            .groupBy("proceso")
            .count()
            .orderBy("proceso")
            .show(truncate=False)
        )

        if (
            registros_guardados == 3134
            and columnas_guardadas == 5
            and claves_nulas == 0
            and claves_duplicadas == 0
            and cantidades_nulas == 0
            and cantidades_cero == 978
        ):
            print("\nALMACENAMIENTO FACT AGROINDUSTRIA: CORRECTO")
        else:
            print("\nATENCION: Revisar tabla almacenada")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 64 FINALIZADO =====")

===== PASO 64: GUARDAR FACT AGROINDUSTRIA =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Verificando HDFS

===== VERIFICACION HDFS =====
Registros guardados: 3134
Columnas guardadas: 5

PROGRESO: 80% - Verificando integridad
Claves foraneas nulas: 0
Claves duplicadas: 0
Cantidades nulas: 0
Cantidades iguales a cero: 978

===== REGISTROS POR PROCESO =====
+-----------+-----+
|proceso    |count|
+-----------+-----+
|INGRESO    |360  |
|PRODUCCION |1202 |
|UTILIZACION|360  |
|VENTA      |1212 |
+-----------+-----+


ALMACENAMIENTO FACT AGROINDUSTRIA: CORRECTO
PROGRESO: 100% - Verificacion finalizada
===== PASO 64 FINALIZADO =====


## Modelo dimensional Gold: tabla de hechos de costos agrícolas

Se construye la tabla de hechos de costos agrícolas
a partir de los registros analíticos disponibles en Gold.

La tabla se relaciona con las dimensiones de tiempo
y región de costos mediante claves foráneas.

Se conserva el tipo de costo, su valor reportado
y la unidad de medida original.

Los tipos de costos considerados son jornal agrícola,
alquiler de tractor y precio de yunta.

Debido a que la unidad está registrada como NO_CONFIRMADA,
no se realizan conversiones ni agregaciones entre conceptos.

Se verifica que las relaciones dimensionales no
generen registros duplicados ni pérdida de información.

In [50]:
from pyspark.sql import functions as F

print("===== PASO 65: FACT COSTOS MENSUALES =====")
print("PROGRESO: 0% - Leyendo tablas Gold", flush=True)

# 1. Leer tablas desde HDFS
costos_fact_origen = spark.read.parquet(
    f"{BASE_GOLD}/costos_mensuales"
)

tiempo_costos = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
)

region_costos = spark.read.parquet(
    f"{BASE_GOLD}/dim_region_costos"
)

total_origen_costos = costos_fact_origen.count()

print("Registros originales:", total_origen_costos)

print("\nPROGRESO: 25% - Relacionando dimensiones", flush=True)

# 2. Asociar cada registro con tiempo y region
costos_relacionados = (
    costos_fact_origen.alias("c")
    .join(
        tiempo_costos.select(
            "periodo", "tiempo_id"
        ).alias("t"),
        F.col("c.periodo") == F.col("t.periodo"),
        "left"
    )
    .join(
        region_costos.select(
            "region", "region_id"
        ).alias("r"),
        F.col("c.region") == F.col("r.region"),
        "left"
    )
)

print("\nPROGRESO: 50% - Construyendo tabla de hechos", flush=True)

fact_costos_mensuales = costos_relacionados.select(
    F.col("t.tiempo_id").alias("tiempo_id"),
    F.col("r.region_id").alias("region_id"),
    F.col("c.tipo_costo").alias("tipo_costo"),
    F.col("c.valor_costo").alias("valor_costo"),
    F.col("c.unidad").alias("unidad")
)

print("\n===== ESQUEMA FACT COSTOS =====")
fact_costos_mensuales.printSchema()

print("\nPROGRESO: 70% - Verificando integridad", flush=True)

total_fact_costos = fact_costos_mensuales.count()

claves_nulas_costos = fact_costos_mensuales.filter(
    F.col("tiempo_id").isNull()
    | F.col("region_id").isNull()
).count()

duplicados_fact_costos = (
    fact_costos_mensuales
    .groupBy("tiempo_id", "region_id", "tipo_costo")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\n===== VALIDACION =====")
print("Registros originales:", total_origen_costos)
print("Registros tabla de hechos:", total_fact_costos)
print("Claves foraneas nulas:", claves_nulas_costos)
print("Claves duplicadas:", duplicados_fact_costos)

print("\nPROGRESO: 85% - Revisando resultados", flush=True)

print("\n===== REGISTROS POR TIPO DE COSTO =====")

fact_costos_mensuales.groupBy(
    "tipo_costo", "unidad"
).count().orderBy(
    "tipo_costo"
).show(truncate=False)

print("\n===== MUESTRA FACT =====")

fact_costos_mensuales.orderBy(
    "tiempo_id",
    "region_id",
    "tipo_costo"
).show(12, truncate=False)

print("\n===== RESULTADO FINAL =====")

if (
    total_origen_costos == 6292
    and total_fact_costos == total_origen_costos
    and claves_nulas_costos == 0
    and duplicados_fact_costos == 0
):
    fact_costos_validada = True

    print(
        "TABLA DE HECHOS COSTOS: "
        "CONSTRUIDA CORRECTAMENTE"
    )
else:
    fact_costos_validada = False

    print(
        "ATENCION: REVISAR RELACIONES "
        "DIMENSIONALES DE COSTOS"
    )

print("\nPROGRESO: 100% - Construccion finalizada")
print("===== PASO 65 FINALIZADO =====")

===== PASO 65: FACT COSTOS MENSUALES =====
PROGRESO: 0% - Leyendo tablas Gold
Registros originales: 6292

PROGRESO: 25% - Relacionando dimensiones

PROGRESO: 50% - Construyendo tabla de hechos

===== ESQUEMA FACT COSTOS =====
root
 |-- tiempo_id: integer (nullable = true)
 |-- region_id: integer (nullable = true)
 |-- tipo_costo: string (nullable = true)
 |-- valor_costo: double (nullable = true)
 |-- unidad: string (nullable = true)


PROGRESO: 70% - Verificando integridad

===== VALIDACION =====
Registros originales: 6292
Registros tabla de hechos: 6292
Claves foraneas nulas: 0
Claves duplicadas: 0

PROGRESO: 85% - Revisando resultados

===== REGISTROS POR TIPO DE COSTO =====
+----------------+-------------+-----+
|tipo_costo      |unidad       |count|
+----------------+-------------+-----+
|ALQUILER_TRACTOR|NO_CONFIRMADA|2275 |
|PRECIO_YUNTA    |NO_CONFIRMADA|1890 |
|VALOR_JORNAL    |NO_CONFIRMADA|2127 |
+----------------+-------------+-----+


===== MUESTRA FACT =====
+---------+--

## Validación de medidas: hechos de costos agrícolas

Se verifica que la tabla de hechos de costos agrícolas
conserve los 6 292 registros de la tabla Gold original.

La validación compara individualmente el valor reportado
y la unidad de medida de cada costo, considerando el mes,
la región y el tipo de costo agrícola.

Se comprueba que la incorporación de claves dimensionales
no haya modificado los valores originales.

La unidad NO_CONFIRMADA se conserva, evitando asignar
unidades monetarias o físicas sin evidencia documental.

Los conceptos de jornal, tractor y yunta se mantienen
independientes para evitar agregaciones incorrectas.

In [51]:
from pyspark.sql import functions as F

print("===== PASO 66: VALIDACION MEDIDAS COSTOS =====")
print("PROGRESO: 0% - Preparando validacion", flush=True)

# 1. Preparar los datos originales con claves dimensionales.
origen_comparacion_costos = (
    costos_fact_origen.alias("c")
    .join(
        region_costos.select(
            "region_id", "region"
        ).alias("r"),
        F.col("c.region") == F.col("r.region"),
        "left"
    )
    .select(
        (
            F.year(F.col("c.periodo")) * 100
            + F.month(F.col("c.periodo"))
        ).alias("tiempo_id"),
        F.col("r.region_id").alias("region_id"),
        F.col("c.tipo_costo").alias("tipo_costo"),
        F.col("c.valor_costo").alias("valor_costo"),
        F.col("c.unidad").alias("unidad")
    )
)

print("\nPROGRESO: 25% - Comparando registros", flush=True)

registros_origen_costos = origen_comparacion_costos.count()
registros_fact_costos = fact_costos_mensuales.count()

print("Registros origen:", registros_origen_costos)
print("Registros fact:", registros_fact_costos)

# 2. Comparar cada registro, incluyendo NULL.
print("\nPROGRESO: 50% - Comparando valores", flush=True)

columnas_costos = [
    "tiempo_id",
    "region_id",
    "tipo_costo",
    "valor_costo",
    "unidad"
]

diferencias_origen_costos = (
    origen_comparacion_costos
    .select(*columnas_costos)
    .exceptAll(
        fact_costos_mensuales.select(*columnas_costos)
    )
    .count()
)

diferencias_fact_costos = (
    fact_costos_mensuales
    .select(*columnas_costos)
    .exceptAll(
        origen_comparacion_costos.select(*columnas_costos)
    )
    .count()
)

total_diferencias_costos = (
    diferencias_origen_costos + diferencias_fact_costos
)

print("Filas diferentes:", total_diferencias_costos)

# 3. Comprobar valores nulos y unidades.
print("\nPROGRESO: 75% - Verificando medidas y unidades", flush=True)

nulos_origen_costos = origen_comparacion_costos.filter(
    F.col("valor_costo").isNull()
).count()

nulos_fact_costos = fact_costos_mensuales.filter(
    F.col("valor_costo").isNull()
).count()

unidades_diferentes = fact_costos_mensuales.filter(
    F.col("unidad").isNull()
    | (F.col("unidad") != "NO_CONFIRMADA")
).count()

print("Valores nulos origen:", nulos_origen_costos)
print("Valores nulos fact:", nulos_fact_costos)
print("Unidades distintas de NO_CONFIRMADA:", unidades_diferentes)

print("\n===== REGISTROS POR TIPO DE COSTO =====")

fact_costos_mensuales.groupBy(
    "tipo_costo", "unidad"
).count().orderBy(
    "tipo_costo"
).show(truncate=False)

# 4. Resultado final.
print("\n===== RESULTADO FINAL =====")

if (
    registros_origen_costos == registros_fact_costos == 6292
    and total_diferencias_costos == 0
    and nulos_origen_costos == nulos_fact_costos
    and unidades_diferentes == 0
    and globals().get("fact_costos_validada", False)
):
    fact_costos_medidas_validada = True
    print("VALIDACION MEDIDAS COSTOS: CORRECTA")
else:
    fact_costos_medidas_validada = False
    print("ATENCION: REVISAR MEDIDAS DE COSTOS")

print("\nPROGRESO: 100% - Validacion finalizada")
print("===== PASO 66 FINALIZADO =====")

===== PASO 66: VALIDACION MEDIDAS COSTOS =====
PROGRESO: 0% - Preparando validacion

PROGRESO: 25% - Comparando registros
Registros origen: 6292
Registros fact: 6292

PROGRESO: 50% - Comparando valores
Filas diferentes: 0

PROGRESO: 75% - Verificando medidas y unidades
Valores nulos origen: 0
Valores nulos fact: 0
Unidades distintas de NO_CONFIRMADA: 0

===== REGISTROS POR TIPO DE COSTO =====
+----------------+-------------+-----+
|tipo_costo      |unidad       |count|
+----------------+-------------+-----+
|ALQUILER_TRACTOR|NO_CONFIRMADA|2275 |
|PRECIO_YUNTA    |NO_CONFIRMADA|1890 |
|VALOR_JORNAL    |NO_CONFIRMADA|2127 |
+----------------+-------------+-----+


===== RESULTADO FINAL =====
VALIDACION MEDIDAS COSTOS: CORRECTA

PROGRESO: 100% - Validacion finalizada
===== PASO 66 FINALIZADO =====


## Almacenamiento Gold: tabla de hechos de costos agrícolas

Se almacena en Hadoop HDFS la tabla de hechos de costos
agrícolas del modelo dimensional agrocomercial Danper,
utilizando el formato Parquet.

La tabla contiene 6 292 registros y se relaciona con
las dimensiones de tiempo y región de costos.

Se conservan los conceptos de alquiler de tractor,
precio de yunta y valor de jornal agrícola.

La validación previa confirmó que los valores y unidades
coinciden con la tabla Gold original, sin diferencias.

Las unidades de medida permanecen como NO_CONFIRMADA.
Por este motivo, no se realizan conversiones ni
agregaciones entre tipos de costos diferentes.

In [52]:
from pyspark.sql import functions as F

print("===== PASO 67: GUARDAR FACT COSTOS MENSUALES =====")

if not (
    globals().get("fact_costos_validada", False)
    and globals().get("fact_costos_medidas_validada", False)
):
    print("ERROR: Primero debes validar los Pasos 65 y 66.")

else:
    ruta_fact_costos = f"{BASE_GOLD}/fact_costos_mensuales"

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_fact_costos
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La tabla ya existe en HDFS.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_fact_costos)

    else:
        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            fact_costos_mensuales
            .coalesce(2)
            .write
            .mode("errorifexists")
            .parquet(ruta_fact_costos)
        )

        print("PROGRESO: 60% - Leyendo desde HDFS", flush=True)

        fact_costos_guardada = spark.read.parquet(
            ruta_fact_costos
        )

        registros_guardados = fact_costos_guardada.count()
        columnas_guardadas = len(fact_costos_guardada.columns)

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print(
            "\nPROGRESO: 80% - Verificando integridad",
            flush=True
        )

        claves_nulas = fact_costos_guardada.filter(
            F.col("tiempo_id").isNull()
            | F.col("region_id").isNull()
        ).count()

        claves_duplicadas = (
            fact_costos_guardada
            .groupBy(
                "tiempo_id",
                "region_id",
                "tipo_costo"
            )
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        valores_nulos = fact_costos_guardada.filter(
            F.col("valor_costo").isNull()
        ).count()

        unidades_incorrectas = fact_costos_guardada.filter(
            F.col("unidad").isNull()
            | (F.col("unidad") != "NO_CONFIRMADA")
        ).count()

        print("Claves foraneas nulas:", claves_nulas)
        print("Claves duplicadas:", claves_duplicadas)
        print("Valores de costo nulos:", valores_nulos)
        print("Unidades incorrectas:", unidades_incorrectas)

        print("\n===== REGISTROS POR TIPO DE COSTO =====")

        (
            fact_costos_guardada
            .groupBy("tipo_costo", "unidad")
            .count()
            .orderBy("tipo_costo")
            .show(truncate=False)
        )

        if (
            registros_guardados == 6292
            and columnas_guardadas == 5
            and claves_nulas == 0
            and claves_duplicadas == 0
            and valores_nulos == 0
            and unidades_incorrectas == 0
        ):
            print("\nALMACENAMIENTO FACT COSTOS: CORRECTO")
        else:
            print("\nATENCION: Revisar tabla almacenada")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 67 FINALIZADO =====")

===== PASO 67: GUARDAR FACT COSTOS MENSUALES =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Leyendo desde HDFS



===== VERIFICACION HDFS =====
Registros guardados: 6292
Columnas guardadas: 5

PROGRESO: 80% - Verificando integridad
Claves foraneas nulas: 0
Claves duplicadas: 0
Valores de costo nulos: 0
Unidades incorrectas: 0

===== REGISTROS POR TIPO DE COSTO =====
+----------------+-------------+-----+
|tipo_costo      |unidad       |count|
+----------------+-------------+-----+
|ALQUILER_TRACTOR|NO_CONFIRMADA|2275 |
|PRECIO_YUNTA    |NO_CONFIRMADA|1890 |
|VALOR_JORNAL    |NO_CONFIRMADA|2127 |
+----------------+-------------+-----+


ALMACENAMIENTO FACT COSTOS: CORRECTO
PROGRESO: 100% - Verificacion finalizada
===== PASO 67 FINALIZADO =====


## Modelo dimensional Gold: hechos de abastecimiento mayorista

Se construye la tabla de hechos de abastecimiento mayorista
a partir de los datos analíticos disponibles en Gold.

La información corresponde a espárrago y alcachofa del
Gran Mercado Mayorista de Lima.

La tabla se relaciona con las dimensiones de tiempo,
cultivo y mercado mediante claves foráneas.

Se conservan la fecha exacta del reporte, las unidades
comerciales, los indicadores de ingreso y los precios
reportados.

Los valores nulos permanecen sin modificaciones y se
identifica la disponibilidad de información de ingreso.

Las cantidades representan reportes puntuales y no deben
interpretarse como acumulados mensuales.

In [53]:
from pyspark.sql import functions as F

print("===== PASO 68: FACT ABASTECIMIENTO MAYORISTA =====")
print("PROGRESO: 0% - Leyendo tablas Gold", flush=True)

# 1. Leer los datos desde HDFS
abastecimiento_fact_origen = spark.read.parquet(
    f"{BASE_GOLD}/abastecimiento_mayorista"
)

tiempo_abast = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
)

cultivo_abast = spark.read.parquet(
    f"{BASE_GOLD}/dim_cultivo"
)

mercado_abast = spark.read.parquet(
    f"{BASE_GOLD}/dim_mercado"
)

total_origen_abast = abastecimiento_fact_origen.count()

print("Registros originales:", total_origen_abast)

print("\nPROGRESO: 25% - Relacionando dimensiones", flush=True)

# 2. Relacionar tiempo, cultivo y mercado.
# El primer dia del mes se usa solo para localizar dim_tiempo.
abastecimiento_relacionado = (
    abastecimiento_fact_origen.alias("a")
    .join(
        tiempo_abast.select(
            "periodo", "tiempo_id"
        ).alias("t"),
        F.trunc(F.col("a.fecha_reporte"), "month")
        == F.col("t.periodo"),
        "left"
    )
    .join(
        cultivo_abast.select(
            "cultivo", "cultivo_id"
        ).alias("c"),
        F.col("a.producto") == F.col("c.cultivo"),
        "left"
    )
    .join(
        mercado_abast.select(
            "mercado", "mercado_id"
        ).alias("m"),
        F.col("a.mercado") == F.col("m.mercado"),
        "left"
    )
)

print("\nPROGRESO: 50% - Construyendo tabla de hechos", flush=True)

# 3. Conservar la fecha original y todas las medidas relevantes
fact_abastecimiento_mayorista = (
    abastecimiento_relacionado
    .select(
        F.col("t.tiempo_id").alias("tiempo_id"),
        F.col("c.cultivo_id").alias("cultivo_id"),
        F.col("m.mercado_id").alias("mercado_id"),
        F.col("a.fecha_reporte").alias("fecha_reporte"),
        F.col("a.unidad_medida").alias("unidad_medida"),
        F.col("a.equivalencia_kg").alias("equivalencia_kg"),
        F.col("a.ingreso_ayer_t").alias("ingreso_ayer_t"),
        F.col("a.ingreso_hoy_t").alias("ingreso_hoy_t"),
        F.col("a.promedio_ingreso_7dias_t").alias(
            "promedio_ingreso_7dias_t"
        ),
        F.col("a.promedio_ingreso_4semanas_t").alias(
            "promedio_ingreso_4semanas_t"
        ),
        F.col("a.precio_ayer_soles").alias(
            "precio_ayer_soles"
        ),
        F.col("a.precio_hoy_soles").alias(
            "precio_hoy_soles"
        ),
        F.col("a.precio_7dias_soles").alias(
            "precio_7dias_soles"
        ),
        F.col("a.tendencia").alias("tendencia"),
        F.col("a.tiene_dato_ingreso").alias(
            "tiene_dato_ingreso"
        )
    )
)

print("\n===== ESQUEMA FACT ABASTECIMIENTO =====")
fact_abastecimiento_mayorista.printSchema()

print("\nPROGRESO: 70% - Verificando integridad", flush=True)

total_fact_abast = fact_abastecimiento_mayorista.count()

claves_nulas_abast = fact_abastecimiento_mayorista.filter(
    F.col("tiempo_id").isNull()
    | F.col("cultivo_id").isNull()
    | F.col("mercado_id").isNull()
).count()

# La fecha exacta debe formar parte de la clave natural.
duplicados_fact_abast = (
    fact_abastecimiento_mayorista
    .groupBy(
        "fecha_reporte",
        "cultivo_id",
        "mercado_id"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

ingresos_nulos_abast = fact_abastecimiento_mayorista.filter(
    F.col("ingreso_hoy_t").isNull()
).count()

errores_indicador_abast = fact_abastecimiento_mayorista.filter(
    ~F.col("tiene_dato_ingreso").eqNullSafe(
        F.col("ingreso_hoy_t").isNotNull()
    )
).count()

print("\n===== VALIDACION =====")
print("Registros originales:", total_origen_abast)
print("Registros tabla de hechos:", total_fact_abast)
print("Claves foraneas nulas:", claves_nulas_abast)
print("Claves duplicadas:", duplicados_fact_abast)
print("Ingresos nulos conservados:", ingresos_nulos_abast)
print("Errores en indicador de ingreso:", errores_indicador_abast)

print("\nPROGRESO: 85% - Revisando resultados", flush=True)

print("\n===== DISPONIBILIDAD POR CULTIVO =====")

(
    fact_abastecimiento_mayorista.alias("f")
    .join(
        cultivo_abast.select(
            "cultivo_id", "cultivo"
        ).alias("c"),
        "cultivo_id",
        "left"
    )
    .groupBy("cultivo", "tiene_dato_ingreso")
    .count()
    .orderBy("cultivo", "tiene_dato_ingreso")
    .show(truncate=False)
)

print("\n===== MUESTRA FACT =====")

fact_abastecimiento_mayorista.orderBy(
    "fecha_reporte",
    "cultivo_id"
).show(10, truncate=False)

print("\n===== RESULTADO FINAL =====")

if (
    total_origen_abast == 66
    and total_fact_abast == total_origen_abast
    and claves_nulas_abast == 0
    and duplicados_fact_abast == 0
    and ingresos_nulos_abast == 28
    and errores_indicador_abast == 0
):
    fact_abastecimiento_validada = True
    print(
        "TABLA DE HECHOS ABASTECIMIENTO: "
        "CONSTRUIDA CORRECTAMENTE"
    )
else:
    fact_abastecimiento_validada = False
    print("ATENCION: REVISAR RELACIONES DE ABASTECIMIENTO")

print("\nPROGRESO: 100% - Construccion finalizada")
print("===== PASO 68 FINALIZADO =====")

===== PASO 68: FACT ABASTECIMIENTO MAYORISTA =====
PROGRESO: 0% - Leyendo tablas Gold
Registros originales: 66

PROGRESO: 25% - Relacionando dimensiones

PROGRESO: 50% - Construyendo tabla de hechos

===== ESQUEMA FACT ABASTECIMIENTO =====
root
 |-- tiempo_id: integer (nullable = true)
 |-- cultivo_id: integer (nullable = true)
 |-- mercado_id: integer (nullable = true)
 |-- fecha_reporte: date (nullable = true)
 |-- unidad_medida: string (nullable = true)
 |-- equivalencia_kg: double (nullable = true)
 |-- ingreso_ayer_t: double (nullable = true)
 |-- ingreso_hoy_t: double (nullable = true)
 |-- promedio_ingreso_7dias_t: double (nullable = true)
 |-- promedio_ingreso_4semanas_t: double (nullable = true)
 |-- precio_ayer_soles: double (nullable = true)
 |-- precio_hoy_soles: double (nullable = true)
 |-- precio_7dias_soles: double (nullable = true)
 |-- tendencia: string (nullable = true)
 |-- tiene_dato_ingreso: boolean (nullable = true)


PROGRESO: 70% - Verificando integridad

=====

## Validación de medidas: hechos de abastecimiento mayorista

Se verifica que la tabla de hechos de abastecimiento
mayorista conserve los 66 registros de Gold.

La comparación se realiza considerando la fecha exacta
del reporte, el cultivo y el mercado correspondiente.

Se revisan los valores de ingreso, precios, promedios,
unidades comerciales, equivalencias en kilogramos
y tendencias reportadas.

Los valores nulos se mantienen sin reemplazarlos por cero.

También se verifica que el indicador tiene_dato_ingreso
identifique correctamente los registros que contienen
información de ingreso.

Esta validación garantiza que el modelo dimensional
no modifique los valores originales de abastecimiento.

In [54]:
from pyspark.sql import functions as F

print("===== PASO 69: VALIDACION MEDIDAS ABASTECIMIENTO =====")
print("PROGRESO: 0% - Preparando validacion", flush=True)

# 1. Preparar el origen con las claves dimensionales
origen_comparacion_abast = (
    abastecimiento_fact_origen.alias("a")
    .join(
        cultivo_abast.select(
            "cultivo_id", "cultivo"
        ).alias("c"),
        F.col("a.producto") == F.col("c.cultivo"),
        "left"
    )
    .join(
        mercado_abast.select(
            "mercado_id", "mercado"
        ).alias("m"),
        F.col("a.mercado") == F.col("m.mercado"),
        "left"
    )
    .select(
        (
            F.year(F.col("a.fecha_reporte")) * 100
            + F.month(F.col("a.fecha_reporte"))
        ).alias("tiempo_id"),
        F.col("c.cultivo_id").alias("cultivo_id"),
        F.col("m.mercado_id").alias("mercado_id"),
        F.col("a.fecha_reporte").alias("fecha_reporte"),
        F.col("a.unidad_medida").alias("unidad_medida"),
        F.col("a.equivalencia_kg").alias("equivalencia_kg"),
        F.col("a.ingreso_ayer_t").alias("ingreso_ayer_t"),
        F.col("a.ingreso_hoy_t").alias("ingreso_hoy_t"),
        F.col("a.promedio_ingreso_7dias_t").alias(
            "promedio_ingreso_7dias_t"
        ),
        F.col("a.promedio_ingreso_4semanas_t").alias(
            "promedio_ingreso_4semanas_t"
        ),
        F.col("a.precio_ayer_soles").alias(
            "precio_ayer_soles"
        ),
        F.col("a.precio_hoy_soles").alias(
            "precio_hoy_soles"
        ),
        F.col("a.precio_7dias_soles").alias(
            "precio_7dias_soles"
        ),
        F.col("a.tendencia").alias("tendencia"),
        F.col("a.tiene_dato_ingreso").alias(
            "tiene_dato_ingreso"
        )
    )
)

print("\nPROGRESO: 25% - Comparando registros", flush=True)

registros_origen_abast = origen_comparacion_abast.count()
registros_fact_abast = fact_abastecimiento_mayorista.count()

print("Registros origen:", registros_origen_abast)
print("Registros fact:", registros_fact_abast)

# 2. Comparar todos los campos, incluyendo NULL
print("\nPROGRESO: 50% - Comparando medidas", flush=True)

columnas_abast = [
    "tiempo_id",
    "cultivo_id",
    "mercado_id",
    "fecha_reporte",
    "unidad_medida",
    "equivalencia_kg",
    "ingreso_ayer_t",
    "ingreso_hoy_t",
    "promedio_ingreso_7dias_t",
    "promedio_ingreso_4semanas_t",
    "precio_ayer_soles",
    "precio_hoy_soles",
    "precio_7dias_soles",
    "tendencia",
    "tiene_dato_ingreso"
]

diferencias_origen_abast = (
    origen_comparacion_abast
    .select(*columnas_abast)
    .exceptAll(
        fact_abastecimiento_mayorista.select(*columnas_abast)
    )
    .count()
)

diferencias_fact_abast = (
    fact_abastecimiento_mayorista
    .select(*columnas_abast)
    .exceptAll(
        origen_comparacion_abast.select(*columnas_abast)
    )
    .count()
)

total_diferencias_abast = (
    diferencias_origen_abast + diferencias_fact_abast
)

print("Filas diferentes:", total_diferencias_abast)

# 3. Verificar NULL y disponibilidad de ingresos
print("\nPROGRESO: 75% - Verificando ingresos faltantes", flush=True)

nulos_origen_abast = origen_comparacion_abast.filter(
    F.col("ingreso_hoy_t").isNull()
).count()

nulos_fact_abast = fact_abastecimiento_mayorista.filter(
    F.col("ingreso_hoy_t").isNull()
).count()

errores_indicador_abast_69 = (
    fact_abastecimiento_mayorista
    .filter(
        ~F.col("tiene_dato_ingreso").eqNullSafe(
            F.col("ingreso_hoy_t").isNotNull()
        )
    )
    .count()
)

print("Ingresos nulos origen:", nulos_origen_abast)
print("Ingresos nulos fact:", nulos_fact_abast)
print("Errores del indicador:", errores_indicador_abast_69)

print("\n===== RESULTADO FINAL =====")

if (
    registros_origen_abast == registros_fact_abast == 66
    and total_diferencias_abast == 0
    and nulos_origen_abast == nulos_fact_abast == 28
    and errores_indicador_abast_69 == 0
    and globals().get("fact_abastecimiento_validada", False)
):
    fact_abastecimiento_medidas_validada = True
    print("VALIDACION MEDIDAS ABASTECIMIENTO: CORRECTA")
else:
    fact_abastecimiento_medidas_validada = False
    print("ATENCION: REVISAR MEDIDAS DE ABASTECIMIENTO")

print("\nPROGRESO: 100% - Validacion finalizada")
print("===== PASO 69 FINALIZADO =====")

===== PASO 69: VALIDACION MEDIDAS ABASTECIMIENTO =====
PROGRESO: 0% - Preparando validacion

PROGRESO: 25% - Comparando registros
Registros origen: 66
Registros fact: 66

PROGRESO: 50% - Comparando medidas
Filas diferentes: 0

PROGRESO: 75% - Verificando ingresos faltantes
Ingresos nulos origen: 28
Ingresos nulos fact: 28
Errores del indicador: 0

===== RESULTADO FINAL =====
VALIDACION MEDIDAS ABASTECIMIENTO: CORRECTA

PROGRESO: 100% - Validacion finalizada
===== PASO 69 FINALIZADO =====


## Almacenamiento Gold: tabla de hechos de abastecimiento mayorista

Se almacena en Hadoop HDFS la tabla de hechos de abastecimiento
mayorista del modelo dimensional agrocomercial Danper,
utilizando el formato Parquet.

La tabla contiene 66 registros de espárrago y alcachofa,
relacionados con las dimensiones de tiempo, cultivo y mercado.

Se conservan las fechas exactas de los reportes, las unidades
comerciales, los precios y los indicadores de ingreso.

La validación previa confirmó que no existen diferencias
respecto a los valores originales de Gold.

Los 28 registros sin ingreso reportado permanecen como nulos.

Las cantidades corresponden a reportes puntuales del mercado
y no representan acumulados mensuales.

In [55]:
from pyspark.sql import functions as F

print("===== PASO 70: GUARDAR FACT ABASTECIMIENTO =====")

if not (
    globals().get("fact_abastecimiento_validada", False)
    and globals().get("fact_abastecimiento_medidas_validada", False)
):
    print("ERROR: Primero debes validar los Pasos 68 y 69.")

else:
    ruta_fact_abast = (
        f"{BASE_GOLD}/fact_abastecimiento_mayorista"
    )

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_fact_abast
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La tabla ya existe en HDFS.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_fact_abast)

    else:
        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            fact_abastecimiento_mayorista
            .coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_fact_abast)
        )

        print("PROGRESO: 60% - Leyendo desde HDFS", flush=True)

        fact_abast_guardada = spark.read.parquet(
            ruta_fact_abast
        )

        registros_guardados = fact_abast_guardada.count()
        columnas_guardadas = len(fact_abast_guardada.columns)

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print("\nPROGRESO: 80% - Verificando integridad", flush=True)

        claves_nulas = fact_abast_guardada.filter(
            F.col("tiempo_id").isNull()
            | F.col("cultivo_id").isNull()
            | F.col("mercado_id").isNull()
        ).count()

        claves_duplicadas = (
            fact_abast_guardada
            .groupBy(
                "fecha_reporte",
                "cultivo_id",
                "mercado_id"
            )
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        ingresos_nulos = fact_abast_guardada.filter(
            F.col("ingreso_hoy_t").isNull()
        ).count()

        errores_indicador = fact_abast_guardada.filter(
            ~F.col("tiene_dato_ingreso").eqNullSafe(
                F.col("ingreso_hoy_t").isNotNull()
            )
        ).count()

        print("Claves foraneas nulas:", claves_nulas)
        print("Claves duplicadas:", claves_duplicadas)
        print("Ingresos nulos conservados:", ingresos_nulos)
        print("Errores del indicador:", errores_indicador)

        print("\n===== DISPONIBILIDAD POR CULTIVO =====")

        (
            fact_abast_guardada.alias("f")
            .join(
                cultivo_abast.select(
                    "cultivo_id", "cultivo"
                ),
                "cultivo_id",
                "left"
            )
            .groupBy("cultivo", "tiene_dato_ingreso")
            .count()
            .orderBy("cultivo", "tiene_dato_ingreso")
            .show(truncate=False)
        )

        print("\n===== RESULTADO FINAL =====")

        if (
            registros_guardados == 66
            and columnas_guardadas == 15
            and claves_nulas == 0
            and claves_duplicadas == 0
            and ingresos_nulos == 28
            and errores_indicador == 0
        ):
            print("ALMACENAMIENTO FACT ABASTECIMIENTO: CORRECTO")
        else:
            print("ATENCION: Revisar tabla almacenada")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 70 FINALIZADO =====")

===== PASO 70: GUARDAR FACT ABASTECIMIENTO =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Leyendo desde HDFS

===== VERIFICACION HDFS =====
Registros guardados: 66
Columnas guardadas: 15

PROGRESO: 80% - Verificando integridad
Claves foraneas nulas: 0
Claves duplicadas: 0
Ingresos nulos conservados: 28
Errores del indicador: 0

===== DISPONIBILIDAD POR CULTIVO =====
+---------+------------------+-----+
|cultivo  |tiene_dato_ingreso|count|
+---------+------------------+-----+
|ALCACHOFA|false             |18   |
|ALCACHOFA|true              |15   |
|ESPARRAGO|false             |10   |
|ESPARRAGO|true              |23   |
+---------+------------------+-----+


===== RESULTADO FINAL =====
ALMACENAMIENTO FACT ABASTECIMIENTO: CORRECTO
PROGRESO: 100% - Verificacion finalizada
===== PASO 70 FINALIZADO =====


## Modelo dimensional Gold: hechos de precios del Mercado de Productores

Se construye la tabla de hechos de precios del Mercado
de Productores de Santa Anita a partir de los registros
analíticos almacenados en Gold.

La tabla contiene información de los años 2024 y 2025
y se relaciona con las dimensiones de tiempo y mercado.

Se conservan la fecha exacta del reporte, la categoría,
el producto, la unidad comercial, los precios en soles
y la variación porcentual reportada.

Los precios de unidades comerciales diferentes no se
agregan ni comparan directamente.

Esta tabla proporciona información complementaria
del mercado interno y permanece separada de los
indicadores específicos de cultivos de Danper.

In [56]:
from pyspark.sql import functions as F

print("===== PASO 71: FACT PRECIOS MERCADO PRODUCTORES =====")
print("PROGRESO: 0% - Leyendo tablas Gold", flush=True)

# 1. Leer los datos y dimensiones desde HDFS.
precios_fact_origen = spark.read.parquet(
    f"{BASE_GOLD}/precios_mercado_productores"
)

tiempo_precios = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
)

mercado_precios = spark.read.parquet(
    f"{BASE_GOLD}/dim_mercado"
)

total_origen_precios = precios_fact_origen.count()

print("Registros originales:", total_origen_precios)

print("\nPROGRESO: 25% - Relacionando dimensiones", flush=True)

# 2. Relacionar el mes y el mercado.
# Mantener la fecha exacta del reporte.
precios_relacionados = (
    precios_fact_origen.alias("p")
    .join(
        tiempo_precios.select(
            "periodo", "tiempo_id"
        ).alias("t"),
        F.trunc(F.col("p.fecha_reporte"), "month")
        == F.col("t.periodo"),
        "left"
    )
    .join(
        mercado_precios.select(
            "mercado", "mercado_id"
        ).alias("m"),
        F.col("p.mercado") == F.col("m.mercado"),
        "left"
    )
)

print("\nPROGRESO: 50% - Construyendo tabla de hechos", flush=True)

# 3. Conservar las medidas y atributos originales.
fact_precios_mercado_productores = (
    precios_relacionados
    .select(
        F.col("t.tiempo_id").alias("tiempo_id"),
        F.col("m.mercado_id").alias("mercado_id"),
        F.col("p.fecha_reporte").alias("fecha_reporte"),
        F.col("p.categoria").alias("categoria"),
        F.col("p.producto").alias("producto"),
        F.col("p.unidad_medida").alias("unidad_medida"),
        F.col("p.precio_ayer_soles").alias(
            "precio_ayer_soles"
        ),
        F.col("p.precio_hoy_soles").alias(
            "precio_hoy_soles"
        ),
        F.col("p.precio_7dias_soles").alias(
            "precio_7dias_soles"
        ),
        F.col("p.variacion_porcentaje").alias(
            "variacion_porcentaje"
        )
    )
)

print("\n===== ESQUEMA FACT PRECIOS =====")
fact_precios_mercado_productores.printSchema()

print("\nPROGRESO: 70% - Verificando integridad", flush=True)

total_fact_precios = fact_precios_mercado_productores.count()

claves_nulas_precios = (
    fact_precios_mercado_productores
    .filter(
        F.col("tiempo_id").isNull()
        | F.col("mercado_id").isNull()
    )
    .count()
)

# La unidad y la fecha exacta son parte del nivel de detalle.
claves_fact_precios = [
    "mercado_id",
    "fecha_reporte",
    "categoria",
    "producto",
    "unidad_medida"
]

duplicados_fact_precios = (
    fact_precios_mercado_productores
    .groupBy(*claves_fact_precios)
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\n===== VALIDACION =====")
print("Registros originales:", total_origen_precios)
print("Registros tabla de hechos:", total_fact_precios)
print("Claves foraneas nulas:", claves_nulas_precios)
print("Claves duplicadas:", duplicados_fact_precios)

print("\nPROGRESO: 85% - Revisando resultados", flush=True)

print("\n===== REGISTROS POR CATEGORIA =====")

(
    fact_precios_mercado_productores
    .groupBy("categoria")
    .count()
    .orderBy("categoria")
    .show(truncate=False)
)

print("\n===== UNIDADES POR CATEGORIA =====")

(
    fact_precios_mercado_productores
    .groupBy("categoria")
    .agg(
        F.countDistinct("unidad_medida").alias(
            "unidades_distintas"
        )
    )
    .orderBy("categoria")
    .show(truncate=False)
)

print("\n===== MUESTRA FACT =====")

fact_precios_mercado_productores.orderBy(
    "fecha_reporte",
    "categoria",
    "producto"
).show(10, truncate=False)

print("\n===== RESULTADO FINAL =====")

if (
    total_origen_precios == 756
    and total_fact_precios == total_origen_precios
    and claves_nulas_precios == 0
    and duplicados_fact_precios == 0
):
    fact_precios_validada = True

    print(
        "TABLA DE HECHOS PRECIOS MERCADO: "
        "CONSTRUIDA CORRECTAMENTE"
    )
else:
    fact_precios_validada = False

    print(
        "ATENCION: REVISAR RELACIONES "
        "DE PRECIOS DEL MERCADO"
    )

print("\nPROGRESO: 100% - Construccion finalizada")
print("===== PASO 71 FINALIZADO =====")

===== PASO 71: FACT PRECIOS MERCADO PRODUCTORES =====
PROGRESO: 0% - Leyendo tablas Gold
Registros originales: 756

PROGRESO: 25% - Relacionando dimensiones

PROGRESO: 50% - Construyendo tabla de hechos

===== ESQUEMA FACT PRECIOS =====
root
 |-- tiempo_id: integer (nullable = true)
 |-- mercado_id: integer (nullable = true)
 |-- fecha_reporte: date (nullable = true)
 |-- categoria: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- unidad_medida: string (nullable = true)
 |-- precio_ayer_soles: double (nullable = true)
 |-- precio_hoy_soles: double (nullable = true)
 |-- precio_7dias_soles: double (nullable = true)
 |-- variacion_porcentaje: double (nullable = true)


PROGRESO: 70% - Verificando integridad

===== VALIDACION =====
Registros originales: 756
Registros tabla de hechos: 756
Claves foraneas nulas: 0
Claves duplicadas: 0

PROGRESO: 85% - Revisando resultados

===== REGISTROS POR CATEGORIA =====
+-----------------+-----+
|categoria        |count|
+---------

## Validación de medidas: hechos de precios del Mercado de Productores

Se verifica que la tabla de hechos de precios del Mercado
de Productores conserve los 756 registros originales de Gold.

La comparación considera el mercado, la fecha del reporte,
la categoría, el producto y su unidad comercial.

Se comprueba que los precios reportados en soles y las
variaciones porcentuales sean idénticos a los valores
originales, incluyendo posibles valores nulos.

Se mantienen separadas las diferentes unidades comerciales
para evitar interpretaciones incorrectas.

Esta validación garantiza que el modelo dimensional no
modifique la información de precios del mercado interno.

In [57]:
from pyspark.sql import functions as F

print("===== PASO 72: VALIDACION MEDIDAS PRECIOS MERCADO =====")
print("PROGRESO: 0% - Preparando validacion", flush=True)

# 1. Preparar datos originales con claves dimensionales.
origen_comparacion_precios = (
    precios_fact_origen.alias("p")
    .join(
        mercado_precios.select(
            "mercado_id", "mercado"
        ).alias("m"),
        F.col("p.mercado") == F.col("m.mercado"),
        "left"
    )
    .select(
        (
            F.year(F.col("p.fecha_reporte")) * 100
            + F.month(F.col("p.fecha_reporte"))
        ).alias("tiempo_id"),
        F.col("m.mercado_id").alias("mercado_id"),
        F.col("p.fecha_reporte").alias("fecha_reporte"),
        F.col("p.categoria").alias("categoria"),
        F.col("p.producto").alias("producto"),
        F.col("p.unidad_medida").alias("unidad_medida"),
        F.col("p.precio_ayer_soles").alias("precio_ayer_soles"),
        F.col("p.precio_hoy_soles").alias("precio_hoy_soles"),
        F.col("p.precio_7dias_soles").alias("precio_7dias_soles"),
        F.col("p.variacion_porcentaje").alias(
            "variacion_porcentaje"
        )
    )
)

print("\nPROGRESO: 25% - Comparando registros", flush=True)

registros_origen_precios = origen_comparacion_precios.count()
registros_fact_precios_72 = (
    fact_precios_mercado_productores.count()
)

print("Registros origen:", registros_origen_precios)
print("Registros fact:", registros_fact_precios_72)

# 2. Comparar todas las columnas, incluyendo NULL.
print("\nPROGRESO: 50% - Comparando valores", flush=True)

columnas_precios = [
    "tiempo_id",
    "mercado_id",
    "fecha_reporte",
    "categoria",
    "producto",
    "unidad_medida",
    "precio_ayer_soles",
    "precio_hoy_soles",
    "precio_7dias_soles",
    "variacion_porcentaje"
]

diferencias_origen_precios = (
    origen_comparacion_precios
    .select(*columnas_precios)
    .exceptAll(
        fact_precios_mercado_productores.select(
            *columnas_precios
        )
    )
    .count()
)

diferencias_fact_precios = (
    fact_precios_mercado_productores
    .select(*columnas_precios)
    .exceptAll(
        origen_comparacion_precios.select(
            *columnas_precios
        )
    )
    .count()
)

total_diferencias_precios = (
    diferencias_origen_precios + diferencias_fact_precios
)

print("Filas diferentes:", total_diferencias_precios)

# 3. Comprobar valores nulos en cada indicador.
print("\nPROGRESO: 75% - Verificando valores faltantes", flush=True)

medidas_precios = [
    "precio_ayer_soles",
    "precio_hoy_soles",
    "precio_7dias_soles",
    "variacion_porcentaje"
]

nulos_iguales_precios = True

print("\n===== VALORES NULOS POR INDICADOR =====")

for medida in medidas_precios:

    nulos_origen = origen_comparacion_precios.filter(
        F.col(medida).isNull()
    ).count()

    nulos_fact = fact_precios_mercado_productores.filter(
        F.col(medida).isNull()
    ).count()

    print(
        f"{medida}: origen={nulos_origen}, fact={nulos_fact}"
    )

    if nulos_origen != nulos_fact:
        nulos_iguales_precios = False

# 4. Resultado final.
print("\n===== RESULTADO FINAL =====")

if (
    registros_origen_precios == registros_fact_precios_72 == 756
    and total_diferencias_precios == 0
    and nulos_iguales_precios
    and globals().get("fact_precios_validada", False)
):
    fact_precios_medidas_validada = True
    print("VALIDACION MEDIDAS PRECIOS MERCADO: CORRECTA")
else:
    fact_precios_medidas_validada = False
    print("ATENCION: REVISAR MEDIDAS DE PRECIOS")

print("\nPROGRESO: 100% - Validacion finalizada")
print("===== PASO 72 FINALIZADO =====")

===== PASO 72: VALIDACION MEDIDAS PRECIOS MERCADO =====
PROGRESO: 0% - Preparando validacion

PROGRESO: 25% - Comparando registros
Registros origen: 756
Registros fact: 756

PROGRESO: 50% - Comparando valores


Filas diferentes: 0

PROGRESO: 75% - Verificando valores faltantes

===== VALORES NULOS POR INDICADOR =====
precio_ayer_soles: origen=0, fact=0
precio_hoy_soles: origen=0, fact=0
precio_7dias_soles: origen=0, fact=0
variacion_porcentaje: origen=0, fact=0

===== RESULTADO FINAL =====
VALIDACION MEDIDAS PRECIOS MERCADO: CORRECTA

PROGRESO: 100% - Validacion finalizada
===== PASO 72 FINALIZADO =====


## Almacenamiento Gold: tabla de hechos de precios del Mercado de Productores

Se almacena en Hadoop HDFS la tabla de hechos de precios
del Mercado de Productores de Santa Anita, utilizando
el formato Parquet.

La tabla contiene 756 registros correspondientes a los
años 2024 y 2025 y se relaciona con las dimensiones
de tiempo y mercado.

Se conservan las fechas exactas, categorías, productos,
unidades comerciales, precios en soles y variaciones
porcentuales reportadas.

La validación previa confirmó que los 756 registros y
sus medidas coinciden con la tabla Gold original.

Esta fuente proporciona información complementaria
del mercado interno y no se integra directamente
con los cultivos específicos de Danper.

In [58]:
from pyspark.sql import functions as F

print("===== PASO 73: GUARDAR FACT PRECIOS MERCADO =====")

if not (
    globals().get("fact_precios_validada", False)
    and globals().get("fact_precios_medidas_validada", False)
):
    print("ERROR: Primero debes validar los Pasos 71 y 72.")

else:
    ruta_fact_precios = (
        f"{BASE_GOLD}/fact_precios_mercado_productores"
    )

    print("PROGRESO: 0% - Comprobando destino", flush=True)

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
        ruta_fact_precios
    )

    filesystem = ruta_java.getFileSystem(hadoop_conf)

    if filesystem.exists(ruta_java):
        print("\nATENCION: La tabla ya existe en HDFS.")
        print("No se sobrescribira ningun archivo.")
        print("Ruta:", ruta_fact_precios)

    else:
        print("PROGRESO: 25% - Guardando Parquet", flush=True)

        (
            fact_precios_mercado_productores
            .coalesce(1)
            .write
            .mode("errorifexists")
            .parquet(ruta_fact_precios)
        )

        print("PROGRESO: 60% - Leyendo desde HDFS", flush=True)

        fact_precios_guardada = spark.read.parquet(
            ruta_fact_precios
        )

        registros_guardados = fact_precios_guardada.count()
        columnas_guardadas = len(
            fact_precios_guardada.columns
        )

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", registros_guardados)
        print("Columnas guardadas:", columnas_guardadas)

        print(
            "\nPROGRESO: 80% - Verificando integridad",
            flush=True
        )

        claves_nulas = fact_precios_guardada.filter(
            F.col("tiempo_id").isNull()
            | F.col("mercado_id").isNull()
        ).count()

        claves_duplicadas = (
            fact_precios_guardada
            .groupBy(
                "mercado_id",
                "fecha_reporte",
                "categoria",
                "producto",
                "unidad_medida"
            )
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        print("Claves foraneas nulas:", claves_nulas)
        print("Claves duplicadas:", claves_duplicadas)

        print("\n===== REGISTROS POR CATEGORIA =====")

        (
            fact_precios_guardada
            .groupBy("categoria")
            .count()
            .orderBy("categoria")
            .show(truncate=False)
        )

        print("\n===== DISPONIBILIDAD DE MEDIDAS =====")

        medidas = [
            "precio_ayer_soles",
            "precio_hoy_soles",
            "precio_7dias_soles",
            "variacion_porcentaje"
        ]

        resumen = fact_precios_guardada.agg(
            *[
                F.count(F.col(medida)).alias(medida)
                for medida in medidas
            ]
        )

        resumen.show(truncate=False)

        medidas_completas = all(
            fact_precios_guardada.filter(
                F.col(medida).isNull()
            ).count() == 0
            for medida in medidas
        )

        print("\n===== RESULTADO FINAL =====")

        if (
            registros_guardados == 756
            and columnas_guardadas == 10
            and claves_nulas == 0
            and claves_duplicadas == 0
            and medidas_completas
        ):
            print("ALMACENAMIENTO FACT PRECIOS: CORRECTO")
        else:
            print("ATENCION: Revisar tabla almacenada")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 73 FINALIZADO =====")

===== PASO 73: GUARDAR FACT PRECIOS MERCADO =====
PROGRESO: 0% - Comprobando destino
PROGRESO: 25% - Guardando Parquet
PROGRESO: 60% - Leyendo desde HDFS

===== VERIFICACION HDFS =====
Registros guardados: 756
Columnas guardadas: 10

PROGRESO: 80% - Verificando integridad
Claves foraneas nulas: 0
Claves duplicadas: 0

===== REGISTROS POR CATEGORIA =====
+-----------------+-----+
|categoria        |count|
+-----------------+-----+
|ACEITE COMESTIBLE|72   |
|ARROZ            |119  |
|AZUCAR           |255  |
|FIDEOS           |47   |
|HARINA           |71   |
|LECHE            |96   |
|MENESTRAS        |96   |
+-----------------+-----+


===== DISPONIBILIDAD DE MEDIDAS =====
+-----------------+----------------+------------------+--------------------+
|precio_ayer_soles|precio_hoy_soles|precio_7dias_soles|variacion_porcentaje|
+-----------------+----------------+------------------+--------------------+
|756              |756             |756               |756                 |
+---------

## KPI Gold: producción agrícola

Se construyen indicadores agrocomerciales a partir de la
tabla de hechos de producción y sus dimensiones asociadas.

Los indicadores considerados son la producción agrícola
total reportada en toneladas y el precio promedio en
chacra expresado en soles por kilogramo.

Los KPI se presentan por cultivo y año, conservando
la cantidad de registros disponibles para interpretar
su cobertura.

Los valores nulos no se reemplazan por cero.

El indicador de rendimiento no se incluye debido a que
no presenta valores disponibles en la fuente.

Los resultados servirán como base para consultas
analíticas y visualizaciones del proyecto Danper.

In [59]:
from pyspark.sql import functions as F

print("===== PASO 74: KPI DE PRODUCCION AGRICOLA =====")
print("PROGRESO: 0% - Leyendo tablas dimensionales", flush=True)

# 1. Leer datos validados desde HDFS.
fact_prod_kpi = spark.read.parquet(
    f"{BASE_GOLD}/fact_produccion_mensual"
)

dim_cultivo_kpi = spark.read.parquet(
    f"{BASE_GOLD}/dim_cultivo"
)

dim_tiempo_kpi = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
)

print("\nPROGRESO: 25% - Incorporando dimensiones", flush=True)

# 2. Relacionar indicadores con cultivo y tiempo.
produccion_kpi_base = (
    fact_prod_kpi.alias("f")
    .join(
        dim_cultivo_kpi.select(
            "cultivo_id", "cultivo"
        ).alias("c"),
        "cultivo_id",
        "inner"
    )
    .join(
        dim_tiempo_kpi.select(
            "tiempo_id", "anio"
        ).alias("t"),
        "tiempo_id",
        "inner"
    )
)

print("\nPROGRESO: 50% - Calculando indicadores", flush=True)

# 3. Agrupar por cultivo y año.
kpi_produccion_anual = (
    produccion_kpi_base
    .groupBy("cultivo", "anio")
    .agg(
        F.sum("produccion_t").alias(
            "produccion_reportada_t"
        ),
        F.avg("precio_chacra_s_kg").alias(
            "precio_promedio_chacra_s_kg"
        ),
        F.count("*").alias("registros_total"),
        F.count("produccion_t").alias(
            "registros_con_produccion"
        ),
        F.count("precio_chacra_s_kg").alias(
            "registros_con_precio"
        )
    )
)

print("\n===== KPI DE PRODUCCION POR CULTIVO Y ANIO =====")

kpi_produccion_anual.orderBy(
    "anio", "cultivo"
).show(30, truncate=False)

print("\nPROGRESO: 75% - Verificando cobertura", flush=True)

print("\n===== COBERTURA DE INDICADORES =====")

kpi_produccion_anual.select(
    "cultivo",
    "anio",
    "registros_total",
    "registros_con_produccion",
    "registros_con_precio"
).orderBy(
    "anio", "cultivo"
).show(30, truncate=False)

print("\n===== VALIDACION =====")

registros_origen = fact_prod_kpi.count()
registros_relacionados = produccion_kpi_base.count()

registros_con_produccion = fact_prod_kpi.filter(
    F.col("produccion_t").isNotNull()
).count()

registros_con_precio = fact_prod_kpi.filter(
    F.col("precio_chacra_s_kg").isNotNull()
).count()

print("Registros originales:", registros_origen)
print("Registros relacionados:", registros_relacionados)
print("Registros con produccion:", registros_con_produccion)
print("Registros con precio:", registros_con_precio)

if (
    registros_origen == registros_relacionados == 561
    and registros_con_produccion == 267
    and registros_con_precio == 267
):
    kpi_produccion_validada = True
    print("\nKPI PRODUCCION: CONSTRUIDOS CORRECTAMENTE")
else:
    kpi_produccion_validada = False
    print("\nATENCION: Revisar KPI de produccion")

print("\nPROGRESO: 100% - Calculo finalizado")
print("===== PASO 74 FINALIZADO =====")

===== PASO 74: KPI DE PRODUCCION AGRICOLA =====
PROGRESO: 0% - Leyendo tablas dimensionales

PROGRESO: 25% - Incorporando dimensiones

PROGRESO: 50% - Calculando indicadores

===== KPI DE PRODUCCION POR CULTIVO Y ANIO =====
+---------+----+----------------------+---------------------------+---------------+------------------------+--------------------+
|cultivo  |anio|produccion_reportada_t|precio_promedio_chacra_s_kg|registros_total|registros_con_produccion|registros_con_precio|
+---------+----+----------------------+---------------------------+---------------+------------------------+--------------------+
|ALCACHOFA|2024|NULL                  |NULL                       |15             |0                       |0                   |
|ARANDANO |2024|NULL                  |NULL                       |55             |0                       |0                   |
|ESPARRAGO|2024|NULL                  |NULL                       |95             |0                       |0                 

## Almacenamiento Gold: KPI anuales de producción

Se almacenan los indicadores anuales de producción agrícola
por cultivo, calculados a partir de la tabla de hechos Gold.

Los KPI incluyen la producción total reportada en toneladas,
el precio promedio simple en chacra y la cantidad de registros
utilizados en cada cálculo.

Se preservan los valores nulos correspondientes a 2024,
debido a que no existen registros disponibles de producción
ni de precio para ese año.

La información se almacena en formato Parquet y permitirá
construir visualizaciones y analizar la cobertura de los datos.

Los resultados representan únicamente los registros
disponibles y no necesariamente totales regionales oficiales.

In [60]:
from pyspark.sql import functions as F

print("===== PASO 75: GUARDAR KPI PRODUCCION =====")

if not globals().get("kpi_produccion_validada", False):
    print("ERROR: Primero debes validar el Paso 74.")

else:
    ruta_kpi_produccion = (
        f"{BASE_GOLD}/kpi_produccion_anual"
    )

    print("PROGRESO: 0% - Validando KPI", flush=True)

    total_kpi = kpi_produccion_anual.count()

    duplicados_kpi = (
        kpi_produccion_anual
        .groupBy("cultivo", "anio")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    cobertura_invalida = kpi_produccion_anual.filter(
        (F.col("registros_con_produccion") > F.col("registros_total"))
        | (F.col("registros_con_precio") > F.col("registros_total"))
        | (F.col("registros_con_produccion") < 0)
        | (F.col("registros_con_precio") < 0)
    ).count()

    # En 2024, los dos KPI deben permanecer nulos.
    errores_2024 = kpi_produccion_anual.filter(
        (F.col("anio") == 2024)
        & (
            F.col("produccion_reportada_t").isNotNull()
            | F.col("precio_promedio_chacra_s_kg").isNotNull()
            | (F.col("registros_con_produccion") != 0)
            | (F.col("registros_con_precio") != 0)
        )
    ).count()

    print("Registros KPI:", total_kpi)
    print("Cultivo-anio duplicados:", duplicados_kpi)
    print("Errores de cobertura:", cobertura_invalida)
    print("Errores en datos de 2024:", errores_2024)

    kpi_listo = (
        total_kpi == 6
        and duplicados_kpi == 0
        and cobertura_invalida == 0
        and errores_2024 == 0
    )

    if not kpi_listo:
        print("\nATENCION: Revisar los KPI antes de guardar.")

    else:
        print(
            "\nPROGRESO: 30% - Comprobando destino",
            flush=True
        )

        hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

        ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
            ruta_kpi_produccion
        )

        filesystem = ruta_java.getFileSystem(hadoop_conf)

        if filesystem.exists(ruta_java):
            print("\nATENCION: La ruta ya existe.")
            print("No se sobrescribira ningun archivo.")
            print("Ruta:", ruta_kpi_produccion)

        else:
            print(
                "PROGRESO: 50% - Guardando Parquet",
                flush=True
            )

            (
                kpi_produccion_anual
                .coalesce(1)
                .write
                .mode("errorifexists")
                .parquet(ruta_kpi_produccion)
            )

            print(
                "PROGRESO: 75% - Verificando HDFS",
                flush=True
            )

            kpi_guardado = spark.read.parquet(
                ruta_kpi_produccion
            )

            registros_guardados = kpi_guardado.count()

            columnas_guardadas = len(kpi_guardado.columns)

            print("\n===== VERIFICACION HDFS =====")
            print("Registros guardados:", registros_guardados)
            print("Columnas guardadas:", columnas_guardadas)

            kpi_guardado.orderBy(
                "anio", "cultivo"
            ).show(10, truncate=False)

            if (
                registros_guardados == 6
                and columnas_guardadas == 7
            ):
                print("\nALMACENAMIENTO KPI PRODUCCION: CORRECTO")
            else:
                print("\nATENCION: Revisar KPI almacenados")

            print(
                "PROGRESO: 100% - Verificacion finalizada",
                flush=True
            )

print("===== PASO 75 FINALIZADO =====")

===== PASO 75: GUARDAR KPI PRODUCCION =====
PROGRESO: 0% - Validando KPI
Registros KPI: 6
Cultivo-anio duplicados: 0
Errores de cobertura: 0
Errores en datos de 2024: 0

PROGRESO: 30% - Comprobando destino
PROGRESO: 50% - Guardando Parquet
PROGRESO: 75% - Verificando HDFS

===== VERIFICACION HDFS =====
Registros guardados: 6
Columnas guardadas: 7
+---------+----+----------------------+---------------------------+---------------+------------------------+--------------------+
|cultivo  |anio|produccion_reportada_t|precio_promedio_chacra_s_kg|registros_total|registros_con_produccion|registros_con_precio|
+---------+----+----------------------+---------------------------+---------------+------------------------+--------------------+
|ALCACHOFA|2024|NULL                  |NULL                       |15             |0                       |0                   |
|ARANDANO |2024|NULL                  |NULL                       |55             |0                       |0                   |
|

## KPI Gold: exportaciones regionales y Danper

Se construyen indicadores anuales de exportaciones a partir
de las tablas de hechos y dimensiones almacenadas en Gold.

Para las exportaciones regionales se calcula el volumen
reportado en toneladas y el valor FOB en miles de dólares,
agrupados por cultivo y año.

Para las exportaciones propias de Danper se calcula el valor
FOB en dólares y la cantidad de registros aduaneros,
agrupados por país de destino y año.

Se incluyen contadores de disponibilidad para identificar
indicadores incompletos y conservar sus valores nulos.

Las fuentes permanecen separadas porque representan ámbitos
diferentes. Los datos de 2026 son parciales y los indicadores
no deben interpretarse como valores anuales completos.

In [62]:
from pyspark.sql import functions as F

print("===== PASO 76: KPI DE EXPORTACIONES =====")
print("PROGRESO: 0% - Leyendo tablas Gold", flush=True)

fact_regional_kpi = spark.read.parquet(
    f"{BASE_GOLD}/fact_exportaciones_regionales"
)

fact_danper_kpi = spark.read.parquet(
    f"{BASE_GOLD}/fact_exportaciones_danper"
)

dim_tiempo_exp_kpi = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
).select("tiempo_id", "anio")

dim_cultivo_exp_kpi = spark.read.parquet(
    f"{BASE_GOLD}/dim_cultivo"
).select("cultivo_id", "cultivo")

dim_pais_exp_kpi = spark.read.parquet(
    f"{BASE_GOLD}/dim_pais_exportacion"
).select("pais_id", "pais")

print("\nPROGRESO: 20% - Relacionando dimensiones", flush=True)

regional_base_kpi = (
    fact_regional_kpi
    .join(dim_tiempo_exp_kpi, "tiempo_id", "left")
    .join(dim_cultivo_exp_kpi, "cultivo_id", "left")
)

danper_base_kpi = (
    fact_danper_kpi
    .join(dim_tiempo_exp_kpi, "tiempo_id", "left")
    .join(dim_pais_exp_kpi, "pais_id", "left")
)

print("\nPROGRESO: 45% - Calculando KPI regionales", flush=True)

kpi_exportaciones_regionales_anual = (
    regional_base_kpi
    .groupBy("cultivo", "anio")
    .agg(
        F.sum("volumen_exportado_t").alias(
            "volumen_exportado_reportado_t"
        ),
        F.sum("valor_fob_miles_usd").alias(
            "valor_fob_reportado_miles_usd"
        ),
        F.count("*").alias("registros_total"),
        F.count("volumen_exportado_t").alias(
            "registros_con_volumen"
        ),
        F.count("valor_fob_miles_usd").alias(
            "registros_con_valor_fob"
        )
    )
)

print("\n===== KPI EXPORTACIONES REGIONALES =====")

kpi_exportaciones_regionales_anual.orderBy(
    "anio", "cultivo"
).show(65, truncate=False)

print("\nPROGRESO: 65% - Calculando KPI Danper", flush=True)

kpi_exportaciones_danper_anual = (
    danper_base_kpi
    .groupBy("pais", "anio")
    .agg(
        F.sum("valor_fob_usd").alias(
            "valor_fob_reportado_usd"
        ),
        F.sum("cantidad_registros_aduaneros").alias(
            "cantidad_registros_aduaneros"
        ),
        F.count("*").alias("registros_total")
    )
)

print("\n===== RESUMEN ANUAL EXPORTACIONES DANPER =====")

kpi_exportaciones_danper_anual.groupBy("anio").agg(
    F.sum("valor_fob_reportado_usd").alias(
        "valor_fob_usd"
    ),
    F.sum("cantidad_registros_aduaneros").alias(
        "registros_aduaneros"
    ),
    F.sum("registros_total").alias(
        "filas_originales"
    ),
    F.count("*").alias("paises_con_registros")
).orderBy("anio").show(truncate=False)

print("\nPROGRESO: 85% - Validando integridad", flush=True)

regional_origen = fact_regional_kpi.count()
regional_relacionado = regional_base_kpi.count()

danper_origen = fact_danper_kpi.count()
danper_relacionado = danper_base_kpi.count()

claves_nulas_regional = regional_base_kpi.filter(
    F.col("cultivo").isNull() | F.col("anio").isNull()
).count()

claves_nulas_danper = danper_base_kpi.filter(
    F.col("pais").isNull() | F.col("anio").isNull()
).count()

duplicados_regional = (
    kpi_exportaciones_regionales_anual
    .groupBy("cultivo", "anio")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

duplicados_danper = (
    kpi_exportaciones_danper_anual
    .groupBy("pais", "anio")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

total_kpi_regional = kpi_exportaciones_regionales_anual.count()
total_kpi_danper = kpi_exportaciones_danper_anual.count()

print("\n===== VALIDACION =====")
print("Regionales originales:", regional_origen)
print("Regionales relacionadas:", regional_relacionado)
print("Danper originales:", danper_origen)
print("Danper relacionadas:", danper_relacionado)
print("KPI regionales:", total_kpi_regional)
print("KPI Danper:", total_kpi_danper)
print("Relaciones regionales faltantes:", claves_nulas_regional)
print("Relaciones Danper faltantes:", claves_nulas_danper)
print("Duplicados KPI regional:", duplicados_regional)
print("Duplicados KPI Danper:", duplicados_danper)

if (
    regional_origen == regional_relacionado == 720
    and danper_origen == danper_relacionado == 495
    and total_kpi_regional == 60
    and total_kpi_danper > 0
    and claves_nulas_regional == 0
    and claves_nulas_danper == 0
    and duplicados_regional == 0
    and duplicados_danper == 0
):
    kpi_exportaciones_validada = True
    print("\nKPI EXPORTACIONES: CONSTRUIDOS CORRECTAMENTE")
else:
    kpi_exportaciones_validada = False
    print("\nATENCION: REVISAR KPI DE EXPORTACIONES")

print("\nPROGRESO: 100% - Calculo finalizado")
print("===== PASO 76 FINALIZADO =====")

===== PASO 76: KPI DE EXPORTACIONES =====
PROGRESO: 0% - Leyendo tablas Gold

PROGRESO: 20% - Relacionando dimensiones

PROGRESO: 45% - Calculando KPI regionales

===== KPI EXPORTACIONES REGIONALES =====
+---------+----+-----------------------------+-----------------------------+---------------+---------------------+-----------------------+
|cultivo  |anio|volumen_exportado_reportado_t|valor_fob_reportado_miles_usd|registros_total|registros_con_volumen|registros_con_valor_fob|
+---------+----+-----------------------------+-----------------------------+---------------+---------------------+-----------------------+
|ALCACHOFA|2012|24805.145334000008           |77655.18408000002            |12             |12                   |12                     |
|ARANDANO |2012|14.606240000000001           |114.36285999999998           |12             |7                    |12                     |
|ESPARRAGO|2012|71606.03581799999            |197391.37278000006           |12             |12       

## Almacenamiento Gold: KPI anuales de exportaciones

Se almacenan en Hadoop HDFS los indicadores anuales
de exportaciones regionales y exportaciones propias
de Danper, utilizando el formato Parquet.

Los indicadores regionales contienen información
histórica por cultivo y año durante 2012–2023.

Los indicadores de Danper contienen valores FOB
y cantidades de registros aduaneros por país y año,
correspondientes a 2024–2026.

Ambas fuentes se mantienen separadas debido a sus
diferentes ámbitos de análisis.

Los KPI conservan los contadores de cobertura y los
valores nulos originales, sin atribuir exportaciones
propias de Danper a cultivos específicos.

La información de 2026 es parcial.

In [63]:
from pyspark.sql import functions as F

print("===== PASO 77: GUARDAR KPI EXPORTACIONES =====")

if not globals().get("kpi_exportaciones_validada", False):
    print("ERROR: Primero debes validar el Paso 76.")

else:
    print("PROGRESO: 0% - Preparando almacenamiento", flush=True)

    # Nombre, DataFrame, cantidad esperada y claves naturales
    kpis_a_guardar = [
        (
            "kpi_exportaciones_regionales_anual",
            kpi_exportaciones_regionales_anual,
            60,
            ["cultivo", "anio"]
        ),
        (
            "kpi_exportaciones_danper_anual",
            kpi_exportaciones_danper_anual,
            81,
            ["pais", "anio"]
        )
    ]

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

    # Primero validar ambos KPI antes de escribir.
    errores_validacion = []

    for nombre, df, esperado, claves in kpis_a_guardar:

        total = df.count()

        duplicados = (
            df.groupBy(*claves)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        claves_nulas = df.filter(
            F.expr(
                " OR ".join(
                    f"`{campo}` IS NULL"
                    for campo in claves
                )
            )
        ).count()

        print(f"\n===== VALIDACION {nombre.upper()} =====")
        print("Registros:", total)
        print("Duplicados:", duplicados)
        print("Claves nulas:", claves_nulas)

        if (
            total != esperado
            or duplicados != 0
            or claves_nulas != 0
        ):
            errores_validacion.append(nombre)

    if errores_validacion:
        print("\nATENCION: No se guardaran los KPI.")
        print("Revisar:", ", ".join(errores_validacion))

    else:
        print(
            "\nPROGRESO: 30% - Guardando KPI validados",
            flush=True
        )

        resultados_kpi = []

        for nombre, df, esperado, claves in kpis_a_guardar:

            ruta = f"{BASE_GOLD}/{nombre}"

            print(f"\n===== {nombre.upper()} =====")
            print("Ruta:", ruta)

            ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
                ruta
            )

            filesystem = ruta_java.getFileSystem(hadoop_conf)

            # Si existe, verificar sin sobrescribir.
            if filesystem.exists(ruta_java):
                print("La ruta ya existe. Se verificara sin sobrescribir.")
            else:
                print("Guardando Parquet...", flush=True)

                (
                    df.coalesce(1)
                    .write
                    .mode("errorifexists")
                    .parquet(ruta)
                )

            print("Leyendo desde HDFS...", flush=True)

            guardado = spark.read.parquet(ruta)

            registros_guardados = guardado.count()
            columnas_guardadas = len(guardado.columns)

            duplicados_guardados = (
                guardado.groupBy(*claves)
                .count()
                .filter(F.col("count") > 1)
                .count()
            )

            print("Registros guardados:", registros_guardados)
            print("Columnas guardadas:", columnas_guardadas)
            print("Claves duplicadas:", duplicados_guardados)

            # Verificar todos los valores, incluidos los NULL.
            columnas = df.columns

            diferencias = (
                df.select(*columnas)
                .exceptAll(guardado.select(*columnas))
                .count()
            )

            diferencias += (
                guardado.select(*columnas)
                .exceptAll(df.select(*columnas))
                .count()
            )

            print("Filas diferentes:", diferencias)

            if (
                registros_guardados == esperado
                and columnas_guardadas == len(columnas)
                and duplicados_guardados == 0
                and diferencias == 0
            ):
                estado = "CORRECTO"
            else:
                estado = "REVISAR"

            resultados_kpi.append((nombre, estado))

            print("Estado:", estado)

        print("\n===== RESUMEN FINAL =====")

        for nombre, estado in resultados_kpi:
            print(f"{nombre}: {estado}")

        if all(
            estado == "CORRECTO"
            for _, estado in resultados_kpi
        ):
            print("\nALMACENAMIENTO KPI EXPORTACIONES: CORRECTO")
        else:
            print("\nATENCION: Revisar KPI almacenados")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 77 FINALIZADO =====")

===== PASO 77: GUARDAR KPI EXPORTACIONES =====
PROGRESO: 0% - Preparando almacenamiento

===== VALIDACION KPI_EXPORTACIONES_REGIONALES_ANUAL =====
Registros: 60
Duplicados: 0
Claves nulas: 0

===== VALIDACION KPI_EXPORTACIONES_DANPER_ANUAL =====
Registros: 81
Duplicados: 0
Claves nulas: 0

PROGRESO: 30% - Guardando KPI validados

===== KPI_EXPORTACIONES_REGIONALES_ANUAL =====
Ruta: hdfs://localhost:9000/danper/gold/kpi_exportaciones_regionales_anual
Guardando Parquet...
Leyendo desde HDFS...
Registros guardados: 60
Columnas guardadas: 7
Claves duplicadas: 0


ERROR:root:KeyboardInterrupt while sending command.                 (1 + 0) / 2]
Traceback (most recent call last):
  File "/home/danna/BigData_UPAO/bigdata_env/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/danna/BigData_UPAO/bigdata_env/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [64]:
from pyspark.sql import functions as F

print("===== RECUPERACION PASO 77 =====")

ruta = f"{BASE_GOLD}/kpi_exportaciones_regionales_anual"

kpi_regional_guardado = spark.read.parquet(ruta)

total = kpi_regional_guardado.count()
columnas = len(kpi_regional_guardado.columns)

duplicados = (
    kpi_regional_guardado
    .groupBy("cultivo", "anio")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Registros:", total)
print("Columnas:", columnas)
print("Claves duplicadas:", duplicados)

if total == 60 and columnas == 7 and duplicados == 0:
    print("VERIFICACION BASICA KPI REGIONAL: CORRECTA")
else:
    print("ATENCION: REVISAR KPI REGIONAL")

print("===== RECUPERACION FINALIZADA =====")

===== RECUPERACION PASO 77 =====


Registros: 60
Columnas: 7
Claves duplicadas: 0
VERIFICACION BASICA KPI REGIONAL: CORRECTA
===== RECUPERACION FINALIZADA =====


## Recuperación del almacenamiento de KPI de exportaciones Danper

Se continúa el almacenamiento de los KPI de exportaciones
después de verificar que la tabla regional fue guardada
correctamente en Hadoop HDFS.

Se almacena la tabla de indicadores de exportaciones
propias de Danper, agrupados por país y año.

Para evitar operaciones de comparación costosas, se
verifican la cantidad de registros, el esquema, la
unicidad de las claves y los totales de las medidas.

Se conserva la separación entre exportaciones regionales
y exportaciones propias de Danper.

La información del año 2026 corresponde a un periodo parcial.

In [65]:
from pyspark.sql import functions as F

print("===== PASO 77.1: GUARDAR KPI EXPORTACIONES DANPER =====")

ruta_kpi_danper = (
    f"{BASE_GOLD}/kpi_exportaciones_danper_anual"
)

print("PROGRESO: 0% - Validando origen", flush=True)

# 1. Comprobar que los KPI se construyeron correctamente.
if not globals().get("kpi_exportaciones_validada", False):
    print("ERROR: Primero debes ejecutar correctamente el Paso 76.")

else:
    total_origen = kpi_exportaciones_danper_anual.count()

    duplicados_origen = (
        kpi_exportaciones_danper_anual
        .groupBy("pais", "anio")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print("Registros KPI originales:", total_origen)
    print("Claves duplicadas origen:", duplicados_origen)

    if total_origen != 81 or duplicados_origen != 0:
        print("ATENCION: Revisar los KPI antes de guardar.")

    else:
        print(
            "\nPROGRESO: 25% - Comprobando destino HDFS",
            flush=True
        )

        hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

        ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
            ruta_kpi_danper
        )

        filesystem = ruta_java.getFileSystem(hadoop_conf)

        if filesystem.exists(ruta_java):
            print("La ruta ya existe. No se sobrescribira.")
        else:
            print(
                "\nPROGRESO: 50% - Guardando Parquet",
                flush=True
            )

            (
                kpi_exportaciones_danper_anual
                .coalesce(1)
                .write
                .mode("errorifexists")
                .parquet(ruta_kpi_danper)
            )

        print(
            "\nPROGRESO: 75% - Verificando almacenamiento",
            flush=True
        )

        # 2. Leer la tabla directamente desde HDFS.
        kpi_danper_guardado = spark.read.parquet(
            ruta_kpi_danper
        )

        total_guardado = kpi_danper_guardado.count()
        columnas_guardadas = len(kpi_danper_guardado.columns)

        duplicados_guardados = (
            kpi_danper_guardado
            .groupBy("pais", "anio")
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        claves_nulas = kpi_danper_guardado.filter(
            F.col("pais").isNull()
            | F.col("anio").isNull()
        ).count()

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", total_guardado)
        print("Columnas guardadas:", columnas_guardadas)
        print("Claves duplicadas:", duplicados_guardados)
        print("Claves nulas:", claves_nulas)

        # 3. Revisar los totales anuales almacenados.
        print("\n===== RESUMEN ANUAL DANPER EN HDFS =====")

        (
            kpi_danper_guardado
            .groupBy("anio")
            .agg(
                F.round(
                    F.sum("valor_fob_reportado_usd"), 2
                ).alias("valor_fob_usd"),
                F.sum(
                    "cantidad_registros_aduaneros"
                ).alias("registros_aduaneros"),
                F.sum("registros_total").alias(
                    "filas_originales"
                )
            )
            .orderBy("anio")
            .show(truncate=False)
        )

        # 4. Comparar agregados anuales sin exceptAll.
        origen_anual = (
            kpi_exportaciones_danper_anual
            .groupBy("anio")
            .agg(
                F.sum("valor_fob_reportado_usd").alias("fob"),
                F.sum("cantidad_registros_aduaneros").alias(
                    "aduanas"
                ),
                F.sum("registros_total").alias("filas")
            )
        )

        guardado_anual = (
            kpi_danper_guardado
            .groupBy("anio")
            .agg(
                F.sum("valor_fob_reportado_usd").alias("fob"),
                F.sum("cantidad_registros_aduaneros").alias(
                    "aduanas"
                ),
                F.sum("registros_total").alias("filas")
            )
        )

        comparacion = (
            origen_anual.alias("o")
            .join(
                guardado_anual.alias("g"),
                "anio",
                "full"
            )
        )

        errores_totales = comparacion.filter(
            F.col("o.fob").isNull()
            | F.col("g.fob").isNull()
            | (
                F.abs(
                    F.col("o.fob") - F.col("g.fob")
                ) > 0.01
            )
            | ~F.col("o.aduanas").eqNullSafe(F.col("g.aduanas"))
            | ~F.col("o.filas").eqNullSafe(F.col("g.filas"))
        ).count()

        print("Diferencias en totales anuales:", errores_totales)

        print("\n===== RESULTADO FINAL =====")

        if (
            total_guardado == 81
            and columnas_guardadas == 5
            and duplicados_guardados == 0
            and claves_nulas == 0
            and errores_totales == 0
        ):
            kpi_danper_almacenado_validado = True

            print("ALMACENAMIENTO KPI DANPER: CORRECTO")
        else:
            kpi_danper_almacenado_validado = False

            print("ATENCION: Revisar KPI Danper almacenado")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 77.1 FINALIZADO =====")

===== PASO 77.1: GUARDAR KPI EXPORTACIONES DANPER =====
PROGRESO: 0% - Validando origen
Registros KPI originales: 81
Claves duplicadas origen: 0

PROGRESO: 25% - Comprobando destino HDFS

PROGRESO: 50% - Guardando Parquet

PROGRESO: 75% - Verificando almacenamiento

===== VERIFICACION HDFS =====
Registros guardados: 81
Columnas guardadas: 5
Claves duplicadas: 0
Claves nulas: 0

===== RESUMEN ANUAL DANPER EN HDFS =====
+----+--------------+-------------------+----------------+
|anio|valor_fob_usd |registros_aduaneros|filas_originales|
+----+--------------+-------------------+----------------+
|2024|2.7603960728E8|444                |204             |
|2025|3.2994685796E8|542                |223             |
|2026|5.243515171E7 |138                |68              |
+----+--------------+-------------------+----------------+

Diferencias en totales anuales: 0

===== RESULTADO FINAL =====
ALMACENAMIENTO KPI DANPER: CORRECTO
PROGRESO: 100% - Verificacion finalizada
===== PASO 77.1 FINALIZA

## KPI Gold: actividad agroindustrial por cultivo, proceso y presentación

Se construyen indicadores de actividad agroindustrial
a partir de la tabla de hechos validada en Gold.

Los indicadores se agrupan por cultivo, año, proceso y
presentación del producto, considerando los registros
de espárrago y alcachofa entre 2010 y 2024.

Se calcula la cantidad total reportada en toneladas,
la cantidad de registros utilizados y la cantidad
de registros con valores positivos o iguales a cero.

Los procesos INGRESO, PRODUCCION, UTILIZACION y VENTA
se mantienen separados para evitar duplicar o mezclar
volúmenes que representan actividades diferentes.

Los KPI corresponden a los datos disponibles en la
fuente y servirán para analizar la evolución histórica
de las actividades agroindustriales.

In [66]:
from pyspark.sql import functions as F

print("===== PASO 78: KPI AGROINDUSTRIALES =====")
print("PROGRESO: 0% - Leyendo tablas Gold", flush=True)

# 1. Leer la tabla de hechos y sus dimensiones.
fact_agro_kpi = spark.read.parquet(
    f"{BASE_GOLD}/fact_agroindustria_mensual"
)

dim_cultivo_agro_kpi = spark.read.parquet(
    f"{BASE_GOLD}/dim_cultivo"
).select("cultivo_id", "cultivo")

dim_tiempo_agro_kpi = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
).select("tiempo_id", "anio")

print("\nPROGRESO: 25% - Relacionando dimensiones", flush=True)

# 2. Incorporar cultivo y año sin modificar las medidas.
agro_base_kpi = (
    fact_agro_kpi
    .join(dim_cultivo_agro_kpi, "cultivo_id", "left")
    .join(dim_tiempo_agro_kpi, "tiempo_id", "left")
)

print("\nPROGRESO: 50% - Construyendo KPI", flush=True)

# 3. Agrupar manteniendo separados los procesos
#    y las presentaciones comerciales.
kpi_agroindustria_anual = (
    agro_base_kpi
    .groupBy(
        "cultivo",
        "anio",
        "proceso",
        "presentacion"
    )
    .agg(
        F.sum("cantidad_t").alias(
            "cantidad_reportada_t"
        ),
        F.count("*").alias(
            "registros_total"
        ),
        F.count("cantidad_t").alias(
            "registros_con_cantidad"
        ),
        F.sum(
            F.when(F.col("cantidad_t") == 0, 1)
            .otherwise(0)
        ).alias("registros_cantidad_cero"),
        F.sum(
            F.when(F.col("cantidad_t") > 0, 1)
            .otherwise(0)
        ).alias("registros_cantidad_positiva")
    )
)

print("\n===== RESUMEN POR CULTIVO Y PROCESO =====")

# Solo para visualizar cada proceso independientemente.
(
    kpi_agroindustria_anual
    .groupBy("cultivo", "proceso")
    .agg(
        F.round(
            F.sum("cantidad_reportada_t"), 2
        ).alias("cantidad_reportada_t"),
        F.sum("registros_total").alias(
            "registros_total"
        ),
        F.sum("registros_cantidad_cero").alias(
            "registros_cero"
        )
    )
    .orderBy("cultivo", "proceso")
    .show(20, truncate=False)
)

print("\n===== MUESTRA KPI POR ANIO Y PRESENTACION =====")

(
    kpi_agroindustria_anual
    .orderBy("anio", "cultivo", "proceso", "presentacion")
    .show(25, truncate=False)
)

print("\nPROGRESO: 75% - Validando integridad", flush=True)

registros_origen_agro = fact_agro_kpi.count()

registros_relacionados_agro = agro_base_kpi.count()

registros_kpi_agro = kpi_agroindustria_anual.count()

relaciones_faltantes_agro = agro_base_kpi.filter(
    F.col("cultivo").isNull()
    | F.col("anio").isNull()
).count()

duplicados_kpi_agro = (
    kpi_agroindustria_anual
    .groupBy(
        "cultivo",
        "anio",
        "proceso",
        "presentacion"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Comprobar que los contadores de los KPI
# representan exactamente los registros originales.
contadores_agro = kpi_agroindustria_anual.agg(
    F.sum("registros_total").alias("total"),
    F.sum("registros_con_cantidad").alias("con_cantidad"),
    F.sum("registros_cantidad_cero").alias("ceros"),
    F.sum("registros_cantidad_positiva").alias("positivos")
).first()

ceros_origen_agro = fact_agro_kpi.filter(
    F.col("cantidad_t") == 0
).count()

nulos_origen_agro = fact_agro_kpi.filter(
    F.col("cantidad_t").isNull()
).count()

negativos_origen_agro = fact_agro_kpi.filter(
    F.col("cantidad_t") < 0
).count()

print("\n===== VALIDACION =====")

print("Registros originales:", registros_origen_agro)
print("Registros relacionados:", registros_relacionados_agro)
print("Registros KPI:", registros_kpi_agro)
print("Relaciones faltantes:", relaciones_faltantes_agro)
print("Duplicados KPI:", duplicados_kpi_agro)

print("Registros representados en KPI:", contadores_agro["total"])
print("Registros con cantidad:", contadores_agro["con_cantidad"])
print("Registros con cantidad cero:", contadores_agro["ceros"])
print("Registros con cantidad positiva:", contadores_agro["positivos"])

print("Ceros en origen:", ceros_origen_agro)
print("Nulos en origen:", nulos_origen_agro)
print("Negativos en origen:", negativos_origen_agro)

print("\n===== RESULTADO FINAL =====")

if (
    registros_origen_agro == registros_relacionados_agro == 3134
    and registros_kpi_agro > 0
    and relaciones_faltantes_agro == 0
    and duplicados_kpi_agro == 0
    and contadores_agro["total"] == 3134
    and contadores_agro["con_cantidad"] == 3134
    and contadores_agro["ceros"] == ceros_origen_agro == 978
    and contadores_agro["positivos"] == 3134 - 978
    and nulos_origen_agro == 0
    and negativos_origen_agro == 0
):
    kpi_agroindustria_validada = True
    print("KPI AGROINDUSTRIA: CONSTRUIDOS CORRECTAMENTE")
else:
    kpi_agroindustria_validada = False
    print("ATENCION: REVISAR KPI AGROINDUSTRIA")

print("\nPROGRESO: 100% - Calculo finalizado")
print("===== PASO 78 FINALIZADO =====")

===== PASO 78: KPI AGROINDUSTRIALES =====
PROGRESO: 0% - Leyendo tablas Gold

PROGRESO: 25% - Relacionando dimensiones

PROGRESO: 50% - Construyendo KPI

===== RESUMEN POR CULTIVO Y PROCESO =====
+---------+-----------+--------------------+---------------+--------------+
|cultivo  |proceso    |cantidad_reportada_t|registros_total|registros_cero|
+---------+-----------+--------------------+---------------+--------------+
|ALCACHOFA|INGRESO    |1106802.27          |180            |25            |
|ALCACHOFA|PRODUCCION |362601.87           |540            |293           |
|ALCACHOFA|UTILIZACION|1126123.05          |180            |24            |
|ALCACHOFA|VENTA      |389810.09           |540            |264           |
|ESPARRAGO|INGRESO    |1177213.24          |180            |12            |
|ESPARRAGO|PRODUCCION |600307.95           |662            |159           |
|ESPARRAGO|UTILIZACION|1113890.49          |180            |12            |
|ESPARRAGO|VENTA      |572759.06           |

## Almacenamiento Gold: KPI anuales de actividad agroindustrial

Se almacenan en Hadoop HDFS los indicadores agroindustriales
calculados por cultivo, año, proceso y presentación del producto.

La tabla contiene 262 registros agregados que representan
3134 observaciones originales de espárrago y alcachofa.

Se conserva la separación entre los procesos INGRESO,
PRODUCCION, UTILIZACION y VENTA, evitando sumar directamente
operaciones de naturaleza diferente.

Se mantiene la información correspondiente a 978 registros
con cantidades iguales a cero y 2156 registros con cantidades
positivas.

Los indicadores se almacenan en formato Parquet dentro de
la zona Gold y podrán utilizarse en consultas analíticas
y visualizaciones posteriores.

In [67]:
from pyspark.sql import functions as F

print("===== PASO 79: GUARDAR KPI AGROINDUSTRIA =====")

if not globals().get("kpi_agroindustria_validada", False):
    print("ERROR: Primero debes validar el Paso 78.")

else:
    ruta_kpi_agro = f"{BASE_GOLD}/kpi_agroindustria_anual"

    print("PROGRESO: 0% - Validando KPI", flush=True)

    # 1. Verificar cantidad y unicidad antes de guardar.
    total_origen = kpi_agroindustria_anual.count()
    columnas_origen = len(kpi_agroindustria_anual.columns)

    claves_agro = [
        "cultivo",
        "anio",
        "proceso",
        "presentacion"
    ]

    duplicados_origen = (
        kpi_agroindustria_anual
        .groupBy(*claves_agro)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    resumen_origen = kpi_agroindustria_anual.agg(
        F.sum("registros_total").alias("filas"),
        F.sum("registros_con_cantidad").alias("con_cantidad"),
        F.sum("registros_cantidad_cero").alias("ceros"),
        F.sum("registros_cantidad_positiva").alias("positivos")
    ).first()

    print("Registros KPI:", total_origen)
    print("Columnas:", columnas_origen)
    print("Claves duplicadas:", duplicados_origen)
    print("Registros representados:", resumen_origen["filas"])
    print("Registros con cero:", resumen_origen["ceros"])
    print("Registros positivos:", resumen_origen["positivos"])

    origen_correcto = (
        total_origen == 262
        and columnas_origen == 9
        and duplicados_origen == 0
        and resumen_origen["filas"] == 3134
        and resumen_origen["con_cantidad"] == 3134
        and resumen_origen["ceros"] == 978
        and resumen_origen["positivos"] == 2156
    )

    if not origen_correcto:
        print("\nATENCION: Revisar los KPI antes de guardar.")

    else:
        print(
            "\nPROGRESO: 30% - Comprobando destino HDFS",
            flush=True
        )

        hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

        ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
            ruta_kpi_agro
        )

        filesystem = ruta_java.getFileSystem(hadoop_conf)

        if filesystem.exists(ruta_java):
            print("La ruta ya existe. No se sobrescribira.")
        else:
            print(
                "\nPROGRESO: 50% - Guardando Parquet",
                flush=True
            )

            (
                kpi_agroindustria_anual
                .coalesce(1)
                .write
                .mode("errorifexists")
                .parquet(ruta_kpi_agro)
            )

        print(
            "\nPROGRESO: 75% - Verificando almacenamiento",
            flush=True
        )

        # 2. Verificar directamente desde HDFS.
        kpi_agro_guardado = spark.read.parquet(ruta_kpi_agro)

        total_guardado = kpi_agro_guardado.count()
        columnas_guardadas = len(kpi_agro_guardado.columns)

        duplicados_guardados = (
            kpi_agro_guardado
            .groupBy(*claves_agro)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        claves_nulas = kpi_agro_guardado.filter(
            F.col("cultivo").isNull()
            | F.col("anio").isNull()
            | F.col("proceso").isNull()
            | F.col("presentacion").isNull()
        ).count()

        resumen_guardado = kpi_agro_guardado.agg(
            F.sum("registros_total").alias("filas"),
            F.sum("registros_con_cantidad").alias("con_cantidad"),
            F.sum("registros_cantidad_cero").alias("ceros"),
            F.sum("registros_cantidad_positiva").alias("positivos")
        ).first()

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", total_guardado)
        print("Columnas guardadas:", columnas_guardadas)
        print("Claves duplicadas:", duplicados_guardados)
        print("Claves nulas:", claves_nulas)

        print(
            "Registros representados:",
            resumen_guardado["filas"]
        )
        print(
            "Registros con cero:",
            resumen_guardado["ceros"]
        )
        print(
            "Registros positivos:",
            resumen_guardado["positivos"]
        )

        # 3. Comparar totales generales sin exceptAll.
        medidas_iguales = (
            resumen_origen.asDict()
            == resumen_guardado.asDict()
        )

        print("Contadores iguales al origen:", medidas_iguales)

        # 4. Comprobar suma total por separado para cada proceso.
        print("\n===== RESUMEN GUARDADO POR PROCESO =====")

        (
            kpi_agro_guardado
            .groupBy("cultivo", "proceso")
            .agg(
                F.round(
                    F.sum("cantidad_reportada_t"), 2
                ).alias("cantidad_reportada_t"),
                F.sum("registros_total").alias("registros")
            )
            .orderBy("cultivo", "proceso")
            .show(truncate=False)
        )

        print("\n===== RESULTADO FINAL =====")

        if (
            total_guardado == 262
            and columnas_guardadas == 9
            and duplicados_guardados == 0
            and claves_nulas == 0
            and medidas_iguales
        ):
            kpi_agroindustria_almacenado_validado = True
            print("ALMACENAMIENTO KPI AGROINDUSTRIA: CORRECTO")
        else:
            kpi_agroindustria_almacenado_validado = False
            print("ATENCION: Revisar KPI agroindustrial almacenado")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 79 FINALIZADO =====")

===== PASO 79: GUARDAR KPI AGROINDUSTRIA =====
PROGRESO: 0% - Validando KPI
Registros KPI: 262
Columnas: 9
Claves duplicadas: 0
Registros representados: 3134
Registros con cero: 978
Registros positivos: 2156

PROGRESO: 30% - Comprobando destino HDFS

PROGRESO: 50% - Guardando Parquet

PROGRESO: 75% - Verificando almacenamiento

===== VERIFICACION HDFS =====
Registros guardados: 262
Columnas guardadas: 9
Claves duplicadas: 0
Claves nulas: 0
Registros representados: 3134
Registros con cero: 978
Registros positivos: 2156
Contadores iguales al origen: True

===== RESUMEN GUARDADO POR PROCESO =====
+---------+-----------+--------------------+---------+
|cultivo  |proceso    |cantidad_reportada_t|registros|
+---------+-----------+--------------------+---------+
|ALCACHOFA|INGRESO    |1106802.27          |180      |
|ALCACHOFA|PRODUCCION |362601.87           |540      |
|ALCACHOFA|UTILIZACION|1126123.05          |180      |
|ALCACHOFA|VENTA      |389810.09           |540      |
|ESPARRAGO|ING

## KPI Gold: indicadores de costos agrícolas

Se construyen indicadores de costos agrícolas a partir de
la tabla de hechos almacenada en Hadoop HDFS.

Los indicadores se agrupan por región, año y tipo de costo,
considerando los registros disponibles entre 2018 y 2026.

Para cada grupo se calcula el valor promedio, mínimo y
máximo reportado, así como el número total de registros
y la cantidad de observaciones válidas.

Los tipos ALQUILER_TRACTOR, PRECIO_YUNTA y VALOR_JORNAL
se analizan por separado, evitando agregar medidas de
naturaleza diferente.

Debido a que las unidades originales no están confirmadas,
los resultados se interpretan como valores reportados,
sin atribuirles una unidad monetaria o temporal específica.

Estos indicadores permitirán explorar diferencias regionales
y tendencias históricas, considerando las limitaciones
de la calidad de los datos.

In [68]:
from pyspark.sql import functions as F

print("===== PASO 80: KPI DE COSTOS AGRICOLAS =====")
print("PROGRESO: 0% - Leyendo tablas Gold", flush=True)

# 1. Leer las tablas existentes directamente desde HDFS.
fact_costos_kpi = spark.read.parquet(
    f"{BASE_GOLD}/fact_costos_mensuales"
)

dim_region_costos_kpi = spark.read.parquet(
    f"{BASE_GOLD}/dim_region_costos"
).select("region_id", "region")

dim_tiempo_costos_kpi = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
).select("tiempo_id", "anio")

print("\nPROGRESO: 25% - Relacionando dimensiones", flush=True)

# 2. Incorporar region y año.
costos_base_kpi = (
    fact_costos_kpi
    .join(dim_region_costos_kpi, "region_id", "left")
    .join(dim_tiempo_costos_kpi, "tiempo_id", "left")
)

print("\nPROGRESO: 50% - Calculando KPI", flush=True)

# 3. Calcular estadisticos sin mezclar tipos de costo.
# Se conserva la unidad original para cada grupo.
kpi_costos_anual = (
    costos_base_kpi
    .groupBy(
        "region",
        "anio",
        "tipo_costo",
        "unidad"
    )
    .agg(
        F.avg("valor_costo").alias(
            "valor_promedio_reportado"
        ),
        F.min("valor_costo").alias(
            "valor_minimo_reportado"
        ),
        F.max("valor_costo").alias(
            "valor_maximo_reportado"
        ),
        F.count("*").alias(
            "registros_total"
        ),
        F.count("valor_costo").alias(
            "registros_con_valor"
        )
    )
)

print("\n===== MUESTRA KPI DE COSTOS =====")

(
    kpi_costos_anual
    .orderBy("anio", "region", "tipo_costo")
    .show(25, truncate=False)
)

print("\nPROGRESO: 70% - Revisando cobertura", flush=True)

print("\n===== COBERTURA POR TIPO DE COSTO =====")

(
    kpi_costos_anual
    .groupBy("tipo_costo", "unidad")
    .agg(
        F.sum("registros_total").alias(
            "registros_total"
        ),
        F.sum("registros_con_valor").alias(
            "registros_con_valor"
        ),
        F.count("*").alias(
            "grupos_kpi"
        )
    )
    .orderBy("tipo_costo")
    .show(truncate=False)
)

print("\nPROGRESO: 85% - Validando integridad", flush=True)

registros_origen_costos = fact_costos_kpi.count()
registros_relacionados_costos = costos_base_kpi.count()
registros_kpi_costos = kpi_costos_anual.count()

relaciones_faltantes_costos = costos_base_kpi.filter(
    F.col("region").isNull()
    | F.col("anio").isNull()
).count()

duplicados_kpi_costos = (
    kpi_costos_anual
    .groupBy(
        "region",
        "anio",
        "tipo_costo",
        "unidad"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

resumen_costos = kpi_costos_anual.agg(
    F.sum("registros_total").alias("total"),
    F.sum("registros_con_valor").alias("validos")
).first()

nulos_origen_costos = fact_costos_kpi.filter(
    F.col("valor_costo").isNull()
).count()

unidades_no_confirmadas = fact_costos_kpi.filter(
    F.col("unidad") == "NO_CONFIRMADA"
).count()

print("\n===== VALIDACION =====")

print("Registros originales:", registros_origen_costos)
print("Registros relacionados:", registros_relacionados_costos)
print("Registros KPI:", registros_kpi_costos)
print("Relaciones faltantes:", relaciones_faltantes_costos)
print("KPI duplicados:", duplicados_kpi_costos)
print("Registros representados:", resumen_costos["total"])
print("Registros con valor:", resumen_costos["validos"])
print("Valores nulos en origen:", nulos_origen_costos)
print("Unidades NO_CONFIRMADA:", unidades_no_confirmadas)

print("\n===== RESULTADO FINAL =====")

if (
    registros_origen_costos == registros_relacionados_costos == 6292
    and registros_kpi_costos > 0
    and relaciones_faltantes_costos == 0
    and duplicados_kpi_costos == 0
    and resumen_costos["total"] == 6292
    and resumen_costos["validos"] == (
        6292 - nulos_origen_costos
    )
    and unidades_no_confirmadas == 6292
):
    kpi_costos_validada = True
    print("KPI COSTOS: CONSTRUIDOS CORRECTAMENTE")
else:
    kpi_costos_validada = False
    print("ATENCION: REVISAR KPI DE COSTOS")

print("\nPROGRESO: 100% - Calculo finalizado")
print("===== PASO 80 FINALIZADO =====")

===== PASO 80: KPI DE COSTOS AGRICOLAS =====
PROGRESO: 0% - Leyendo tablas Gold

PROGRESO: 25% - Relacionando dimensiones

PROGRESO: 50% - Calculando KPI

===== MUESTRA KPI DE COSTOS =====
+------------+----+----------------+-------------+------------------------+----------------------+----------------------+---------------+-------------------+
|region      |anio|tipo_costo      |unidad       |valor_promedio_reportado|valor_minimo_reportado|valor_maximo_reportado|registros_total|registros_con_valor|
+------------+----+----------------+-------------+------------------------+----------------------+----------------------+---------------+-------------------+
|AMAZONAS    |2018|ALQUILER_TRACTOR|NO_CONFIRMADA|100.75                  |97.0                  |102.0                 |12             |12                 |
|AMAZONAS    |2018|PRECIO_YUNTA    |NO_CONFIRMADA|82.04166666666667       |80.0                  |83.0                  |12             |12                 |
|AMAZONAS    |2018|VA

## Almacenamiento Gold: KPI anuales de costos agrícolas

Se almacenan en Hadoop HDFS los indicadores anuales de
costos agrícolas calculados por región, año y tipo de costo.

La tabla contiene 594 registros agregados que representan
6292 observaciones originales del periodo 2018–2026.

Los indicadores incluyen valores promedio, mínimo y máximo,
además de contadores de disponibilidad de información.

Los tipos ALQUILER_TRACTOR, PRECIO_YUNTA y VALOR_JORNAL
permanecen separados para evitar combinar medidas
de naturaleza diferente.

Se conserva la unidad NO_CONFIRMADA y los valores iguales
a cero tal como fueron registrados en la fuente.

Los datos se almacenan en formato Parquet dentro de
la zona Gold para su posterior análisis y visualización.

In [69]:
from pyspark.sql import functions as F

print("===== PASO 81: GUARDAR KPI COSTOS AGRICOLAS =====")

if not globals().get("kpi_costos_validada", False):
    print("ERROR: Primero debes validar el Paso 80.")

else:
    ruta_kpi_costos = f"{BASE_GOLD}/kpi_costos_anual"

    print("PROGRESO: 0% - Validando KPI", flush=True)

    claves_costos = [
        "region",
        "anio",
        "tipo_costo",
        "unidad"
    ]

    # 1. Verificar la tabla original antes de guardarla.
    total_origen = kpi_costos_anual.count()
    columnas_origen = len(kpi_costos_anual.columns)

    duplicados_origen = (
        kpi_costos_anual
        .groupBy(*claves_costos)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    claves_nulas_origen = (
        kpi_costos_anual
        .filter(
            F.col("region").isNull()
            | F.col("anio").isNull()
            | F.col("tipo_costo").isNull()
            | F.col("unidad").isNull()
        )
        .count()
    )

    resumen_origen = kpi_costos_anual.agg(
        F.sum("registros_total").alias("total"),
        F.sum("registros_con_valor").alias("validos")
    ).first()

    print("Registros KPI:", total_origen)
    print("Columnas:", columnas_origen)
    print("Claves duplicadas:", duplicados_origen)
    print("Claves nulas:", claves_nulas_origen)
    print("Registros representados:", resumen_origen["total"])
    print("Registros con valor:", resumen_origen["validos"])

    origen_correcto = (
        total_origen == 594
        and columnas_origen == 9
        and duplicados_origen == 0
        and claves_nulas_origen == 0
        and resumen_origen["total"] == 6292
        and resumen_origen["validos"] == 6292
    )

    if not origen_correcto:
        print("\nATENCION: Revisar los KPI antes de guardar.")

    else:
        print(
            "\nPROGRESO: 30% - Comprobando destino HDFS",
            flush=True
        )

        hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

        ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
            ruta_kpi_costos
        )

        filesystem = ruta_java.getFileSystem(hadoop_conf)

        if filesystem.exists(ruta_java):
            print("La ruta ya existe. No se sobrescribira.")
        else:
            print(
                "\nPROGRESO: 50% - Guardando Parquet",
                flush=True
            )

            (
                kpi_costos_anual
                .coalesce(1)
                .write
                .mode("errorifexists")
                .parquet(ruta_kpi_costos)
            )

        print(
            "\nPROGRESO: 75% - Verificando HDFS",
            flush=True
        )

        # 2. Leer los datos desde HDFS.
        kpi_costos_guardado = spark.read.parquet(
            ruta_kpi_costos
        )

        total_guardado = kpi_costos_guardado.count()
        columnas_guardadas = len(kpi_costos_guardado.columns)

        duplicados_guardados = (
            kpi_costos_guardado
            .groupBy(*claves_costos)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        claves_nulas_guardadas = (
            kpi_costos_guardado
            .filter(
                F.col("region").isNull()
                | F.col("anio").isNull()
                | F.col("tipo_costo").isNull()
                | F.col("unidad").isNull()
            )
            .count()
        )

        resumen_guardado = kpi_costos_guardado.agg(
            F.sum("registros_total").alias("total"),
            F.sum("registros_con_valor").alias("validos")
        ).first()

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", total_guardado)
        print("Columnas guardadas:", columnas_guardadas)
        print("Claves duplicadas:", duplicados_guardados)
        print("Claves nulas:", claves_nulas_guardadas)
        print(
            "Registros representados:",
            resumen_guardado["total"]
        )
        print(
            "Registros con valor:",
            resumen_guardado["validos"]
        )

        # 3. Comparar los contadores sin utilizar exceptAll.
        contadores_iguales = (
            resumen_origen.asDict()
            == resumen_guardado.asDict()
        )

        print("Contadores iguales al origen:", contadores_iguales)

        # 4. Verificar la distribución por tipo de costo.
        print("\n===== RESUMEN POR TIPO DE COSTO =====")

        (
            kpi_costos_guardado
            .groupBy("tipo_costo", "unidad")
            .agg(
                F.sum("registros_total").alias("registros"),
                F.count("*").alias("grupos_kpi")
            )
            .orderBy("tipo_costo")
            .show(truncate=False)
        )

        distribucion_guardada = {
            fila["tipo_costo"]: (
                fila["registros"],
                fila["grupos_kpi"]
            )
            for fila in (
                kpi_costos_guardado
                .groupBy("tipo_costo")
                .agg(
                    F.sum("registros_total").alias("registros"),
                    F.count("*").alias("grupos_kpi")
                )
                .collect()
            )
        }

        distribucion_esperada = {
            "ALQUILER_TRACTOR": (2275, 210),
            "PRECIO_YUNTA": (1890, 175),
            "VALOR_JORNAL": (2127, 209)
        }

        distribucion_correcta = (
            distribucion_guardada == distribucion_esperada
        )

        print("Distribucion correcta:", distribucion_correcta)

        print("\n===== RESULTADO FINAL =====")

        if (
            total_guardado == 594
            and columnas_guardadas == 9
            and duplicados_guardados == 0
            and claves_nulas_guardadas == 0
            and contadores_iguales
            and distribucion_correcta
        ):
            kpi_costos_almacenado_validado = True
            print("ALMACENAMIENTO KPI COSTOS: CORRECTO")
        else:
            kpi_costos_almacenado_validado = False
            print("ATENCION: Revisar KPI de costos almacenado")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 81 FINALIZADO =====")

===== PASO 81: GUARDAR KPI COSTOS AGRICOLAS =====
PROGRESO: 0% - Validando KPI
Registros KPI: 594
Columnas: 9
Claves duplicadas: 0
Claves nulas: 0
Registros representados: 6292
Registros con valor: 6292

PROGRESO: 30% - Comprobando destino HDFS

PROGRESO: 50% - Guardando Parquet

PROGRESO: 75% - Verificando HDFS

===== VERIFICACION HDFS =====
Registros guardados: 594
Columnas guardadas: 9
Claves duplicadas: 0
Claves nulas: 0
Registros representados: 6292
Registros con valor: 6292
Contadores iguales al origen: True

===== RESUMEN POR TIPO DE COSTO =====
+----------------+-------------+---------+----------+
|tipo_costo      |unidad       |registros|grupos_kpi|
+----------------+-------------+---------+----------+
|ALQUILER_TRACTOR|NO_CONFIRMADA|2275     |210       |
|PRECIO_YUNTA    |NO_CONFIRMADA|1890     |175       |
|VALOR_JORNAL    |NO_CONFIRMADA|2127     |209       |
+----------------+-------------+---------+----------+

Distribucion correcta: True

===== RESULTADO FINAL =====
ALMAC

## KPI Gold: abastecimiento del mercado mayorista

Se construyen indicadores de abastecimiento mayorista
para alcachofa y espárrago a partir de la tabla de
hechos almacenada en Hadoop HDFS.

Los indicadores se agrupan por cultivo, mercado, año
y unidad comercial.

Se calcula el ingreso promedio reportado en toneladas,
el precio promedio reportado en la unidad comercial
original y el porcentaje de reportes que contienen
información de ingreso.

Los valores nulos se conservan y no se reemplazan por
cero. Los precios de diferentes unidades comerciales
no se mezclan.

Los indicadores describen las observaciones disponibles
en fechas específicas y no representan acumulados
anuales de abastecimiento.

In [70]:
from pyspark.sql import functions as F

print("===== PASO 82: KPI ABASTECIMIENTO MAYORISTA =====")
print("PROGRESO: 0% - Leyendo tablas Gold", flush=True)

# 1. Leer tablas desde HDFS.
fact_abast_kpi = spark.read.parquet(
    f"{BASE_GOLD}/fact_abastecimiento_mayorista"
)

dim_cultivo_abast_kpi = spark.read.parquet(
    f"{BASE_GOLD}/dim_cultivo"
).select("cultivo_id", "cultivo")

dim_tiempo_abast_kpi = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
).select("tiempo_id", "anio")

dim_mercado_abast_kpi = spark.read.parquet(
    f"{BASE_GOLD}/dim_mercado"
).select("mercado_id", "mercado")

print(
    "\nPROGRESO: 25% - Relacionando dimensiones",
    flush=True
)

# 2. Incorporar cultivo, año y mercado.
abast_base_kpi = (
    fact_abast_kpi
    .join(dim_cultivo_abast_kpi, "cultivo_id", "left")
    .join(dim_tiempo_abast_kpi, "tiempo_id", "left")
    .join(dim_mercado_abast_kpi, "mercado_id", "left")
)

print("\nPROGRESO: 50% - Calculando indicadores", flush=True)

# 3. Calcular indicadores sin sumar los ingresos
#    como si fueran acumulados anuales.
kpi_abastecimiento_anual = (
    abast_base_kpi
    .groupBy(
        "cultivo",
        "mercado",
        "anio",
        "unidad_medida",
        "equivalencia_kg"
    )
    .agg(
        F.avg("ingreso_hoy_t").alias(
            "ingreso_promedio_reportado_t"
        ),
        F.avg("precio_hoy_soles").alias(
            "precio_promedio_reportado_soles"
        ),
        F.count("*").alias(
            "reportes_total"
        ),
        F.count("ingreso_hoy_t").alias(
            "reportes_con_ingreso"
        ),
        F.count("precio_hoy_soles").alias(
            "reportes_con_precio"
        )
    )
    .withColumn(
        "cobertura_ingreso_pct",
        F.round(
            F.col("reportes_con_ingreso")
            / F.col("reportes_total") * 100,
            2
        )
    )
)

print("\n===== KPI ABASTECIMIENTO POR CULTIVO Y ANIO =====")

kpi_abastecimiento_anual.orderBy(
    "anio", "cultivo"
).show(20, truncate=False)

print("\nPROGRESO: 75% - Verificando cobertura", flush=True)

print("\n===== COBERTURA POR CULTIVO =====")

(
    kpi_abastecimiento_anual
    .groupBy("cultivo", "unidad_medida")
    .agg(
        F.sum("reportes_total").alias(
            "reportes_total"
        ),
        F.sum("reportes_con_ingreso").alias(
            "reportes_con_ingreso"
        ),
        F.sum("reportes_con_precio").alias(
            "reportes_con_precio"
        )
    )
    .orderBy("cultivo")
    .show(truncate=False)
)

print("\nPROGRESO: 85% - Validando integridad", flush=True)

registros_origen_abast = fact_abast_kpi.count()
registros_relacionados_abast = abast_base_kpi.count()
registros_kpi_abast = kpi_abastecimiento_anual.count()

relaciones_faltantes_abast = abast_base_kpi.filter(
    F.col("cultivo").isNull()
    | F.col("mercado").isNull()
    | F.col("anio").isNull()
).count()

duplicados_kpi_abast = (
    kpi_abastecimiento_anual
    .groupBy(
        "cultivo",
        "mercado",
        "anio",
        "unidad_medida",
        "equivalencia_kg"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

resumen_abast = kpi_abastecimiento_anual.agg(
    F.sum("reportes_total").alias("total"),
    F.sum("reportes_con_ingreso").alias("con_ingreso"),
    F.sum("reportes_con_precio").alias("con_precio")
).first()

nulos_ingreso_origen = fact_abast_kpi.filter(
    F.col("ingreso_hoy_t").isNull()
).count()

precios_disponibles_origen = fact_abast_kpi.filter(
    F.col("precio_hoy_soles").isNotNull()
).count()

errores_cobertura = kpi_abastecimiento_anual.filter(
    (F.col("cobertura_ingreso_pct") < 0)
    | (F.col("cobertura_ingreso_pct") > 100)
    | (F.col("reportes_con_ingreso") > F.col("reportes_total"))
).count()

print("\n===== VALIDACION =====")
print("Registros originales:", registros_origen_abast)
print("Registros relacionados:", registros_relacionados_abast)
print("Registros KPI:", registros_kpi_abast)
print("Relaciones faltantes:", relaciones_faltantes_abast)
print("KPI duplicados:", duplicados_kpi_abast)
print("Reportes representados:", resumen_abast["total"])
print("Reportes con ingreso:", resumen_abast["con_ingreso"])
print("Reportes con precio:", resumen_abast["con_precio"])
print("Ingresos nulos originales:", nulos_ingreso_origen)
print("Precios disponibles originales:", precios_disponibles_origen)
print("Errores de cobertura:", errores_cobertura)

print("\n===== RESULTADO FINAL =====")

if (
    registros_origen_abast == registros_relacionados_abast == 66
    and registros_kpi_abast > 0
    and relaciones_faltantes_abast == 0
    and duplicados_kpi_abast == 0
    and resumen_abast["total"] == 66
    and resumen_abast["con_ingreso"] == 38
    and nulos_ingreso_origen == 28
    and resumen_abast["con_precio"] == precios_disponibles_origen
    and errores_cobertura == 0
):
    kpi_abastecimiento_validada = True
    print("KPI ABASTECIMIENTO: CONSTRUIDOS CORRECTAMENTE")
else:
    kpi_abastecimiento_validada = False
    print("ATENCION: REVISAR KPI DE ABASTECIMIENTO")

print("\nPROGRESO: 100% - Calculo finalizado")
print("===== PASO 82 FINALIZADO =====")

===== PASO 82: KPI ABASTECIMIENTO MAYORISTA =====
PROGRESO: 0% - Leyendo tablas Gold

PROGRESO: 25% - Relacionando dimensiones

PROGRESO: 50% - Calculando indicadores

===== KPI ABASTECIMIENTO POR CULTIVO Y ANIO =====
+---------+---------+----+-------------+---------------+----------------------------+-------------------------------+--------------+--------------------+-------------------+---------------------+
|cultivo  |mercado  |anio|unidad_medida|equivalencia_kg|ingreso_promedio_reportado_t|precio_promedio_reportado_soles|reportes_total|reportes_con_ingreso|reportes_con_precio|cobertura_ingreso_pct|
+---------+---------+----+-------------+---------------+----------------------------+-------------------------------+--------------+--------------------+-------------------+---------------------+
|ALCACHOFA|GMML_LIMA|2024|DOCENA       |4.0            |4.4                         |20.770833333333332             |12            |5                   |12                 |41.67                

## Almacenamiento Gold: KPI anuales de abastecimiento mayorista

Se almacenan en Hadoop HDFS los indicadores de abastecimiento
mayorista por cultivo, mercado, año y unidad comercial.

La tabla contiene 6 registros agregados que representan
66 observaciones originales de alcachofa y espárrago.

Se conservan los promedios de ingreso reportado, los precios
promedio y los contadores de disponibilidad de información.

Los indicadores incluyen el porcentaje de cobertura de ingresos,
que permite identificar limitaciones de los reportes disponibles.

Los precios mantienen sus unidades comerciales originales:
DOCENA para alcachofa y PAQUETE para espárrago.

Los datos corresponden a reportes puntuales del mercado.
No representan volúmenes acumulados de abastecimiento anual.

La información se almacena en formato Parquet en la zona Gold.

In [71]:
from pyspark.sql import functions as F

print("===== PASO 83: GUARDAR KPI ABASTECIMIENTO =====")

if not globals().get("kpi_abastecimiento_validada", False):
    print("ERROR: Primero debes validar el Paso 82.")

else:
    ruta_kpi_abast = f"{BASE_GOLD}/kpi_abastecimiento_anual"

    claves_abast = [
        "cultivo",
        "mercado",
        "anio",
        "unidad_medida",
        "equivalencia_kg"
    ]

    print("PROGRESO: 0% - Validando KPI", flush=True)

    # 1. Validar la tabla original antes de guardarla.
    total_origen = kpi_abastecimiento_anual.count()
    columnas_origen = len(kpi_abastecimiento_anual.columns)

    duplicados_origen = (
        kpi_abastecimiento_anual
        .groupBy(*claves_abast)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    claves_nulas_origen = (
        kpi_abastecimiento_anual
        .filter(
            F.col("cultivo").isNull()
            | F.col("mercado").isNull()
            | F.col("anio").isNull()
            | F.col("unidad_medida").isNull()
            | F.col("equivalencia_kg").isNull()
        )
        .count()
    )

    resumen_origen = kpi_abastecimiento_anual.agg(
        F.sum("reportes_total").alias("total"),
        F.sum("reportes_con_ingreso").alias("con_ingreso"),
        F.sum("reportes_con_precio").alias("con_precio")
    ).first()

    print("Registros KPI:", total_origen)
    print("Columnas:", columnas_origen)
    print("Claves duplicadas:", duplicados_origen)
    print("Claves nulas:", claves_nulas_origen)
    print("Reportes representados:", resumen_origen["total"])
    print("Reportes con ingreso:", resumen_origen["con_ingreso"])
    print("Reportes con precio:", resumen_origen["con_precio"])

    origen_correcto = (
        total_origen == 6
        and columnas_origen == 11
        and duplicados_origen == 0
        and claves_nulas_origen == 0
        and resumen_origen["total"] == 66
        and resumen_origen["con_ingreso"] == 38
        and resumen_origen["con_precio"] == 66
    )

    if not origen_correcto:
        print("\nATENCION: Revisar los KPI antes de guardar.")

    else:
        print(
            "\nPROGRESO: 30% - Comprobando destino HDFS",
            flush=True
        )

        hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

        ruta_java = spark._jvm.org.apache.hadoop.fs.Path(
            ruta_kpi_abast
        )

        filesystem = ruta_java.getFileSystem(hadoop_conf)

        if filesystem.exists(ruta_java):
            print("La ruta ya existe. No se sobrescribira.")
        else:
            print(
                "\nPROGRESO: 50% - Guardando Parquet",
                flush=True
            )

            (
                kpi_abastecimiento_anual
                .coalesce(1)
                .write
                .mode("errorifexists")
                .parquet(ruta_kpi_abast)
            )

        print(
            "\nPROGRESO: 75% - Verificando HDFS",
            flush=True
        )

        # 2. Leer directamente la tabla almacenada.
        kpi_abast_guardado = spark.read.parquet(
            ruta_kpi_abast
        )

        total_guardado = kpi_abast_guardado.count()
        columnas_guardadas = len(kpi_abast_guardado.columns)

        duplicados_guardados = (
            kpi_abast_guardado
            .groupBy(*claves_abast)
            .count()
            .filter(F.col("count") > 1)
            .count()
        )

        claves_nulas_guardadas = (
            kpi_abast_guardado
            .filter(
                F.col("cultivo").isNull()
                | F.col("mercado").isNull()
                | F.col("anio").isNull()
                | F.col("unidad_medida").isNull()
                | F.col("equivalencia_kg").isNull()
            )
            .count()
        )

        resumen_guardado = kpi_abast_guardado.agg(
            F.sum("reportes_total").alias("total"),
            F.sum("reportes_con_ingreso").alias("con_ingreso"),
            F.sum("reportes_con_precio").alias("con_precio")
        ).first()

        print("\n===== VERIFICACION HDFS =====")
        print("Registros guardados:", total_guardado)
        print("Columnas guardadas:", columnas_guardadas)
        print("Claves duplicadas:", duplicados_guardados)
        print("Claves nulas:", claves_nulas_guardadas)
        print("Reportes representados:", resumen_guardado["total"])
        print("Reportes con ingreso:", resumen_guardado["con_ingreso"])
        print("Reportes con precio:", resumen_guardado["con_precio"])

        contadores_iguales = (
            resumen_origen.asDict()
            == resumen_guardado.asDict()
        )

        print("Contadores iguales al origen:", contadores_iguales)

        # 3. Comprobar cobertura y unidades comerciales.
        errores_cobertura = (
            kpi_abast_guardado
            .filter(
                (F.col("cobertura_ingreso_pct") < 0)
                | (F.col("cobertura_ingreso_pct") > 100)
                | (
                    F.abs(
                        F.col("cobertura_ingreso_pct")
                        - F.round(
                            F.col("reportes_con_ingreso")
                            / F.col("reportes_total") * 100,
                            2
                        )
                    ) > 0.01
                )
            )
            .count()
        )

        errores_unidad = (
            kpi_abast_guardado
            .filter(
                (
                    (F.col("cultivo") == "ALCACHOFA")
                    & (
                        (F.col("unidad_medida") != "DOCENA")
                        | (F.col("equivalencia_kg") != 4.0)
                    )
                )
                |
                (
                    (F.col("cultivo") == "ESPARRAGO")
                    & (
                        (F.col("unidad_medida") != "PAQUETE")
                        | (F.col("equivalencia_kg") != 0.5)
                    )
                )
                |
                ~F.col("cultivo").isin("ALCACHOFA", "ESPARRAGO")
            )
            .count()
        )

        print("Errores de cobertura:", errores_cobertura)
        print("Errores de unidad:", errores_unidad)

        print("\n===== KPI ALMACENADOS =====")

        (
            kpi_abast_guardado
            .orderBy("anio", "cultivo")
            .show(10, truncate=False)
        )

        print("\n===== RESULTADO FINAL =====")

        if (
            total_guardado == 6
            and columnas_guardadas == 11
            and duplicados_guardados == 0
            and claves_nulas_guardadas == 0
            and contadores_iguales
            and errores_cobertura == 0
            and errores_unidad == 0
        ):
            kpi_abastecimiento_almacenado_validado = True
            print("ALMACENAMIENTO KPI ABASTECIMIENTO: CORRECTO")
        else:
            kpi_abastecimiento_almacenado_validado = False
            print("ATENCION: Revisar KPI de abastecimiento")

        print(
            "PROGRESO: 100% - Verificacion finalizada",
            flush=True
        )

print("===== PASO 83 FINALIZADO =====")

===== PASO 83: GUARDAR KPI ABASTECIMIENTO =====
PROGRESO: 0% - Validando KPI
Registros KPI: 6
Columnas: 11
Claves duplicadas: 0
Claves nulas: 0
Reportes representados: 66
Reportes con ingreso: 38
Reportes con precio: 66

PROGRESO: 30% - Comprobando destino HDFS

PROGRESO: 50% - Guardando Parquet

PROGRESO: 75% - Verificando HDFS

===== VERIFICACION HDFS =====
Registros guardados: 6
Columnas guardadas: 11
Claves duplicadas: 0
Claves nulas: 0
Reportes representados: 66
Reportes con ingreso: 38
Reportes con precio: 66
Contadores iguales al origen: True
Errores de cobertura: 0
Errores de unidad: 0

===== KPI ALMACENADOS =====
+---------+---------+----+-------------+---------------+----------------------------+-------------------------------+--------------+--------------------+-------------------+---------------------+
|cultivo  |mercado  |anio|unidad_medida|equivalencia_kg|ingreso_promedio_reportado_t|precio_promedio_reportado_soles|reportes_total|reportes_con_ingreso|reportes_con_precio|c

## Auditoría final de los componentes de la zona Gold

Se realiza una auditoría general de las 26 tablas
almacenadas en la zona Gold del Data Lake agrocomercial.

La auditoría comprende siete tablas analíticas,
seis dimensiones, siete tablas de hechos y seis
tablas de indicadores KPI.

Para cada componente se verifica su existencia en
Hadoop HDFS, la lectura en formato Parquet y la
cantidad de registros almacenados.

Los resultados se comparan con las cantidades
verificadas durante las etapas de construcción.

Esta auditoría permite identificar tablas faltantes,
incompletas o con cantidades inesperadas antes de
generar los entregables finales del proyecto.

No se modifican ni sobrescriben archivos existentes.

In [72]:
print("===== PASO 84: AUDITORIA GENERAL GOLD =====")
print("PROGRESO: 0% - Preparando inventario", flush=True)

# Inventario esperado según los pasos ya validados.
# Formato: (nombre, categoria, registros_esperados)

inventario_gold = [
    # Tablas analíticas
    ("produccion_mensual", "ANALITICA", 561),
    ("exportaciones_regionales", "ANALITICA", 720),
    ("exportaciones_danper", "ANALITICA", 495),
    ("agroindustria_mensual", "ANALITICA", 3134),
    ("costos_mensuales", "ANALITICA", 6292),
    ("abastecimiento_mayorista", "ANALITICA", 66),
    ("precios_mercado_productores", "ANALITICA", 756),

    # Dimensiones
    ("dim_tiempo", "DIMENSION", 201),
    ("dim_cultivo", "DIMENSION", 5),
    ("dim_ubicacion_produccion", "DIMENSION", 22),
    ("dim_region_costos", "DIMENSION", 25),
    ("dim_pais_exportacion", "DIMENSION", 35),
    ("dim_mercado", "DIMENSION", 2),

    # Tablas de hechos
    ("fact_produccion_mensual", "HECHOS", 561),
    ("fact_exportaciones_regionales", "HECHOS", 720),
    ("fact_exportaciones_danper", "HECHOS", 495),
    ("fact_agroindustria_mensual", "HECHOS", 3134),
    ("fact_costos_mensuales", "HECHOS", 6292),
    ("fact_abastecimiento_mayorista", "HECHOS", 66),
    ("fact_precios_mercado_productores", "HECHOS", 756),

    # Indicadores KPI
    ("kpi_produccion_anual", "KPI", 6),
    ("kpi_exportaciones_regionales_anual", "KPI", 60),
    ("kpi_exportaciones_danper_anual", "KPI", 81),
    ("kpi_agroindustria_anual", "KPI", 262),
    ("kpi_costos_anual", "KPI", 594),
    ("kpi_abastecimiento_anual", "KPI", 6)
]

print("Componentes esperados:", len(inventario_gold))

# Acceso de lectura al sistema HDFS.
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
Path = spark._jvm.org.apache.hadoop.fs.Path

resultados_auditoria = []

print(
    "\nPROGRESO: 10% - Verificando rutas y registros",
    flush=True
)

# Revisar componente por componente.
for indice, (nombre, categoria, esperado) in enumerate(
    inventario_gold, start=1
):
    ruta = f"{BASE_GOLD}/{nombre}"
    ruta_java = Path(ruta)
    filesystem = ruta_java.getFileSystem(hadoop_conf)

    existe = filesystem.exists(ruta_java)

    registros = None
    columnas = None

    if not existe:
        estado = "NO EXISTE"

    else:
        try:
            df_auditoria = spark.read.parquet(ruta)

            registros = df_auditoria.count()
            columnas = len(df_auditoria.columns)

            if registros == esperado:
                estado = "CORRECTO"
            else:
                estado = "REVISAR CANTIDAD"

        except Exception as error:
            estado = "ERROR LECTURA"
            print(
                f"Detalle del error en {nombre}: "
                f"{str(error)[:200]}"
            )

    resultados_auditoria.append({
        "nombre": nombre,
        "categoria": categoria,
        "esperados": esperado,
        "encontrados": registros,
        "columnas": columnas,
        "estado": estado
    })

    porcentaje = int(
        10 + indice / len(inventario_gold) * 80
    )

    print(
        f"[{indice:02d}/{len(inventario_gold)}] "
        f"{nombre}: {estado} "
        f"({registros}/{esperado}) "
        f"- {porcentaje}%",
        flush=True
    )

print("\nPROGRESO: 90% - Generando resumen", flush=True)

# Crear DataFrame con el inventario auditado.
auditoria_gold_df = spark.createDataFrame(
    resultados_auditoria
)

print("\n===== INVENTARIO GOLD =====")

auditoria_gold_df.select(
    "categoria",
    "nombre",
    "esperados",
    "encontrados",
    "columnas",
    "estado"
).orderBy(
    "categoria", "nombre"
).show(30, truncate=False)

print("\n===== RESUMEN POR CATEGORIA =====")

auditoria_gold_df.groupBy(
    "categoria", "estado"
).count().orderBy(
    "categoria", "estado"
).show(truncate=False)

# Validación final
componentes_correctos = sum(
    1
    for resultado in resultados_auditoria
    if resultado["estado"] == "CORRECTO"
)

componentes_con_error = len(
    inventario_gold
) - componentes_correctos

print("\n===== VALIDACION FINAL =====")
print("Componentes esperados:", len(inventario_gold))
print("Componentes correctos:", componentes_correctos)
print("Componentes con problemas:", componentes_con_error)

auditoria_gold_validada = (
    len(inventario_gold) == 26
    and componentes_correctos == 26
    and componentes_con_error == 0
)

print("\n===== RESULTADO FINAL =====")

if auditoria_gold_validada:
    print("AUDITORIA GENERAL GOLD: CORRECTA")
else:
    print("ATENCION: REVISAR COMPONENTES GOLD")

print("\nPROGRESO: 100% - Auditoria finalizada")
print("===== PASO 84 FINALIZADO =====")

===== PASO 84: AUDITORIA GENERAL GOLD =====
PROGRESO: 0% - Preparando inventario
Componentes esperados: 26

PROGRESO: 10% - Verificando rutas y registros
[01/26] produccion_mensual: CORRECTO (561/561) - 13%
[02/26] exportaciones_regionales: CORRECTO (720/720) - 16%
[03/26] exportaciones_danper: CORRECTO (495/495) - 19%
[04/26] agroindustria_mensual: CORRECTO (3134/3134) - 22%
[05/26] costos_mensuales: CORRECTO (6292/6292) - 25%
[06/26] abastecimiento_mayorista: CORRECTO (66/66) - 28%
[07/26] precios_mercado_productores: CORRECTO (756/756) - 31%
[08/26] dim_tiempo: CORRECTO (201/201) - 34%
[09/26] dim_cultivo: CORRECTO (5/5) - 37%
[10/26] dim_ubicacion_produccion: CORRECTO (22/22) - 40%
[11/26] dim_region_costos: CORRECTO (25/25) - 43%
[12/26] dim_pais_exportacion: CORRECTO (35/35) - 46%
[13/26] dim_mercado: CORRECTO (2/2) - 50%
[14/26] fact_produccion_mensual: CORRECTO (561/561) - 53%
[15/26] fact_exportaciones_regionales: CORRECTO (720/720) - 56%
[16/26] fact_exportaciones_danper: COR

+---------+----------------------------------+---------+-----------+--------+--------+
|categoria|nombre                            |esperados|encontrados|columnas|estado  |
+---------+----------------------------------+---------+-----------+--------+--------+
|ANALITICA|abastecimiento_mayorista          |66       |66         |16      |CORRECTO|
|ANALITICA|agroindustria_mensual             |3134     |3134       |7       |CORRECTO|
|ANALITICA|costos_mensuales                  |6292     |6292       |7       |CORRECTO|
|ANALITICA|exportaciones_danper              |495      |495        |6       |CORRECTO|
|ANALITICA|exportaciones_regionales          |720      |720        |7       |CORRECTO|
|ANALITICA|precios_mercado_productores       |756      |756        |11      |CORRECTO|
|ANALITICA|produccion_mensual                |561      |561        |13      |CORRECTO|
|DIMENSION|dim_cultivo                       |5        |5          |6       |CORRECTO|
|DIMENSION|dim_mercado                     

## Diagnóstico final de calidad de datos Gold

Se realiza un diagnóstico de calidad sobre las tablas
almacenadas en la zona Gold del Data Lake agrocomercial.

El análisis identifica valores nulos, unidades no confirmadas,
registros iguales a cero y limitaciones temporales que
afectan la interpretación de los indicadores.

Se examinan especialmente la producción agrícola,
las exportaciones regionales, los costos agrícolas,
la agroindustria y el abastecimiento mayorista.

Los datos faltantes no se reemplazan por cero y los valores
observados se conservan sin modificaciones.

Los resultados permitirán documentar restricciones de uso
y establecer recomendaciones para el dashboard y los
modelos analíticos posteriores.

In [73]:
from pyspark.sql import functions as F

print("===== PASO 85: DIAGNOSTICO DE CALIDAD GOLD =====")
print("PROGRESO: 0% - Leyendo componentes", flush=True)

# 1. Leer tablas almacenadas en HDFS.
prod_calidad = spark.read.parquet(
    f"{BASE_GOLD}/fact_produccion_mensual"
)

regional_calidad = spark.read.parquet(
    f"{BASE_GOLD}/fact_exportaciones_regionales"
)

danper_calidad = spark.read.parquet(
    f"{BASE_GOLD}/fact_exportaciones_danper"
)

agro_calidad = spark.read.parquet(
    f"{BASE_GOLD}/fact_agroindustria_mensual"
)

costos_calidad = spark.read.parquet(
    f"{BASE_GOLD}/fact_costos_mensuales"
)

abast_calidad = spark.read.parquet(
    f"{BASE_GOLD}/fact_abastecimiento_mayorista"
)

tiempo_calidad = spark.read.parquet(
    f"{BASE_GOLD}/dim_tiempo"
).select("tiempo_id", "anio")

print(
    "\nPROGRESO: 20% - Analizando produccion",
    flush=True
)

# 2. Produccion y cobertura temporal.
prod_anual_calidad = prod_calidad.join(
    tiempo_calidad,
    "tiempo_id",
    "left"
)

resumen_prod_2024 = (
    prod_anual_calidad
    .filter(F.col("anio") == 2024)
    .agg(
        F.count("*").alias("total"),
        F.sum(
            F.when(F.col("produccion_t").isNull(), 1)
            .otherwise(0)
        ).alias("sin_produccion"),
        F.sum(
            F.when(F.col("precio_chacra_s_kg").isNull(), 1)
            .otherwise(0)
        ).alias("sin_precio")
    )
    .first()
)

rendimiento_nulo = prod_calidad.filter(
    F.col("rendimiento_kg_ha").isNull()
).count()

print(
    "\nPROGRESO: 40% - Analizando exportaciones",
    flush=True
)

# 3. Exportaciones y disponibilidad de volumen.
volumen_regional_nulo = regional_calidad.filter(
    F.col("volumen_exportado_t").isNull()
).count()

danper_2026 = (
    danper_calidad
    .join(tiempo_calidad, "tiempo_id", "left")
    .filter(F.col("anio") == 2026)
    .count()
)

print(
    "\nPROGRESO: 60% - Analizando costos y agroindustria",
    flush=True
)

# 4. Unidades sin confirmar y cantidades iguales a cero.
unidades_no_confirmadas = costos_calidad.filter(
    F.col("unidad") == "NO_CONFIRMADA"
).count()

cantidades_cero_agro = agro_calidad.filter(
    F.col("cantidad_t") == 0
).count()

print(
    "\nPROGRESO: 80% - Analizando abastecimiento",
    flush=True
)

# 5. Ingresos faltantes.
ingresos_nulos_abast = abast_calidad.filter(
    F.col("ingreso_hoy_t").isNull()
).count()

# 6. Preparar el reporte de calidad.
hallazgos_calidad = [
    (
        "Produccion",
        "Registros del año 2024",
        int(resumen_prod_2024["total"]),
        "Cobertura temporal"
    ),
    (
        "Produccion",
        "Produccion no disponible en 2024",
        int(resumen_prod_2024["sin_produccion"]),
        "Dato faltante"
    ),
    (
        "Produccion",
        "Precio en chacra no disponible en 2024",
        int(resumen_prod_2024["sin_precio"]),
        "Dato faltante"
    ),
    (
        "Produccion",
        "Rendimiento no disponible",
        int(rendimiento_nulo),
        "Dato faltante"
    ),
    (
        "Exportaciones regionales",
        "Volumen exportado no disponible",
        int(volumen_regional_nulo),
        "Dato faltante"
    ),
    (
        "Exportaciones Danper",
        "Registros del año 2026",
        int(danper_2026),
        "Periodo parcial"
    ),
    (
        "Costos agricolas",
        "Unidades no confirmadas",
        int(unidades_no_confirmadas),
        "Unidad no confirmada"
    ),
    (
        "Agroindustria",
        "Cantidades iguales a cero",
        int(cantidades_cero_agro),
        "Valor cero reportado"
    ),
    (
        "Abastecimiento",
        "Ingresos no disponibles",
        int(ingresos_nulos_abast),
        "Dato faltante"
    )
]

calidad_gold_df = spark.createDataFrame(
    hallazgos_calidad,
    [
        "fuente",
        "hallazgo",
        "registros",
        "clasificacion"
    ]
)

print("\n===== REPORTE DE CALIDAD GOLD =====")

calidad_gold_df.show(20, truncate=False)

print("\n===== VALIDACION DE HALLAZGOS =====")

valores_obtenidos = {
    "produccion_2024": int(resumen_prod_2024["total"]),
    "produccion_nula_2024": int(
        resumen_prod_2024["sin_produccion"]
    ),
    "precio_nulo_2024": int(
        resumen_prod_2024["sin_precio"]
    ),
    "rendimiento_nulo": rendimiento_nulo,
    "volumen_regional_nulo": volumen_regional_nulo,
    "danper_2026": danper_2026,
    "costos_unidad_no_confirmada": unidades_no_confirmadas,
    "agro_cantidad_cero": cantidades_cero_agro,
    "abastecimiento_ingreso_nulo": ingresos_nulos_abast
}

valores_esperados = {
    "produccion_2024": 165,
    "produccion_nula_2024": 165,
    "precio_nulo_2024": 165,
    "rendimiento_nulo": 561,
    "volumen_regional_nulo": 38,
    "danper_2026": 68,
    "costos_unidad_no_confirmada": 6292,
    "agro_cantidad_cero": 978,
    "abastecimiento_ingreso_nulo": 28
}

errores_calidad = 0

for indicador, esperado in valores_esperados.items():
    obtenido = valores_obtenidos[indicador]

    correcto = obtenido == esperado

    if not correcto:
        errores_calidad += 1

    print(
        f"{indicador}: {obtenido} "
        f"(esperado: {esperado}) - "
        f"{'CORRECTO' if correcto else 'REVISAR'}"
    )

print("\n===== RESULTADO FINAL =====")

calidad_gold_validada = errores_calidad == 0

if calidad_gold_validada:
    print("DIAGNOSTICO DE CALIDAD GOLD: CORRECTO")
    print("Hallazgos identificados:", len(hallazgos_calidad))
    print("Verificaciones inconsistentes:", errores_calidad)
else:
    print("ATENCION: REVISAR DIAGNOSTICO DE CALIDAD")
    print("Verificaciones inconsistentes:", errores_calidad)

print("\nPROGRESO: 100% - Diagnostico finalizado")
print("===== PASO 85 FINALIZADO =====")

===== PASO 85: DIAGNOSTICO DE CALIDAD GOLD =====
PROGRESO: 0% - Leyendo componentes

PROGRESO: 20% - Analizando produccion

PROGRESO: 40% - Analizando exportaciones

PROGRESO: 60% - Analizando costos y agroindustria

PROGRESO: 80% - Analizando abastecimiento

===== REPORTE DE CALIDAD GOLD =====
+------------------------+--------------------------------------+---------+--------------------+
|fuente                  |hallazgo                              |registros|clasificacion       |
+------------------------+--------------------------------------+---------+--------------------+
|Produccion              |Registros del año 2024                |165      |Cobertura temporal  |
|Produccion              |Produccion no disponible en 2024      |165      |Dato faltante       |
|Produccion              |Precio en chacra no disponible en 2024|165      |Dato faltante       |
|Produccion              |Rendimiento no disponible             |561      |Dato faltante       |
|Exportaciones regionales

## Documentación del diagnóstico de calidad de datos Gold

Se genera un informe técnico de calidad de datos para el
proyecto Big Data y Analítica Agrocomercial de Danper.

El documento registra los principales hallazgos identificados
en las fuentes de producción, exportaciones, costos,
agroindustria y abastecimiento mayorista.

Para cada hallazgo se describe su impacto sobre el análisis
y las recomendaciones para utilizar correctamente los datos.

También se incorpora el resultado de la auditoría de los
26 componentes Gold almacenados en Hadoop HDFS.

El informe se guarda en la carpeta docs del repositorio
GitHub, sin modificar los datos del Data Lake.

In [74]:
from pathlib import Path
from datetime import datetime

print("===== PASO 86: DOCUMENTAR CALIDAD GOLD =====")
print("PROGRESO: 0% - Comprobando validaciones", flush=True)

# 1. Comprobar que las auditorías se hayan completado.
if not (
    globals().get("auditoria_gold_validada", False)
    and globals().get("calidad_gold_validada", False)
):
    print("ERROR: Debes completar correctamente los Pasos 84 y 85.")

else:
    print(
        "\nPROGRESO: 20% - Localizando repositorio",
        flush=True
    )

    # 2. Buscar la raíz del repositorio sin depender
    #    del directorio desde el que se abrió JupyterLab.
    ruta_actual = Path.cwd().resolve()

    candidatos = [ruta_actual] + list(ruta_actual.parents)

    repo = next(
        (
            carpeta
            for carpeta in candidatos
            if (
                carpeta / ".git"
            ).exists()
        ),
        None
    )

    if repo is None:
        print("ERROR: No se encontro el repositorio Git.")
        print("Directorio actual:", ruta_actual)
        print("No se creo ningun archivo.")

    else:
        print("Repositorio:", repo)

        # 3. Preparar la carpeta de documentación.
        carpeta_docs = repo / "docs"
        carpeta_docs.mkdir(parents=True, exist_ok=True)

        destino = carpeta_docs / "reporte_calidad_gold.md"

        print(
            "\nPROGRESO: 40% - Preparando informe",
            flush=True
        )

        hallazgos = [
            (
                "Producción agrícola",
                "165 registros de 2024 sin producción reportada",
                "No es posible calcular la producción de 2024 con esta fuente.",
                "Mostrar el valor como no disponible; evitar comparaciones "
                "anuales de producción con 2025."
            ),
            (
                "Precio en chacra",
                "165 registros de 2024 sin precio reportado",
                "No puede calcularse el precio promedio de 2024.",
                "Mantener los valores nulos y mostrar cobertura."
            ),
            (
                "Rendimiento agrícola",
                "561 registros sin rendimiento disponible",
                "No se puede evaluar el rendimiento con esta columna.",
                "Excluir este indicador hasta disponer de datos válidos."
            ),
            (
                "Exportaciones regionales",
                "38 registros sin volumen exportado",
                "Algunos totales de volumen tienen cobertura incompleta.",
                "Presentar los contadores de registros disponibles."
            ),
            (
                "Exportaciones Danper",
                "68 registros correspondientes a 2026",
                "La información de 2026 representa un periodo parcial.",
                "No comparar directamente con años completos."
            ),
            (
                "Costos agrícolas",
                "6292 registros con unidad NO_CONFIRMADA",
                "No se puede asegurar la unidad de los valores.",
                "Mantener separados los tipos de costo y no atribuir "
                "unidades no verificadas."
            ),
            (
                "Agroindustria",
                "978 registros con cantidad igual a cero",
                "Los ceros pueden afectar agregaciones y promedios.",
                "Conservar los ceros originales y analizar cada proceso "
                "por separado."
            ),
            (
                "Abastecimiento mayorista",
                "28 registros sin ingreso de hoy",
                "Los promedios de ingreso se basan en una cobertura parcial.",
                "Mostrar el porcentaje de cobertura junto al indicador."
            )
        ]

        fecha_informe = datetime.now().strftime("%Y-%m-%d %H:%M")

        lineas = [
            "# Informe de calidad de datos - Zona Gold",
            "",
            "**Proyecto:** Big Data y Analítica Agrocomercial - Danper",
            "",
            f"**Fecha de generación:** {fecha_informe}",
            "",
            "**Entorno:** Hadoop HDFS y Apache Spark (on-premise)",
            "",
            "## 1. Objetivo",
            "",
            "Documentar la calidad, cobertura y principales limitaciones "
            "de las fuentes procesadas en la zona Gold del Data Lake.",
            "",
            "## 2. Auditoría del almacenamiento",
            "",
            "La auditoría verificó la existencia, lectura en formato Parquet "
            "y cantidad esperada de registros de 26 componentes Gold:",
            "",
            "- 7 tablas analíticas.",
            "- 6 dimensiones.",
            "- 7 tablas de hechos.",
            "- 6 tablas de indicadores KPI.",
            "",
            "**Resultado:** 26 componentes correctos y 0 componentes "
            "con problemas en las comprobaciones realizadas.",
            "",
            "Esta auditoría no sustituye las pruebas de calidad semántica "
            "o la validación externa de las fuentes.",
            "",
            "## 3. Hallazgos de calidad",
            ""
        ]

        for numero, (fuente, hallazgo, impacto, recomendacion) in enumerate(
            hallazgos, start=1
        ):
            lineas.extend([
                f"### 3.{numero}. {fuente}",
                "",
                f"**Hallazgo:** {hallazgo}.",
                "",
                f"**Impacto:** {impacto}",
                "",
                f"**Recomendación:** {recomendacion}",
                ""
            ])

        lineas.extend([
            "## 4. Consideraciones para los indicadores",
            "",
            "- Los valores nulos no representan necesariamente cero.",
            "- Los KPI se calculan sobre los registros disponibles.",
            "- Los precios de diferentes unidades comerciales no "
            "se comparan directamente.",
            "- Las exportaciones regionales y las exportaciones propias "
            "de Danper se analizan por separado.",
            "- En agroindustria, INGRESO, PRODUCCION, UTILIZACION "
            "y VENTA representan procesos diferentes.",
            "- Los ingresos mayoristas son reportes puntuales, "
            "no acumulados anuales.",
            "- Los datos de 2026 deben tratarse como parciales.",
            "",
            "## 5. Conclusión",
            "",
            "Los componentes Gold se encuentran almacenados y fueron "
            "verificados con respecto a las cantidades esperadas. "
            "Sin embargo, existen limitaciones de completitud, "
            "cobertura temporal y unidades de medida que deben "
            "considerarse en el dashboard y los análisis posteriores.",
            "",
            "El procesamiento conserva los datos originales y "
            "documenta las restricciones para evitar interpretaciones "
            "incorrectas.",
            ""
        ])

        contenido = "\n".join(lineas)

        print(
            "\nPROGRESO: 70% - Guardando documentacion",
            flush=True
        )

        # 4. Crear el archivo solo si no existe.
        #    El modo 'x' evita sobrescrituras.
        if destino.exists():
            print("ATENCION: El informe ya existe.")
            print("No se sobrescribira:", destino)

        else:
            with destino.open("x", encoding="utf-8") as archivo:
                archivo.write(contenido)

            print("Informe creado:", destino)

        print(
            "\nPROGRESO: 90% - Verificando archivo",
            flush=True
        )

        if destino.is_file():
            texto_guardado = destino.read_text(encoding="utf-8")

            print("Archivo encontrado: SI")
            print("Tamano (caracteres):", len(texto_guardado))
            print("Ruta:", destino)

            secciones_presentes = all(
                seccion in texto_guardado
                for seccion in [
                    "# Informe de calidad de datos",
                    "## 2. Auditoría del almacenamiento",
                    "## 3. Hallazgos de calidad",
                    "## 4. Consideraciones para los indicadores",
                    "## 5. Conclusión"
                ]
            )

            print("Secciones principales:", secciones_presentes)

            if secciones_presentes:
                informe_calidad_documentado = True
                print("\nDOCUMENTACION CALIDAD GOLD: CORRECTA")
            else:
                informe_calidad_documentado = False
                print("\nATENCION: Revisar documento existente")

        else:
            informe_calidad_documentado = False
            print("ERROR: No se encontro el archivo generado.")

        print("\nPROGRESO: 100% - Documentacion finalizada")

print("===== PASO 86 FINALIZADO =====")

===== PASO 86: DOCUMENTAR CALIDAD GOLD =====
PROGRESO: 0% - Comprobando validaciones

PROGRESO: 20% - Localizando repositorio
Repositorio: /home/danna/G07-BigData-AnaliticaAgrocomercial-Danper

PROGRESO: 40% - Preparando informe

PROGRESO: 70% - Guardando documentacion
Informe creado: /home/danna/G07-BigData-AnaliticaAgrocomercial-Danper/docs/reporte_calidad_gold.md

PROGRESO: 90% - Verificando archivo
Archivo encontrado: SI
Tamano (caracteres): 3648
Ruta: /home/danna/G07-BigData-AnaliticaAgrocomercial-Danper/docs/reporte_calidad_gold.md
Secciones principales: True

DOCUMENTACION CALIDAD GOLD: CORRECTA

PROGRESO: 100% - Documentacion finalizada
===== PASO 86 FINALIZADO =====


## Preparación del script Python del procesamiento Gold

Se genera una versión preliminar del código del notebook
de procesamiento Gold para su posterior organización
como script Python reproducible.

El objetivo es reunir las transformaciones, dimensiones,
tablas de hechos, KPI y validaciones desarrolladas
durante la implementación.

Esta exportación permite identificar dependencias,
celdas auxiliares y operaciones que requieren ajustes
antes de ejecutar el procesamiento de forma automatizada.

La versión preliminar no se ejecuta automáticamente
y no modifica los archivos existentes en HDFS.

Posteriormente se organizará el flujo de ejecución
y se incorporarán controles para evitar sobrescrituras.

In [75]:
from pathlib import Path
import json
import ast

print("===== PASO 87: PREPARAR SCRIPT GOLD =====")
print("PROGRESO: 0% - Localizando repositorio", flush=True)

# 1. Encontrar la raíz del repositorio.
actual = Path.cwd().resolve()

repo_gold = next(
    (
        carpeta
        for carpeta in [actual] + list(actual.parents)
        if (carpeta / ".git").exists()
    ),
    None
)

if repo_gold is None:
    print("ERROR: No se encontro el repositorio Git.")

else:
    notebook_gold = (
        repo_gold / "notebooks" / "procesamiento_gold.ipynb"
    )

    print("Repositorio:", repo_gold)
    print("Notebook:", notebook_gold)

    if not notebook_gold.is_file():
        print("ERROR: No se encontro el notebook.")
        print("Comprueba que guardaste el archivo con Ctrl + S.")

    else:
        print(
            "\nPROGRESO: 25% - Leyendo notebook",
            flush=True
        )

        # 2. Leer directamente el formato JSON de Jupyter.
        with notebook_gold.open(
            "r", encoding="utf-8"
        ) as archivo:
            notebook_json = json.load(archivo)

        celdas = notebook_json.get("cells", [])

        celdas_codigo = [
            celda
            for celda in celdas
            if celda.get("cell_type") == "code"
            and "".join(
                celda.get("source", [])
            ).strip()
        ]

        celdas_markdown = [
            celda
            for celda in celdas
            if celda.get("cell_type") == "markdown"
        ]

        print("Celdas totales:", len(celdas))
        print("Celdas de codigo:", len(celdas_codigo))
        print("Celdas Markdown:", len(celdas_markdown))

        print(
            "\nPROGRESO: 50% - Preparando exportacion",
            flush=True
        )

        # 3. Crear carpeta scripts si no existe.
        carpeta_scripts = repo_gold / "scripts"
        carpeta_scripts.mkdir(
            parents=True,
            exist_ok=True
        )

        destino_script = (
            carpeta_scripts
            / "procesamiento_gold_borrador.py"
        )

        encabezado = '''# -*- coding: utf-8 -*-
"""
Proyecto: Big Data y Analitica Agrocomercial - Danper

Archivo: procesamiento_gold_borrador.py

Origen:
    notebooks/procesamiento_gold.ipynb

IMPORTANTE:
    Esta es una exportacion preliminar del notebook.

    NO EJECUTAR DIRECTAMENTE.

    Las celdas deben revisarse y organizarse antes de
    convertir este archivo en un ETL reproducible.

    El notebook contiene pasos de exploracion,
    validacion, almacenamiento y recuperacion.

    Este archivo no ha sido probado como un proceso
    independiente.

    La exportacion no modifica HDFS.
"""
'''

        fragmentos = [encabezado]

        for numero, celda in enumerate(
            celdas_codigo, start=1
        ):
            codigo = "".join(
                celda.get("source", [])
            )

            fragmentos.append(
                "\n\n"
                + "# " + "=" * 65 + "\n"
                + f"# CELDA DE CODIGO {numero}\n"
                + "# " + "=" * 65 + "\n\n"
                + codigo
                + "\n"
            )

        contenido_script = "".join(fragmentos)

        # 4. Detectar elementos que necesitan revisión.
        lineas_especiales = [
            linea.strip()
            for linea in contenido_script.splitlines()
            if linea.lstrip().startswith(
                ("%", "!", "?")
            )
        ]

        try:
            ast.parse(contenido_script)
            sintaxis_python_valida = True
        except SyntaxError:
            sintaxis_python_valida = False

        print(
            "\nPROGRESO: 75% - Guardando borrador",
            flush=True
        )

        # 5. No sobrescribir un script anterior.
        if destino_script.exists():
            print("El borrador ya existe.")
            print("No se sobrescribira:", destino_script)

        else:
            with destino_script.open(
                "x", encoding="utf-8"
            ) as archivo:
                archivo.write(contenido_script)

            print("Borrador creado:", destino_script)

        print(
            "\nPROGRESO: 90% - Verificando archivo",
            flush=True
        )

        existe_script = destino_script.is_file()

        print("\n===== VERIFICACION =====")
        print("Archivo encontrado:", existe_script)

        if existe_script:
            contenido_guardado = (
                destino_script.read_text(
                    encoding="utf-8"
                )
            )

            print(
                "Lineas exportadas:",
                len(contenido_guardado.splitlines())
            )

            print(
                "Tamano del archivo (bytes):",
                destino_script.stat().st_size
            )

        print(
            "Sintaxis del codigo generado valida:",
            sintaxis_python_valida
        )

        print(
            "Lineas especiales detectadas:",
            len(lineas_especiales)
        )

        if lineas_especiales:
            print("\n===== ELEMENTOS A REVISAR =====")

            for linea in lineas_especiales[:10]:
                print(linea[:120])

        print("\n===== RESULTADO FINAL =====")

        if existe_script:
            print("BORRADOR SCRIPT GOLD: GENERADO")
            print(
                "Pendiente: organizar, depurar "
                "y probar su reproducibilidad."
            )
        else:
            print("ATENCION: Revisar generacion del script.")

        print(
            "\nPROGRESO: 100% - Exportacion finalizada"
        )

print("===== PASO 87 FINALIZADO =====")

===== PASO 87: PREPARAR SCRIPT GOLD =====
PROGRESO: 0% - Localizando repositorio
Repositorio: /home/danna/G07-BigData-AnaliticaAgrocomercial-Danper
Notebook: /home/danna/G07-BigData-AnaliticaAgrocomercial-Danper/notebooks/procesamiento_gold.ipynb

PROGRESO: 25% - Leyendo notebook
Celdas totales: 133
Celdas de codigo: 71
Celdas Markdown: 61

PROGRESO: 50% - Preparando exportacion

PROGRESO: 75% - Guardando borrador
Borrador creado: /home/danna/G07-BigData-AnaliticaAgrocomercial-Danper/scripts/procesamiento_gold_borrador.py

PROGRESO: 90% - Verificando archivo

===== VERIFICACION =====
Archivo encontrado: True
Lineas exportadas: 8955
Tamano del archivo (bytes): 244228
Sintaxis del codigo generado valida: True
Lineas especiales detectadas: 0

===== RESULTADO FINAL =====
BORRADOR SCRIPT GOLD: GENERADO
Pendiente: organizar, depurar y probar su reproducibilidad.

PROGRESO: 100% - Exportacion finalizada
===== PASO 87 FINALIZADO =====


## Revisión estructural del script Gold

Se analiza el borrador del script Python obtenido
a partir del notebook de procesamiento Gold.

El objetivo es identificar los bloques de construcción,
validación, almacenamiento y recuperación presentes
en el código.

Se revisan especialmente las operaciones de escritura,
las dependencias de variables y las instrucciones
que podrían repetir procesamientos existentes.

Esta revisión es estática: no ejecuta el script ni
modifica los componentes de Hadoop HDFS.

Sus resultados servirán para organizar una versión
definitiva y reproducible del ETL Gold.

In [76]:
from pathlib import Path
import ast
import re
from collections import Counter

print("===== PASO 88: AUDITORIA DEL BORRADOR GOLD =====")
print("PROGRESO: 0% - Localizando script", flush=True)

repo = Path(
    "/home/danna/G07-BigData-AnaliticaAgrocomercial-Danper"
)

ruta_borrador = (
    repo / "scripts" / "procesamiento_gold_borrador.py"
)

if not ruta_borrador.is_file():
    print("ERROR: No se encontro el borrador del Paso 87.")

else:
    codigo = ruta_borrador.read_text(encoding="utf-8")

    print("Archivo:", ruta_borrador)
    print("Lineas:", len(codigo.splitlines()))

    print(
        "\nPROGRESO: 25% - Identificando bloques",
        flush=True
    )

    # Cada bloque corresponde a una celda exportada.
    patron_celdas = re.compile(
        r"(?m)^# CELDA DE CODIGO (\d+)\s*$"
    )

    coincidencias = list(
        patron_celdas.finditer(codigo)
    )

    bloques = []

    for posicion, coincidencia in enumerate(coincidencias):

        inicio = coincidencia.end()

        fin = (
            coincidencias[posicion + 1].start()
            if posicion + 1 < len(coincidencias)
            else len(codigo)
        )

        bloques.append({
            "celda": int(coincidencia.group(1)),
            "codigo": codigo[inicio:fin]
        })

    print("Bloques identificados:", len(bloques))

    print(
        "\nPROGRESO: 50% - Clasificando operaciones",
        flush=True
    )

    patrones = {
        "ESCRITURA": [
            ".write",
            "saveAsTable(",
            "insertInto(",
            "save(",
            "saveAsTextFile("
        ],
        "DIMENSION": [
            "dim_tiempo",
            "dim_cultivo",
            "dim_ubicacion_produccion",
            "dim_region_costos",
            "dim_pais_exportacion",
            "dim_mercado"
        ],
        "HECHOS": [
            "fact_produccion",
            "fact_exportaciones",
            "fact_agroindustria",
            "fact_costos",
            "fact_abastecimiento",
            "fact_precios"
        ],
        "KPI": [
            "kpi_produccion",
            "kpi_exportaciones",
            "kpi_agroindustria",
            "kpi_costos",
            "kpi_abastecimiento"
        ],
        "VALIDACION": [
            "exceptAll(",
            "duplicados",
            "claves_nulas",
            "validada",
            "validado"
        ]
    }

    categorias = Counter()
    celdas_escritura = []
    celdas_exceptall = []
    celdas_con_reinicio = []

    for bloque in bloques:

        numero = bloque["celda"]
        texto = bloque["codigo"]

        etiquetas = []

        for categoria, expresiones in patrones.items():

            if any(
                expresion in texto
                for expresion in expresiones
            ):
                etiquetas.append(categoria)
                categorias[categoria] += 1

        if "ESCRITURA" in etiquetas:
            celdas_escritura.append(numero)

        if "exceptAll(" in texto:
            celdas_exceptall.append(numero)

        if (
            "spark.stop(" in texto
            or "restart" in texto.lower()
        ):
            celdas_con_reinicio.append(numero)

    print("\n===== CLASIFICACION DE BLOQUES =====")

    for categoria in patrones:
        print(
            f"{categoria}: "
            f"{categorias[categoria]} celdas"
        )

    print("\n===== OPERACIONES A REVISAR =====")

    print(
        "Celdas con escrituras:",
        celdas_escritura
    )

    print(
        "Celdas con exceptAll:",
        celdas_exceptall
    )

    print(
        "Celdas con referencias a parada/reinicio:",
        celdas_con_reinicio
    )

    print(
        "\nPROGRESO: 75% - Revisando sintaxis",
        flush=True
    )

    try:
        arbol = ast.parse(codigo)
        sintaxis_correcta = True
        print("Sintaxis Python: CORRECTA")
    except SyntaxError as error:
        sintaxis_correcta = False
        print("Sintaxis Python: REVISAR")
        print("Detalle:", error)

    # Deteccion adicional de instrucciones sensibles.
    indicadores_riesgo = {
        "modo_overwrite": len(
            re.findall(
                r"""mode\(\s*['"]overwrite['"]\s*\)""",
                codigo,
                flags=re.IGNORECASE
            )
        ),
        "modo_errorifexists": len(
            re.findall(
                r"""mode\(\s*['"]errorifexists['"]\s*\)""",
                codigo,
                flags=re.IGNORECASE
            )
        ),
        "exceptAll": codigo.count("exceptAll("),
        "spark_stop": codigo.count("spark.stop(")
    }

    print("\n===== INDICADORES DE REVISION =====")

    for indicador, cantidad in indicadores_riesgo.items():
        print(f"{indicador}: {cantidad}")

    print(
        "\nPROGRESO: 90% - Preparando resumen",
        flush=True
    )

    auditoria_borrador = {
        "bloques": len(bloques),
        "celdas_escritura": celdas_escritura,
        "celdas_exceptall": celdas_exceptall,
        "indicadores": indicadores_riesgo,
        "sintaxis_correcta": sintaxis_correcta
    }

    print("\n===== RESULTADO FINAL =====")

    if sintaxis_correcta and len(bloques) == 71:
        print("AUDITORIA BORRADOR GOLD: COMPLETADA")
        print("Pendiente: depurar y organizar el ETL definitivo.")
    else:
        print("ATENCION: Revisar estructura del borrador.")

    print(
        "\nPROGRESO: 100% - Revision finalizada"
    )

print("===== PASO 88 FINALIZADO =====")

===== PASO 88: AUDITORIA DEL BORRADOR GOLD =====
PROGRESO: 0% - Localizando script
Archivo: /home/danna/G07-BigData-AnaliticaAgrocomercial-Danper/scripts/procesamiento_gold_borrador.py
Lineas: 8955

PROGRESO: 25% - Identificando bloques
Bloques identificados: 71

PROGRESO: 50% - Clasificando operaciones

===== CLASIFICACION DE BLOQUES =====
ESCRITURA: 23 celdas
DIMENSION: 20 celdas
HECHOS: 28 celdas
KPI: 13 celdas
VALIDACION: 63 celdas

===== OPERACIONES A REVISAR =====
Celdas con escrituras: [6, 11, 14, 17, 20, 24, 27, 33, 35, 38, 41, 44, 47, 50, 53, 56, 58, 60, 62, 64, 66, 68, 71]
Celdas con exceptAll: [23, 24, 26, 27, 37, 40, 43, 46, 49, 52, 55, 60]
Celdas con referencias a parada/reinicio: []

PROGRESO: 75% - Revisando sintaxis
Sintaxis Python: CORRECTA

===== INDICADORES DE REVISION =====
modo_overwrite: 0
modo_errorifexists: 22
exceptAll: 24
spark_stop: 0

PROGRESO: 90% - Preparando resumen

===== RESULTADO FINAL =====
AUDITORIA BORRADOR GOLD: COMPLETADA
Pendiente: depurar y orga

## Identificación de dependencias del procesamiento Gold

Se analiza la estructura del código exportado para
identificar las variables principales, los bloques
de construcción y las operaciones de almacenamiento.

Se registra el orden de las celdas del notebook y
se identifican aquellas que contienen instrucciones
de escritura en Hadoop HDFS.

Esta revisión permitirá separar las transformaciones
necesarias de las operaciones exploratorias y de
recuperación utilizadas durante el desarrollo.

El resultado servirá para planificar el flujo
reproducible del procesamiento Gold.

No se ejecutan transformaciones Spark ni se
modifican los datos almacenados.

In [77]:
from pathlib import Path
import json
import ast
import re

print("===== PASO 89: DEPENDENCIAS DEL SCRIPT GOLD =====")
print("PROGRESO: 0% - Leyendo notebook", flush=True)

repo_89 = Path(
    "/home/danna/G07-BigData-AnaliticaAgrocomercial-Danper"
)

notebook_89 = (
    repo_89 / "notebooks" / "procesamiento_gold.ipynb"
)

if not notebook_89.exists():
    print("ERROR: No se encontro el notebook.")

else:
    with notebook_89.open("r", encoding="utf-8") as archivo:
        datos_notebook_89 = json.load(archivo)

    celdas_codigo_89 = [
        "".join(c.get("source", []))
        for c in datos_notebook_89.get("cells", [])
        if c.get("cell_type") == "code"
        and "".join(c.get("source", [])).strip()
    ]

    print("Celdas de codigo actuales:", len(celdas_codigo_89))

    print(
        "\nPROGRESO: 25% - Analizando instrucciones",
        flush=True
    )

    inventario_89 = []

    for numero, codigo in enumerate(celdas_codigo_89, start=1):

        # Tomar una descripción corta de cada celda.
        lineas_utiles = [
            linea.strip()
            for linea in codigo.splitlines()
            if linea.strip()
            and not linea.strip().startswith("#")
        ]

        primera_linea = (
            lineas_utiles[0][:95]
            if lineas_utiles
            else "SIN INSTRUCCIONES"
        )

        # Detectar variables asignadas mediante AST.
        variables = []

        try:
            arbol = ast.parse(codigo)

            for nodo in ast.walk(arbol):
                if isinstance(nodo, ast.Assign):
                    for destino in nodo.targets:
                        if isinstance(destino, ast.Name):
                            variables.append(destino.id)

                elif isinstance(nodo, ast.AnnAssign):
                    if isinstance(nodo.target, ast.Name):
                        variables.append(nodo.target.id)

        except SyntaxError:
            pass

        variables = list(dict.fromkeys(variables))

        # Identificar operaciones relevantes.
        escritura = bool(
            re.search(
                r"\.write\b|saveAsTable\s*\(|insertInto\s*\(",
                codigo
            )
        )

        comparacion_pesada = "exceptAll(" in codigo

        usa_hdfs = (
            "BASE_GOLD" in codigo
            or "BASE_SILVER" in codigo
            or "hdfs://" in codigo
        )

        # Etiquetas orientativas; una celda puede tener varias.
        etiquetas = []

        if "SparkSession.builder" in codigo:
            etiquetas.append("INICIALIZACION")

        if "dim_" in codigo:
            etiquetas.append("DIMENSION")

        if "fact_" in codigo:
            etiquetas.append("HECHOS")

        if "kpi_" in codigo:
            etiquetas.append("KPI")

        if escritura:
            etiquetas.append("ESCRITURA")

        if comparacion_pesada:
            etiquetas.append("EXCEPTALL")

        if not etiquetas:
            etiquetas.append("OTRO")

        inventario_89.append({
            "celda": numero,
            "primera_linea": primera_linea,
            "variables": variables,
            "escritura": escritura,
            "exceptall": comparacion_pesada,
            "usa_hdfs": usa_hdfs,
            "etiquetas": etiquetas
        })

    print(
        "\nPROGRESO: 50% - Identificando celdas importantes",
        flush=True
    )

    print("\n===== CELDAS DE INICIALIZACION =====")

    for item in inventario_89:
        if (
            item["celda"] <= 10
            or "INICIALIZACION" in item["etiquetas"]
        ):
            print(
                f"Celda {item['celda']}: "
                f"{item['primera_linea']}"
            )

    print("\n===== CELDAS CON ESCRITURAS =====")

    for item in inventario_89:
        if item["escritura"]:
            print(
                f"Celda {item['celda']:02d} | "
                f"{','.join(item['etiquetas'])} | "
                f"Variables: {', '.join(item['variables'][:6])}"
            )

    print("\n===== CELDAS CON EXCEPTALL =====")

    celdas_exceptall_89 = [
        item["celda"]
        for item in inventario_89
        if item["exceptall"]
    ]

    print(celdas_exceptall_89)

    print(
        "\nPROGRESO: 75% - Preparando inventario",
        flush=True
    )

    carpeta_docs_89 = repo_89 / "docs"
    carpeta_docs_89.mkdir(parents=True, exist_ok=True)

    ruta_inventario_89 = (
        carpeta_docs_89 / "inventario_celdas_gold.json"
    )

    # No sobrescribir un inventario anterior.
    if ruta_inventario_89.exists():
        print("El inventario ya existe; no se sobrescribira.")
    else:
        with ruta_inventario_89.open(
            "x", encoding="utf-8"
        ) as archivo:
            json.dump(
                inventario_89,
                archivo,
                ensure_ascii=False,
                indent=2
            )

        print("Inventario creado:", ruta_inventario_89)

    print(
        "\nPROGRESO: 90% - Resumen final",
        flush=True
    )

    total_escrituras_89 = sum(
        item["escritura"]
        for item in inventario_89
    )

    print("\n===== RESUMEN =====")
    print("Celdas analizadas:", len(inventario_89))
    print("Celdas con escritura:", total_escrituras_89)
    print("Celdas con exceptAll:", len(celdas_exceptall_89))
    print("Inventario disponible:", ruta_inventario_89.is_file())

    print("\n===== RESULTADO FINAL =====")

    if (
        len(inventario_89) >= 71
        and ruta_inventario_89.is_file()
    ):
        print("INVENTARIO DE DEPENDENCIAS GOLD: GENERADO")
    else:
        print("ATENCION: REVISAR INVENTARIO GOLD")

    print("\nPROGRESO: 100% - Revision finalizada")

print("===== PASO 89 FINALIZADO =====")

===== PASO 89: DEPENDENCIAS DEL SCRIPT GOLD =====
PROGRESO: 0% - Leyendo notebook
Celdas de codigo actuales: 73

PROGRESO: 25% - Analizando instrucciones

PROGRESO: 50% - Identificando celdas importantes

===== CELDAS DE INICIALIZACION =====
Celda 1: from pyspark.sql import SparkSession
Celda 2: produccion = spark.read.parquet(f'{BASE_SILVER}/produccion')
Celda 3: produccion.select(
Celda 4: from pyspark.sql import functions as F
Celda 5: from pyspark.sql import functions as F
Celda 6: print("===== PASO 26: GUARDAR GOLD EN HDFS =====")
Celda 7: from pyspark.sql import functions as F
Celda 8: from pyspark.sql import functions as F
Celda 9: from pyspark.sql import functions as F
Celda 10: from pyspark.sql import functions as F

===== CELDAS CON ESCRITURAS =====
Celda 06 | ESCRITURA | Variables: ruta_gold, hadoop_conf, ruta_java, filesystem, gold_verificacion
Celda 11 | ESCRITURA | Variables: ruta_exportaciones_gold, hadoop_conf, ruta_java, filesystem, exportaciones_guardadas, registros_g

## Identificación de los bloques principales del ETL Gold

Se examinan las celdas del notebook para localizar
las instrucciones responsables de construir las tablas
analíticas, dimensiones, hechos e indicadores KPI.

Se identifican las variables creadas, las fuentes
utilizadas y las celdas responsables del almacenamiento.

El objetivo es determinar qué bloques deben conservarse
en el script definitivo de procesamiento Gold.

Las celdas de recuperación, documentación y exploración
se mantendrán fuera del flujo ETL de producción.

Esta revisión no ejecuta operaciones Spark ni modifica
los archivos almacenados en Hadoop HDFS.

In [78]:
from pathlib import Path
import json
import ast
import re

print("===== PASO 90: BLOQUES PRINCIPALES ETL GOLD =====")
print("PROGRESO: 0% - Leyendo notebook", flush=True)

repo_90 = Path(
    "/home/danna/G07-BigData-AnaliticaAgrocomercial-Danper"
)

ruta_notebook_90 = (
    repo_90 / "notebooks" / "procesamiento_gold.ipynb"
)

if not ruta_notebook_90.is_file():
    print("ERROR: No se encontro el notebook.")

else:
    with ruta_notebook_90.open(
        "r", encoding="utf-8"
    ) as archivo:
        notebook_90 = json.load(archivo)

    celdas_90 = [
        "".join(celda.get("source", []))
        for celda in notebook_90.get("cells", [])
        if (
            celda.get("cell_type") == "code"
            and "".join(celda.get("source", [])).strip()
        )
    ]

    print("Celdas de codigo:", len(celdas_90))

    print(
        "\nPROGRESO: 25% - Identificando tablas",
        flush=True
    )

    # Tablas que deben formar parte del ETL final.
    componentes_90 = [
        "produccion_mensual",
        "exportaciones_regionales",
        "exportaciones_danper",
        "agroindustria_mensual",
        "costos_mensuales",
        "abastecimiento_mayorista",
        "precios_mercado_productores",
        "dim_tiempo",
        "dim_cultivo",
        "dim_ubicacion_produccion",
        "dim_region_costos",
        "dim_pais_exportacion",
        "dim_mercado",
        "fact_produccion_mensual",
        "fact_exportaciones_regionales",
        "fact_exportaciones_danper",
        "fact_agroindustria_mensual",
        "fact_costos_mensuales",
        "fact_abastecimiento_mayorista",
        "fact_precios_mercado_productores",
        "kpi_produccion_anual",
        "kpi_exportaciones_regionales_anual",
        "kpi_exportaciones_danper_anual",
        "kpi_agroindustria_anual",
        "kpi_costos_anual",
        "kpi_abastecimiento_anual"
    ]

    # Localizar las escrituras reales a Parquet
    # y las referencias a cada componente.
    for categoria, nombres in [
        ("TABLAS ANALITICAS", componentes_90[:7]),
        ("DIMENSIONES", componentes_90[7:13]),
        ("TABLAS DE HECHOS", componentes_90[13:20]),
        ("KPI", componentes_90[20:])
    ]:
        print(f"\n===== {categoria} =====")

        for nombre in nombres:
            referencias = []
            escrituras = []

            for numero, codigo in enumerate(
                celdas_90, start=1
            ):
                if nombre in codigo:
                    referencias.append(numero)

                    if re.search(
                        r"\.write\b|\.parquet\s*\(",
                        codigo
                    ) and (
                        ".write" in codigo
                    ):
                        escrituras.append(numero)

            print(
                f"{nombre}: "
                f"referencias={referencias[:12]} "
                f"escrituras_posibles={escrituras}"
            )

    print(
        "\nPROGRESO: 60% - Localizando construcciones",
        flush=True
    )

    # Variables de DataFrames construidos en el notebook.
    prefijos_objetivo = (
        "dim_",
        "fact_",
        "kpi_"
    )

    variables_principales = {}

    for numero, codigo in enumerate(
        celdas_90, start=1
    ):
        try:
            arbol = ast.parse(codigo)
        except SyntaxError:
            continue

        # Revisar asignaciones del nivel principal.
        for nodo in arbol.body:
            if isinstance(nodo, ast.Assign):
                destinos = nodo.targets

            elif isinstance(nodo, ast.AnnAssign):
                destinos = [nodo.target]

            else:
                continue

            for destino in destinos:
                if not isinstance(destino, ast.Name):
                    continue

                nombre_variable = destino.id

                if nombre_variable.startswith(
                    prefijos_objetivo
                ):
                    variables_principales.setdefault(
                        nombre_variable, []
                    ).append(numero)

    print("\n===== VARIABLES DIMENSIONALES Y KPI =====")

    for variable, numeros in sorted(
        variables_principales.items()
    ):
        # Evitar saturar la salida con variables
        # temporales de validación.
        if (
            variable in componentes_90
            or variable.endswith("_anual")
            or variable.endswith("_mensual")
            or variable.startswith("dim_")
        ):
            print(
                f"{variable}: celdas {numeros}"
            )

    print(
        "\nPROGRESO: 80% - Detectando codigo auxiliar",
        flush=True
    )

    palabras_auxiliares = [
        "RECUPERACION",
        "AUDITORIA",
        "DOCUMENTAR",
        "BORRADOR SCRIPT",
        "DEPENDENCIAS DEL SCRIPT",
        "DIAGNOSTICO DE CALIDAD"
    ]

    celdas_auxiliares_90 = []

    for numero, codigo in enumerate(
        celdas_90, start=1
    ):
        encabezado = "\n".join(
            codigo.splitlines()[:12]
        ).upper()

        if any(
            palabra in encabezado
            for palabra in palabras_auxiliares
        ):
            celdas_auxiliares_90.append(numero)

    print(
        "Celdas auxiliares detectadas:",
        celdas_auxiliares_90
    )

    # Localizar los parámetros iniciales del entorno.
    print("\n===== CONFIGURACION DEL ENTORNO =====")

    for numero, codigo in enumerate(
        celdas_90[:5], start=1
    ):
        print(f"\nCelda {numero}:")

        for linea in codigo.splitlines():
            if any(
                patron in linea
                for patron in [
                    "SparkSession",
                    "BASE_SILVER",
                    "BASE_GOLD",
                    ".appName(",
                    "spark.version"
                ]
            ):
                print(" ", linea.strip()[:130])

    print("\n===== RESULTADO FINAL =====")
    print("Componentes Gold objetivo:", len(componentes_90))
    print("Celdas de codigo revisadas:", len(celdas_90))

    bloques_gold_identificados = True

    print("IDENTIFICACION BLOQUES GOLD: COMPLETADA")

    print(
        "\nPROGRESO: 100% - Revision finalizada"
    )

print("===== PASO 90 FINALIZADO =====")

===== PASO 90: BLOQUES PRINCIPALES ETL GOLD =====
PROGRESO: 0% - Leyendo notebook
Celdas de codigo: 73

PROGRESO: 25% - Identificando tablas

===== TABLAS ANALITICAS =====
produccion_mensual: referencias=[4, 5, 6, 28, 32, 34, 36, 37, 38, 57, 69, 70] escrituras_posibles=[6, 38]
exportaciones_regionales: referencias=[7, 8, 10, 11, 28, 32, 39, 40, 41, 59, 60, 61] escrituras_posibles=[11, 41, 60]
exportaciones_danper: referencias=[12, 13, 14, 28, 34, 42, 43, 44, 59, 60, 62, 69] escrituras_posibles=[14, 44, 60, 62]
agroindustria_mensual: referencias=[15, 16, 17, 28, 32, 45, 46, 47, 63, 69, 70] escrituras_posibles=[17, 47]
costos_mensuales: referencias=[18, 19, 20, 28, 34, 48, 49, 50, 65, 69, 70] escrituras_posibles=[20, 50]
abastecimiento_mayorista: referencias=[22, 23, 24, 28, 32, 34, 51, 52, 53, 67, 69, 70] escrituras_posibles=[24, 53]
precios_mercado_productores: referencias=[25, 26, 27, 28, 34, 54, 55, 56, 69] escrituras_posibles=[27, 56]

===== DIMENSIONES =====
dim_tiempo: referencias

## Preparación del código fuente para el ETL definitivo

Se extraen las celdas de construcción del notebook
para revisar sus dependencias y organizar el
procesamiento Gold en un script Python reproducible.

Se incluyen la inicialización de Spark, las
transformaciones analíticas, las dimensiones,
las tablas de hechos y los indicadores KPI.

Esta extracción es únicamente documental:
no ejecuta transformaciones ni modifica HDFS.

El código será revisado antes de generar el
script definitivo del proyecto.

In [79]:
from pathlib import Path
import json

print("===== PASO 91: PREPARAR FUENTES ETL GOLD =====")

repo = Path(
    "/home/danna/G07-BigData-AnaliticaAgrocomercial-Danper"
)

notebook = repo / "notebooks" / "procesamiento_gold.ipynb"

destino = repo / "scripts" / "fuentes_etl_gold_revision.txt"

if not notebook.is_file():
    print("ERROR: No se encontro el notebook.")

else:
    print("PROGRESO: 20% - Leyendo notebook", flush=True)

    with notebook.open("r", encoding="utf-8") as archivo:
        datos = json.load(archivo)

    celdas_codigo = [
        "".join(c.get("source", []))
        for c in datos.get("cells", [])
        if c.get("cell_type") == "code"
        and "".join(c.get("source", [])).strip()
    ]

    # Celdas de inicializacion, construccion y
    # almacenamiento. Excluimos auditorias finales,
    # recuperaciones y generacion de documentos.
    celdas_seleccionadas = [
        1, 2, 3, 4, 5, 6,
        7, 8, 9, 10, 11,
        12, 13, 14,
        15, 16, 17,
        18, 19, 20,
        21, 22, 23, 24,
        25, 26, 27,
        29, 30, 31, 32, 33, 34, 35,
        36, 38,
        39, 41,
        42, 44,
        45, 47,
        48, 50,
        51, 53,
        54, 56,
        57, 58,
        59, 62,
        63, 64,
        65, 66,
        67, 68
    ]

    print(
        "PROGRESO: 50% - Extrayendo instrucciones",
        flush=True
    )

    fragmentos = [
        "PROYECTO: BIG DATA Y ANALITICA AGROCOMERCIAL - DANPER",
        "ARCHIVO: FUENTES PARA REVISAR EL ETL GOLD",
        "NOTA: NO ES UN SCRIPT EJECUTABLE",
        ""
    ]

    for numero in celdas_seleccionadas:
        if numero <= len(celdas_codigo):
            fragmentos.extend([
                "",
                "=" * 70,
                f"CELDA ORIGINAL DE CODIGO {numero}",
                "=" * 70,
                celdas_codigo[numero - 1],
                ""
            ])

    contenido = "\n".join(fragmentos)

    print(
        "PROGRESO: 75% - Guardando archivo",
        flush=True
    )

    destino.parent.mkdir(parents=True, exist_ok=True)

    if destino.exists():
        print("ATENCION: El archivo ya existe.")
        print("No se sobrescribira.")
    else:
        with destino.open("x", encoding="utf-8") as archivo:
            archivo.write(contenido)

        print("Archivo creado correctamente.")

    print("\n===== VERIFICACION =====")
    print("Archivo:", destino)
    print("Existe:", destino.is_file())

    if destino.is_file():
        print("Tamano (bytes):", destino.stat().st_size)
        print(
            "Celdas seleccionadas:",
            len(celdas_seleccionadas)
        )

        print("\nFUENTES ETL GOLD: PREPARADAS")

    print("PROGRESO: 100% - Finalizado")

print("===== PASO 91 FINALIZADO =====")

===== PASO 91: PREPARAR FUENTES ETL GOLD =====
PROGRESO: 20% - Leyendo notebook
PROGRESO: 50% - Extrayendo instrucciones
PROGRESO: 75% - Guardando archivo
Archivo creado correctamente.

===== VERIFICACION =====
Archivo: /home/danna/G07-BigData-AnaliticaAgrocomercial-Danper/scripts/fuentes_etl_gold_revision.txt
Existe: True
Tamano (bytes): 191695
Celdas seleccionadas: 58

FUENTES ETL GOLD: PREPARADAS
PROGRESO: 100% - Finalizado
===== PASO 91 FINALIZADO =====
